# Notebook intent and run tag

In [ ]:
import datetime as dt
RUN_TAG = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
print("RUN_TAG:", RUN_TAG)

# Imports and paths

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb
import pyarrow.parquet as pq

ARTIFACT_ROOT = Path("artifacts/failure_analysis_v2")
DUCKDB_PATH = ARTIFACT_ROOT / "failure_analysis_v2.duckdb"
PARQUET_DIR = ARTIFACT_ROOT / "parquet_rich_v2_20260306_114748"  # update if you want newest

assert DUCKDB_PATH.exists(), f"Missing {DUCKDB_PATH}"
assert PARQUET_DIR.exists(), f"Missing {PARQUET_DIR}"

con = duckdb.connect(str(DUCKDB_PATH))

def qdf(sql: str) -> pd.DataFrame:
    return con.execute(sql).df()

def read_pq(name: str, cols=None, verbose=True) -> pd.DataFrame:
    path = PARQUET_DIR / f"{name}.parquet"
    pf = pq.ParquetFile(path)
    available = pf.schema_arrow.names
    if cols is None:
        return pd.read_parquet(path)
    cols_use = [c for c in cols if c in available]
    missing = [c for c in cols if c not in available]
    if verbose and missing:
        print(f"[read_pq] {name}: requested {len(cols)} cols, using {len(cols_use)}. Missing: {missing}")
    if not cols_use:
        raise ValueError(f"[read_pq] None of requested columns exist. Requested={cols}")
    return pd.read_parquet(path, columns=cols_use)

## Load baseline V2 (light) for guardrails work

In [ ]:
BASE_COLS = [
    "row_id", "Year", "HCPCS_Cd", "hcpcs_desc",
    "provider_type", "Place_Of_Srvc", "state",
    "has_lag", "route",
    "expected_cost_source", "expected_cost_support_tier", "cold_start_anchor_source",
    "hcpcs_stability_group", "n_years_present_for_code", "years_list",
    "observed_cost", "expected_cost", "residual", "abs_residual", "oe_ratio",
    "is_any_catastrophic",
    # catastrophic buckets (V2)
    "cat_abs_residual_q99",
    "cat_oe_q99",
    "cat_large_positive_residual",
    "cat_large_negative_residual",
    "cat_large_error_high_support",
    "cat_large_error_tail_row",
    "cat_underprediction",
    "cat_overprediction",
    "cat_high_conf_anomaly",
    "cat_cold_low_support_failure",
]

v2 = read_pq("failure_df_full_v2", cols=BASE_COLS)
print(v2.shape)
v2["route"] = np.where(v2["has_lag"].astype(bool), "hot_start", "cold_start")  # enforce canonical

## Define evaluation helpers (A–G recap metrics)

In [ ]:
CAT_COLS = [
    "cat_abs_residual_q99",
    "cat_oe_q99",
    "cat_large_positive_residual",
    "cat_large_negative_residual",
    "cat_large_error_high_support",
    "cat_large_error_tail_row",
    "cat_underprediction",
    "cat_overprediction",
    "cat_high_conf_anomaly",
    "cat_cold_low_support_failure",
]

def summarize_cat_rates(df: pd.DataFrame, group_cols: list[str], *, year=None) -> pd.DataFrame:
    x = df
    if year is not None:
        x = x.loc[x["Year"] == year].copy()

    agg_dict = {
        "n_rows": ("row_id", "size"),
        "any_cat_rate": ("is_any_catastrophic", "mean"),
    }
    for c in CAT_COLS:
        if c in x.columns:
            agg_dict[f"{c}_rate"] = (c, "mean")

    out = (
        x.groupby(group_cols, dropna=False)
        .agg(**agg_dict)
        .reset_index()
    )
    rate_cols = [c for c in out.columns if c.endswith("_rate")]
    out[rate_cols] = out[rate_cols] * 100
    return out

def headline_metrics(df: pd.DataFrame, label: str) -> pd.DataFrame:
    # overall + 2023 slice
    def one(sub, name):
        return pd.DataFrame({
            "slice": [name],
            "n_rows": [len(sub)],
            "any_cat_rate_pct": [100 * sub["is_any_catastrophic"].mean()],
            "median_abs_residual": [sub["abs_residual"].median()],
            "median_oe_ratio": [sub["oe_ratio"].median()],
        })
    overall = one(df, "overall")
    y2023 = one(df.loc[df["Year"] == 2023], "year_2023")
    out = pd.concat([overall, y2023], axis=0).reset_index(drop=True)
    out.insert(0, "label", label)
    return out

## Baseline snapshot (store what “V2” looks like)

In [ ]:
baseline_headlines = headline_metrics(v2, "V2_baseline")
display(baseline_headlines)

# 2023 stable vs 2023_only (your G1 view)
g1_baseline = summarize_cat_rates(
    v2.loc[v2["hcpcs_stability_group"].isin(["stable_all_years", "2023_only"])],
    group_cols=["hcpcs_stability_group"],
    year=2023
)
display(g1_baseline)

# Phase 1. Coverage-shift handling for 2023-only codes


We already have `hcpcs_stability_group` attached. Now we define a “coverage regime feature set” and implement a conservative expected-cost rule for `2023_only` rows.

What to implement first (minimal, testable)

For `Year == 2023` and `hcpcs_stability_group == "2023_only"` (which we found are 100% `cold_start`), replace `expected_cost` with a conservative prior that reduces catastrophic flags.

A simple and defensible first guardrail is:
- Use a blended expected cost:
- `expected_guard` = `max(expected_cost, floor_prior)`
- where `floor_prior` is a robust baseline like the median `observed_cost` for that code family or a global cold-start prior.
- Start with a global prior first (simple), then refine:
    1. global prior for 2023-only cold rows
	2. family-specific prior (A9*, J3490/J9999/J3590)
	3. anchor-context-specific priors (`grp_mean` vs `hcpcs_mean` vs `pt_pos_state_mean`)

### Baseline trasholds from V2 for reuse in guardrails notebook

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# Baseline thresholds (V2): recompute from failure_df_full_v2.parquet
# - Matches your original Section A logic: GLOBAL across all years
# - Prints values
# - Stores in thresholds_v2 dict for reuse in guardrails notebook
# ============================================================

# Load only needed columns (fast + avoids schema issues)
base = read_pq(
    "failure_df_full_v2",
    cols=[
        "abs_residual",
        "oe_ratio",
        "residual",
        "observed_cost",
        "expected_cost",
    ],
    verbose=True,
)

# Defensive: ensure numeric
for c in ["abs_residual", "oe_ratio", "residual", "observed_cost", "expected_cost"]:
    base[c] = pd.to_numeric(base[c], errors="coerce")

# Quantile-based thresholds (GLOBAL)
abs_resid_q99 = base["abs_residual"].quantile(0.99)
oe_q99        = base["oe_ratio"].quantile(0.99)
resid_pos_q99 = base["residual"].quantile(0.99)
resid_neg_q01 = base["residual"].quantile(0.01)

# Operational bucket level thresholds (GLOBAL)
obs_q99 = base["observed_cost"].quantile(0.99)
exp_q99 = base["expected_cost"].quantile(0.99)

thresholds_v2 = {
    "abs_resid_q99": float(abs_resid_q99),
    "oe_q99": float(oe_q99),
    "resid_pos_q99": float(resid_pos_q99),
    "resid_neg_q01": float(resid_neg_q01),
    "obs_q99": float(obs_q99),
    "exp_q99": float(exp_q99),
    "oe_underprediction_cut": 3.0,        # your fixed rule
    "oe_overprediction_cut": 1 / 3.0,     # your fixed rule
}

print("Baseline V2 thresholds (GLOBAL across all years)")
print("  abs_resid_q99:", thresholds_v2["abs_resid_q99"])
print("  oe_q99       :", thresholds_v2["oe_q99"])
print("  resid_pos_q99:", thresholds_v2["resid_pos_q99"])
print("  resid_neg_q01:", thresholds_v2["resid_neg_q01"])
print("  obs_q99      :", thresholds_v2["obs_q99"])
print("  exp_q99      :", thresholds_v2["exp_q99"])
print("  oe_underprediction_cut:", thresholds_v2["oe_underprediction_cut"])
print("  oe_overprediction_cut :", thresholds_v2["oe_overprediction_cut"])

## A helper like `apply_catastrophic_flags(df, thresholds_v2, prefix="")` 

so you can:
- take any scored benchmarking dataframe (baseline V2, guardrailed V2, future variants),
- and stamp on the same catastrophic flags using the fixed baseline thresholds,
- optionally with a prefix so you can keep multiple flag sets side-by-side in one table (ex: `v2__cat_*` and `gr__cat_*`).

In [ ]:
import numpy as np
import pandas as pd

def apply_catastrophic_flags(df: pd.DataFrame, thresholds: dict, prefix: str = "") -> pd.DataFrame:
    """
    Apply the *baseline* catastrophic bucket logic (Section A) to ANY scored dataframe.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain at least:
        - abs_residual, oe_ratio, residual, observed_cost, expected_cost
        - has_lag (bool-ish)
        - expected_cost_support_tier (string-ish)
        - is_top_1pct_avg_mdcr_stdzd_amt (bool-ish) for tail-row bucket
        - high_confidence_anomaly_candidate (bool-ish) for high-conf anomaly bucket
    thresholds : dict
        Your fixed baseline thresholds_v2 dict, e.g.:
        - abs_resid_q99, oe_q99, resid_pos_q99, resid_neg_q01, obs_q99, exp_q99
        - oe_underprediction_cut (3.0), oe_overprediction_cut (1/3)
    prefix : str
        Optional column prefix. Example: "v2__" or "gr__".
        If prefix != "", all created columns are named like f"{prefix}cat_abs_residual_q99".

    Returns
    -------
    pd.DataFrame
        Copy of df with new catastrophic flag columns appended.
    """
    out = df.copy()

    # -----------------------------
    # 0) Validate required columns
    # -----------------------------
    required = [
        "abs_residual", "oe_ratio", "residual", "observed_cost", "expected_cost",
        "has_lag", "expected_cost_support_tier",
    ]
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError(f"apply_catastrophic_flags: missing required columns: {missing}")

    # Optional-but-used columns (if missing, we treat as False)
    if "is_top_1pct_avg_mdcr_stdzd_amt" not in out.columns:
        out["is_top_1pct_avg_mdcr_stdzd_amt"] = False
    if "high_confidence_anomaly_candidate" not in out.columns:
        out["high_confidence_anomaly_candidate"] = False

    # -----------------------------
    # 1) Pull thresholds (fixed)
    # -----------------------------
    abs_resid_q99 = thresholds["abs_resid_q99"]
    oe_q99        = thresholds["oe_q99"]
    resid_pos_q99 = thresholds["resid_pos_q99"]
    resid_neg_q01 = thresholds["resid_neg_q01"]
    obs_q99       = thresholds["obs_q99"]
    exp_q99       = thresholds["exp_q99"]
    oe_under_cut  = thresholds.get("oe_underprediction_cut", 3.0)
    oe_over_cut   = thresholds.get("oe_overprediction_cut", 1 / 3.0)

    # Helper to build prefixed column names
    def col(name: str) -> str:
        return f"{prefix}{name}" if prefix else name

    # -----------------------------
    # 2) Quantile-based buckets
    # -----------------------------
    # Very high absolute error (top 1% under baseline threshold)
    out[col("cat_abs_residual_q99")] = out["abs_residual"] >= abs_resid_q99

    # Very high O/E ratio (top 1% under baseline threshold)
    out[col("cat_oe_q99")] = out["oe_ratio"] >= oe_q99

    # Very large positive residual (top 1% residual under baseline threshold)
    out[col("cat_large_positive_residual")] = out["residual"] >= resid_pos_q99

    # Very large negative residual (bottom 1% residual under baseline threshold)
    out[col("cat_large_negative_residual")] = out["residual"] <= resid_neg_q01

    # Large error on high-support rows
    out[col("cat_large_error_high_support")] = (
        out["expected_cost_support_tier"].isin(["high", "medium_high"])
        & (out["abs_residual"] >= abs_resid_q99)
    )

    # Large error within tail rows (top 1% observed-cost rows flag computed upstream)
    out[col("cat_large_error_tail_row")] = (
        out["is_top_1pct_avg_mdcr_stdzd_amt"].astype(bool)
        & (out["abs_residual"] >= abs_resid_q99)
    )

    # -----------------------------
    # 3) Operational / narrative buckets
    # -----------------------------
    # Catastrophic underprediction: very high observed + expected far too low
    out[col("cat_underprediction")] = (
        (out["observed_cost"] >= obs_q99)
        & (out["oe_ratio"] >= oe_under_cut)
    )

    # Catastrophic overprediction: very high expected + observed far lower than expected
    out[col("cat_overprediction")] = (
        (out["expected_cost"] >= exp_q99)
        & (out["oe_ratio"] <= oe_over_cut)
    )

    # High-confidence anomaly candidate: carry forward the upstream boolean
    out[col("cat_high_conf_anomaly")] = out["high_confidence_anomaly_candidate"].astype(bool)

    # Cold-row low-support failure: cold row + weak tier + very large miss
    out[col("cat_cold_low_support_failure")] = (
        (~out["has_lag"].astype(bool))
        & out["expected_cost_support_tier"].isin(["medium", "low"])
        & (out["abs_residual"] >= abs_resid_q99)
    )

    # Hot-row failure: hot row + very large miss
    out[col("cat_hot_failure")] = (
        out["has_lag"].astype(bool)
        & (out["abs_residual"] >= abs_resid_q99)
    )

    # Cold-row failure: cold row + very large miss
    out[col("cat_cold_failure")] = (
        (~out["has_lag"].astype(bool))
        & (out["abs_residual"] >= abs_resid_q99)
    )

    # -----------------------------
    # 4) Union flag: is_any_catastrophic
    # -----------------------------
    cat_cols = [
        col("cat_abs_residual_q99"),
        col("cat_oe_q99"),
        col("cat_large_positive_residual"),
        col("cat_large_negative_residual"),
        col("cat_large_error_high_support"),
        col("cat_large_error_tail_row"),
        col("cat_underprediction"),
        col("cat_overprediction"),
        col("cat_high_conf_anomaly"),
        col("cat_cold_low_support_failure"),
        col("cat_hot_failure"),
        col("cat_cold_failure"),
    ]
    out[col("is_any_catastrophic")] = out[cat_cols].fillna(False).any(axis=1)

    return out

# The core philosophy for 2023-only guardrails

We are in a **no-training-support regime** for these codes. So our goal is not “perfect prediction.” It’s:

1. **Stop pathological underestimation** that creates huge residuals for 2023-only codes.
2. Keep the guardrail **simple, auditable, and conservative**, especially for high-risk families.
3. Measure success using **fixed baseline thresholds** (our `thresholds_v2`) so comparisons are apples-to-apples.

## Section A. Global priors for 2023-only cold rows (simplest first)

### A1) Global cold-start floor (one number, from train cold rows)


#### A1.1. Compute global cold-start floor from train (2020–2022, cold rows)

Notes:
- Median is a strong first choice because it is robust to outliers.
- If we want a more conservative floor later, we can swap median for e.g. quantile(0.25) or quantile(0.10).


In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# A1.1) Global cold-start floor prior from TRAIN years (2020–2022)
# - Use cold-start rows only (has_lag == False)
# - Use observed_cost distribution (train target)
# - Pick a robust statistic (median by default)
# ============================================================

TRAIN_YEARS = [2020, 2021, 2022]

train_cold = read_pq(
    "failure_df_full_v2",
    cols=["Year", "has_lag", "observed_cost"],
)

train_cold = train_cold.loc[
    train_cold["Year"].isin(TRAIN_YEARS) & (~train_cold["has_lag"].astype(bool))
].copy()

train_cold["observed_cost"] = pd.to_numeric(train_cold["observed_cost"], errors="coerce")

global_cold_floor = float(train_cold["observed_cost"].median())

print("A1 global cold-start floor prior (median observed_cost, train cold rows):", global_cold_floor)
print("Train cold rows used:", len(train_cold))

#### A1.2. Apply A1 guardrail to 2023-only rows and recompute metrics

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# A1.2) Apply floor to 2023-only rows (Year=2023 & hcpcs_stability_group==2023_only)
# - expected_cost_guard = max(expected_cost, global_cold_floor)
# - recompute residual/abs_residual/oe_ratio
# - apply baseline catastrophic flags using thresholds_v2
# ============================================================

# Load a working slice with only what we need
cols_needed = [
    "row_id", "Year", "HCPCS_Cd", "hcpcs_stability_group",
    "has_lag", "expected_cost_support_tier", "is_top_1pct_avg_mdcr_stdzd_amt",
    "high_confidence_anomaly_candidate",
    "observed_cost", "expected_cost",
    "residual", "abs_residual", "oe_ratio",
    "is_any_catastrophic",
]
df = read_pq("failure_df_full_v2", cols=cols_needed)

# Focus: 2023-only slice (this is the population we are trying to improve first)
slice_2023_only = df.loc[
    (df["Year"] == 2023) & (df["hcpcs_stability_group"] == "2023_only")
].copy()

print("Rows in 2023-only slice:", len(slice_2023_only))
print("Share cold-start (has_lag==False):", (~slice_2023_only["has_lag"].astype(bool)).mean())

# Keep baseline columns for comparison
slice_2023_only["expected_cost_baseline"] = pd.to_numeric(slice_2023_only["expected_cost"], errors="coerce")
slice_2023_only["oe_ratio_baseline"] = pd.to_numeric(slice_2023_only["oe_ratio"], errors="coerce")
slice_2023_only["abs_residual_baseline"] = pd.to_numeric(slice_2023_only["abs_residual"], errors="coerce")
slice_2023_only["is_any_catastrophic_baseline"] = slice_2023_only["is_any_catastrophic"].astype(bool)

# Apply guardrail floor
slice_2023_only["expected_cost_guard"] = np.maximum(
    slice_2023_only["expected_cost_baseline"].to_numpy(),
    global_cold_floor
)

# Recompute metrics under guardrail
slice_2023_only["residual_guard"] = (
    pd.to_numeric(slice_2023_only["observed_cost"], errors="coerce")
    - slice_2023_only["expected_cost_guard"]
)
slice_2023_only["abs_residual_guard"] = slice_2023_only["residual_guard"].abs()

# Denominator safety (tiny epsilon) to avoid division-by-zero surprises
EPS = 1e-9
slice_2023_only["oe_ratio_guard"] = (
    pd.to_numeric(slice_2023_only["observed_cost"], errors="coerce")
    / np.maximum(slice_2023_only["expected_cost_guard"].to_numpy(), EPS)
)

# Build a "scored" df in the same schema apply_catastrophic_flags expects
guard_scored = slice_2023_only.copy()
guard_scored["expected_cost"] = guard_scored["expected_cost_guard"]
guard_scored["residual"] = guard_scored["residual_guard"]
guard_scored["abs_residual"] = guard_scored["abs_residual_guard"]
guard_scored["oe_ratio"] = guard_scored["oe_ratio_guard"]

# Apply baseline bucket logic to guardrailed metrics
guard_scored = apply_catastrophic_flags(guard_scored, thresholds_v2, prefix="gr__")

#### A1.3. Compare baseline vs A1 guardrails on 2023-only slice

In [ ]:
import numpy as np
import pandas as pd

# 1) Apply baseline bucket logic to BASELINE metrics (prefix bl__)
baseline_scored = slice_2023_only.copy()
baseline_scored = apply_catastrophic_flags(baseline_scored, thresholds_v2, prefix="bl__")

# 2) You already have guard_scored with gr__ flags. If not, recreate:
# guard_scored = apply_catastrophic_flags(guard_scored, thresholds_v2, prefix="gr__")

# 3) Compare catastrophic rate (baseline vs guardrail)
baseline_cat_rate = baseline_scored["bl__is_any_catastrophic"].mean() * 100
guard_cat_rate = guard_scored["gr__is_any_catastrophic"].mean() * 100

print("2023-only catastrophic rate (baseline):   ", round(baseline_cat_rate, 4), "%")
print("2023-only catastrophic rate (A1 guardrail):", round(guard_cat_rate, 4), "%")
print("Absolute change (pp):", round(guard_cat_rate - baseline_cat_rate, 4))

# 4) Bucket mix comparison with real baseline rates
bucket_cols = [
    "cat_abs_residual_q99",
    "cat_oe_q99",
    "cat_large_positive_residual",
    "cat_large_negative_residual",
    "cat_large_error_high_support",
    "cat_large_error_tail_row",
    "cat_underprediction",
    "cat_overprediction",
    "cat_high_conf_anomaly",
    "cat_cold_low_support_failure",
    "cat_hot_failure",
    "cat_cold_failure",
]

bucket_compare = []
for b in bucket_cols:
    base_col = f"bl__{b}"
    gr_col = f"gr__{b}"
    base_rate = baseline_scored[base_col].mean() * 100
    gr_rate = guard_scored[gr_col].mean() * 100
    bucket_compare.append({
        "bucket": b,
        "baseline_rate_%": base_rate,
        "A1_rate_%": gr_rate,
        "delta_pp": gr_rate - base_rate
    })

bucket_compare = pd.DataFrame(bucket_compare).sort_values("delta_pp")
display(bucket_compare)

#### Diagnostic for `cat_large_negative_residual` blow-up

In [ ]:
new_cat = guard_scored["gr__is_any_catastrophic"] & (~baseline_scored["bl__is_any_catastrophic"])
print("New catastrophes created by A1:", new_cat.mean() * 100)

# Which bucket(s) are driving the new catastrophes?
driver_cols = [f"gr__{b}" for b in bucket_cols]
driver_rates = guard_scored.loc[new_cat, driver_cols].mean().sort_values(ascending=False) * 100
display(driver_rates)

#### A1.3.a. Apply floor only when expected is tiny (expected_cost < 1)

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# A1.3.a) "Tiny-expected-only" floor guardrail
# Rule:
#   expected_guard = expected_cost if expected_cost >= 1
#                 else max(expected_cost, global_cold_floor)
#
# Motivation:
# - Fix denominator pathologies (tiny expected) without pushing up expected
#   for already-reasonable rows (which created the negative-residual catastrophe wave).
# ============================================================

# Start from the same 2023-only slice you used earlier
a1b = slice_2023_only.copy()

# Keep baseline copies for reference (optional, but handy)
a1b["expected_cost_baseline"] = pd.to_numeric(a1b["expected_cost"], errors="coerce")
a1b["observed_cost_num"] = pd.to_numeric(a1b["observed_cost"], errors="coerce")

# Apply conditional floor only for tiny expected (< 1)
tiny_mask = a1b["expected_cost_baseline"] < 1

a1b["expected_cost_guard"] = a1b["expected_cost_baseline"].to_numpy()
a1b.loc[tiny_mask, "expected_cost_guard"] = np.maximum(
    a1b.loc[tiny_mask, "expected_cost_baseline"].to_numpy(),
    global_cold_floor
)

# Recompute metrics under guardrail
a1b["residual_guard"] = a1b["observed_cost_num"] - a1b["expected_cost_guard"]
a1b["abs_residual_guard"] = a1b["residual_guard"].abs()

EPS = 1e-9
a1b["oe_ratio_guard"] = a1b["observed_cost_num"] / np.maximum(a1b["expected_cost_guard"].to_numpy(), EPS)

# Build guard-scored frame with the schema expected by apply_catastrophic_flags
guard_scored_a1b = a1b.copy()
guard_scored_a1b["expected_cost"] = guard_scored_a1b["expected_cost_guard"]
guard_scored_a1b["residual"] = guard_scored_a1b["residual_guard"]
guard_scored_a1b["abs_residual"] = guard_scored_a1b["abs_residual_guard"]
guard_scored_a1b["oe_ratio"] = guard_scored_a1b["oe_ratio_guard"]

# Apply baseline catastrophic labels on the guardrailed metrics
guard_scored_a1b = apply_catastrophic_flags(guard_scored_a1b, thresholds_v2, prefix="grA1b__")

# ============================================================
# Compare baseline vs A1b (2023-only slice)
# ============================================================

baseline_cat_rate = baseline_scored["bl__is_any_catastrophic"].mean() * 100
a1b_cat_rate = guard_scored_a1b["grA1b__is_any_catastrophic"].mean() * 100

print("2023-only catastrophic rate (baseline):", round(baseline_cat_rate, 4), "%")
print("2023-only catastrophic rate (A1b):     ", round(a1b_cat_rate, 4), "%")
print("Absolute change (pp):", round(a1b_cat_rate - baseline_cat_rate, 4))

# Bucket mix comparison
bucket_compare_a1b = []
for b in bucket_cols:
    base_rate = baseline_scored[f"bl__{b}"].mean() * 100
    gr_rate = guard_scored_a1b[f"grA1b__{b}"].mean() * 100
    bucket_compare_a1b.append({
        "bucket": b,
        "baseline_rate_%": base_rate,
        "A1b_rate_%": gr_rate,
        "delta_pp": gr_rate - base_rate
    })

bucket_compare_a1b = pd.DataFrame(bucket_compare_a1b).sort_values("delta_pp")
display(bucket_compare_a1b)

# Optional: confirm we did NOT create a new negative-residual wave
new_cat_a1b = guard_scored_a1b["grA1b__is_any_catastrophic"] & (~baseline_scored["bl__is_any_catastrophic"])
print("New catastrophes created by A1b:", round(new_cat_a1b.mean() * 100, 4), "%")

if new_cat_a1b.any():
    driver_cols = [f"grA1b__{b}" for b in bucket_cols]
    driver_rates = guard_scored_a1b.loc[new_cat_a1b, driver_cols].mean().sort_values(ascending=False) * 100
    display(driver_rates)

#### A1.3.b he key fix: do not let the floor exceed observed for these rows

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# A1c) Global cold-start floor, but bounded by observed_cost
# Rule (only when expected < 1):
#   expected_guard = max(expected, min(global_floor, observed_cost))
# This prevents creating negative residuals by forcing expected > observed.
# ============================================================

a1c = slice_2023_only.copy()

a1c["expected_cost_baseline"] = pd.to_numeric(a1c["expected_cost"], errors="coerce")
a1c["observed_cost_num"] = pd.to_numeric(a1c["observed_cost"], errors="coerce")

tiny_mask = a1c["expected_cost_baseline"] < 1

# default: no change
a1c["expected_cost_guard"] = a1c["expected_cost_baseline"].to_numpy()

# guarded floor (bounded by observed)
bounded_floor = np.minimum(global_cold_floor, a1c.loc[tiny_mask, "observed_cost_num"].to_numpy())
a1c.loc[tiny_mask, "expected_cost_guard"] = np.maximum(
    a1c.loc[tiny_mask, "expected_cost_baseline"].to_numpy(),
    bounded_floor
)

# recompute metrics
a1c["residual_guard"] = a1c["observed_cost_num"] - a1c["expected_cost_guard"]
a1c["abs_residual_guard"] = a1c["residual_guard"].abs()

EPS = 1e-9
a1c["oe_ratio_guard"] = a1c["observed_cost_num"] / np.maximum(a1c["expected_cost_guard"].to_numpy(), EPS)

guard_scored_a1c = a1c.copy()
guard_scored_a1c["expected_cost"] = guard_scored_a1c["expected_cost_guard"]
guard_scored_a1c["residual"] = guard_scored_a1c["residual_guard"]
guard_scored_a1c["abs_residual"] = guard_scored_a1c["abs_residual_guard"]
guard_scored_a1c["oe_ratio"] = guard_scored_a1c["oe_ratio_guard"]

guard_scored_a1c = apply_catastrophic_flags(guard_scored_a1c, thresholds_v2, prefix="grA1c__")

# Compare catastrophic rate
baseline_cat_rate = baseline_scored["bl__is_any_catastrophic"].mean() * 100
a1c_cat_rate = guard_scored_a1c["grA1c__is_any_catastrophic"].mean() * 100

print("2023-only catastrophic rate (baseline):", round(baseline_cat_rate, 4), "%")
print("2023-only catastrophic rate (A1c):     ", round(a1c_cat_rate, 4), "%")
print("Absolute change (pp):", round(a1c_cat_rate - baseline_cat_rate, 4))

# Bucket deltas
bucket_compare_a1c = []
for b in bucket_cols:
    base_rate = baseline_scored[f"bl__{b}"].mean() * 100
    gr_rate = guard_scored_a1c[f"grA1c__{b}"].mean() * 100
    bucket_compare_a1c.append({"bucket": b, "baseline_rate_%": base_rate, "A1c_rate_%": gr_rate, "delta_pp": gr_rate - base_rate})

display(pd.DataFrame(bucket_compare_a1c).sort_values("delta_pp"))

# New catastrophes check
new_cat_a1c = guard_scored_a1c["grA1c__is_any_catastrophic"] & (~baseline_scored["bl__is_any_catastrophic"])
print("New catastrophes created by A1c:", round(new_cat_a1c.mean() * 100, 4), "%")

if new_cat_a1c.any():
    driver_cols = [f"grA1c__{b}" for b in bucket_cols]
    display(guard_scored_a1c.loc[new_cat_a1c, driver_cols].mean().sort_values(ascending=False) * 100)

#### A1.3.c. Grid search for 2023-only cold-start floor guardrail (bounded by observed)

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# A1.3.c) Grid search for 2023-only cold-start floor guardrail (bounded by observed)
# Rule:
#   if expected_cost < trigger:
#       expected_guard = min(observed_cost, max(expected_cost, floor_prior))
#   else:
#       expected_guard = expected_cost
#
# Grid:
#   trigger_lt in {0.1, 0.5, 1.0}
#   floor in {train cold observed_cost p25, median, p75}
#
# Evaluate on 2023-only slice using FIXED thresholds_v2 (apples-to-apples).
# Produces:
#   - results_df (summary table)
#   - bucket_deltas_df (long bucket delta table)
#   - best_row (best config row)
#   - floor_candidates (dict)
# ============================================================

# -----------------------------
# Preconditions
# -----------------------------
assert "slice_2023_only" in globals(), "slice_2023_only not found."
assert "thresholds_v2" in globals(), "thresholds_v2 not found."
assert "apply_catastrophic_flags" in globals(), "apply_catastrophic_flags not found."
assert "read_pq" in globals(), "read_pq not found."
assert "baseline_scored" in globals(), "baseline_scored not found. Build it via apply_catastrophic_flags(..., prefix='bl__')."
assert "bucket_cols" in globals(), "bucket_cols not found."

# Ensure numeric on slice
work0 = slice_2023_only.copy()
for c in ["observed_cost", "expected_cost", "residual", "abs_residual", "oe_ratio"]:
    if c in work0.columns:
        work0[c] = pd.to_numeric(work0[c], errors="coerce")

# -----------------------------
# 1) Floor candidates from TRAIN cold rows (2020–2022, has_lag == False)
# -----------------------------
TRAIN_YEARS = [2020, 2021, 2022]

train_cold = read_pq("failure_df_full_v2", cols=["Year", "has_lag", "observed_cost"])
train_cold = train_cold.loc[
    train_cold["Year"].isin(TRAIN_YEARS) & (~train_cold["has_lag"].astype(bool))
].copy()
train_cold["observed_cost"] = pd.to_numeric(train_cold["observed_cost"], errors="coerce")

floor_candidates = {
    "p25": float(train_cold["observed_cost"].quantile(0.25)),
    "median": float(train_cold["observed_cost"].median()),
    "p75": float(train_cold["observed_cost"].quantile(0.75)),
}
print("Train-cold observed_cost floor candidates:", floor_candidates)

# -----------------------------
# 2) Baseline reference (already scored in baseline_scored)
# -----------------------------
baseline_cat_rate = baseline_scored["bl__is_any_catastrophic"].mean() * 100
baseline_bucket_rates = {b: baseline_scored[f"bl__{b}"].mean() * 100 for b in bucket_cols}

# -----------------------------
# 3) Grid search
# -----------------------------
trigger_thresholds = [0.1, 0.5, 1.0]
floor_names = ["p25", "median", "p75"]

EPS = 1e-9
rows = []
bucket_delta_rows = []

# baseline arrays
obs0 = pd.to_numeric(work0["observed_cost"], errors="coerce").to_numpy(dtype="float64")
exp0 = pd.to_numeric(work0["expected_cost"], errors="coerce").to_numpy(dtype="float64")

for trig in trigger_thresholds:
    trig_mask_baseline = (exp0 < trig)
    for floor_name in floor_names:
        floor_val = float(floor_candidates[floor_name])

        exp_guard = exp0.copy()
        mask = exp0 < trig

        # bounded floor: raise toward floor but never exceed observed
        raised = np.maximum(exp0[mask], floor_val)
        exp_guard[mask] = np.minimum(obs0[mask], raised)

        # recompute metrics
        residual_guard = obs0 - exp_guard
        abs_resid_guard = np.abs(residual_guard)
        oe_guard = obs0 / np.maximum(exp_guard, EPS)

        scored = work0.copy()
        scored["expected_cost"] = exp_guard
        scored["residual"] = residual_guard
        scored["abs_residual"] = abs_resid_guard
        scored["oe_ratio"] = oe_guard

        scored = apply_catastrophic_flags(scored, thresholds_v2, prefix="gr__")

        gr_cat_rate = scored["gr__is_any_catastrophic"].mean() * 100
        delta_pp = gr_cat_rate - baseline_cat_rate

        new_cat = scored["gr__is_any_catastrophic"].astype(bool) & (~baseline_scored["bl__is_any_catastrophic"].astype(bool))
        new_cat_rate = new_cat.mean() * 100

        pct_triggered_after = (exp_guard < trig).mean() * 100

        rows.append({
            "trigger_lt": float(trig),
            "floor_name": floor_name,
            "floor_value": floor_val,
            "n_rows": int(len(scored)),
            "pct_rows_triggered_baseline": trig_mask_baseline.mean() * 100,
            "pct_rows_triggered_after": pct_triggered_after,
            "baseline_cat_rate_%": baseline_cat_rate,
            "guard_cat_rate_%": gr_cat_rate,
            "delta_cat_rate_pp": delta_pp,
            "new_cat_rate_%": new_cat_rate,
            "median_expected_baseline": float(np.nanmedian(exp0)),
            "median_expected_guard": float(np.nanmedian(exp_guard)),
            "median_oe_baseline": float(np.nanmedian(pd.to_numeric(work0["oe_ratio"], errors="coerce"))),
            "median_oe_guard": float(np.nanmedian(oe_guard)),
            "median_abs_resid_baseline": float(np.nanmedian(pd.to_numeric(work0["abs_residual"], errors="coerce"))),
            "median_abs_resid_guard": float(np.nanmedian(abs_resid_guard)),
        })

        for b in bucket_cols:
            gr_rate = scored[f"gr__{b}"].mean() * 100
            bucket_delta_rows.append({
                "trigger_lt": float(trig),
                "floor_name": floor_name,
                "bucket": b,
                "baseline_rate_%": baseline_bucket_rates[b],
                "guard_rate_%": gr_rate,
                "delta_pp": gr_rate - baseline_bucket_rates[b],
            })

results_df = pd.DataFrame(rows).sort_values(
    ["delta_cat_rate_pp", "new_cat_rate_%", "trigger_lt", "floor_name"],
    ascending=[True, True, True, True]
).reset_index(drop=True)

bucket_deltas_df = pd.DataFrame(bucket_delta_rows)

print("\nA1.3.c) Grid results (sorted by best improvement = most negative delta_cat_rate_pp)")
display(results_df)

best_row = results_df.iloc[0]
print("\nBest config selected:")
display(pd.DataFrame([best_row]))

best_bucket_deltas = (
    bucket_deltas_df.loc[
        (bucket_deltas_df["trigger_lt"] == float(best_row["trigger_lt"])) &
        (bucket_deltas_df["floor_name"] == str(best_row["floor_name"]))
    ]
    .sort_values("delta_pp")
    .reset_index(drop=True)
)

print(f"\nBucket deltas for best config: trigger<{best_row['trigger_lt']} | floor={best_row['floor_name']}")
display(best_bucket_deltas)

#### A1.3.c.1. Materialize best config (A1c dataframe) as `guard_scored_A1c_best`

> This is generated after finding out that A1c (as referred to as in this notebook "") is the best option. 

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# A1.3.c.1) Materialize best A1c config as guard_scored_A1c_best
# Uses:
#   - best_row (from A1.3.c)
#   - floor_candidates (from A1.3.c)
# Produces:
#   - guard_scored_A1c_best (with baseline+guard columns and gr__ flags)
#   - guard_scored_best (alias)
# ============================================================

assert "best_row" in globals(), "best_row not found. Run A1.3.c first."
assert "floor_candidates" in globals(), "floor_candidates not found. Run A1.3.c first."
assert "slice_2023_only" in globals(), "slice_2023_only not found."
assert "baseline_scored" in globals(), "baseline_scored not found."
assert "thresholds_v2" in globals(), "thresholds_v2 not found."
assert "apply_catastrophic_flags" in globals(), "apply_catastrophic_flags not found."

BEST_TRIGGER_LT = float(best_row["trigger_lt"])
BEST_FLOOR_NAME = str(best_row["floor_name"])
BEST_FLOOR_VAL  = float(floor_candidates[BEST_FLOOR_NAME])

print("Materializing best config:")
print("  trigger_lt:", BEST_TRIGGER_LT)
print("  floor_name :", BEST_FLOOR_NAME)
print("  floor_value:", BEST_FLOOR_VAL)

work = slice_2023_only.copy()

# numeric + float64 arrays (avoid object dtype issues)
work["observed_cost"] = pd.to_numeric(work["observed_cost"], errors="coerce")
work["expected_cost"] = pd.to_numeric(work["expected_cost"], errors="coerce")
work["oe_ratio"] = pd.to_numeric(work["oe_ratio"], errors="coerce")
work["abs_residual"] = pd.to_numeric(work["abs_residual"], errors="coerce")

obs0 = work["observed_cost"].to_numpy(dtype="float64")
exp0 = work["expected_cost"].to_numpy(dtype="float64")

mask = exp0 < BEST_TRIGGER_LT

exp_guard = exp0.copy()
raised = np.maximum(exp0[mask], BEST_FLOOR_VAL)
exp_guard[mask] = np.minimum(obs0[mask], raised)

# safety net (cheap, makes invariant ironclad)
exp_guard[mask] = np.minimum(exp_guard[mask], obs0[mask])

EPS = 1e-9
residual_guard = obs0 - exp_guard
abs_resid_guard = np.abs(residual_guard)
oe_guard = obs0 / np.maximum(exp_guard, EPS)

# Attach baseline/guard columns for later diagnostics
work["expected_cost_baseline"] = exp0
work["expected_cost_guard"] = exp_guard
work["oe_ratio_baseline"] = work["oe_ratio"]
work["abs_residual_baseline"] = work["abs_residual"]

work["residual_guard"] = residual_guard
work["abs_residual_guard"] = abs_resid_guard
work["oe_ratio_guard"] = oe_guard

guard_scored_A1c_best = work.copy()
guard_scored_A1c_best["expected_cost"] = guard_scored_A1c_best["expected_cost_guard"]
guard_scored_A1c_best["residual"] = guard_scored_A1c_best["residual_guard"]
guard_scored_A1c_best["abs_residual"] = guard_scored_A1c_best["abs_residual_guard"]
guard_scored_A1c_best["oe_ratio"] = guard_scored_A1c_best["oe_ratio_guard"]

guard_scored_A1c_best = apply_catastrophic_flags(guard_scored_A1c_best, thresholds_v2, prefix="gr__")

# Invariant check (only where modified)
viol = mask & np.isfinite(obs0) & np.isfinite(exp_guard) & (exp_guard > obs0 + 1e-9)
assert int(viol.sum()) == 0, "Invariant violated: expected_cost_guard exceeds observed_cost in modified rows."

# New catastrophes check
base_any = baseline_scored["bl__is_any_catastrophic"].astype(bool).to_numpy()
gr_any = guard_scored_A1c_best["gr__is_any_catastrophic"].astype(bool).to_numpy()
new_cat = gr_any & (~base_any)

print("Invariant PASS: expected_cost_guard <= observed_cost (within tol) for modified rows.")
print("New catastrophes created (%):", round(new_cat.mean() * 100, 6))
print("Baseline cat rate (%):", round(base_any.mean() * 100, 6))
print("Best-config cat rate (%):", round(gr_any.mean() * 100, 6))

# Canonical pin for downstream cells
guard_scored_best = guard_scored_A1c_best

#### Quick consistency check

Right after running A1.3.c and A1.3.c.1, compare against our known prior “expected” values:

In [ ]:
# 1) Best config should match what you saw before
print(best_row[["trigger_lt", "floor_name", "delta_cat_rate_pp", "new_cat_rate_%"]])

# 2) Cat rate should match what you saw for that config previously
baseline_rate = baseline_scored["bl__is_any_catastrophic"].mean() * 100
guard_rate = guard_scored_best["gr__is_any_catastrophic"].mean() * 100
print("baseline_cat_rate_%:", baseline_rate)
print("guard_cat_rate_%   :", guard_rate)
print("delta_pp           :", guard_rate - baseline_rate)

# 3) Sanity: no new catastrophes (A1c property)
new_cat = guard_scored_best["gr__is_any_catastrophic"] & (~baseline_scored["bl__is_any_catastrophic"])
print("new_cat_%:", new_cat.mean() * 100)

# 4) Bucket delta spotlight: cat_oe_q99 should be the main mover
print(
    pd.DataFrame({
        "baseline_cat_oe_q99_%": [baseline_scored["bl__cat_oe_q99"].mean() * 100],
        "guard_cat_oe_q99_%": [guard_scored_best["gr__cat_oe_q99"].mean() * 100],
    })
)

#### A1.3.d. Who improved and why? (baseline -> guardrail transitions)

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# A1.3.d) Who improved and why? (baseline -> guardrail transitions)
# Robust to duplicate column labels (e.g., 'row_id' appearing twice).
# ============================================================

guard_scored_best = guard_scored  # best config df

def dedupe_col_labels(df: pd.DataFrame) -> pd.DataFrame:
    """Keep first occurrence of any duplicate column label."""
    if df.columns.duplicated().any():
        return df.loc[:, ~df.columns.duplicated()].copy()
    return df.copy()

baseline_scored_d = dedupe_col_labels(baseline_scored)
guard_scored_best_d = dedupe_col_labels(guard_scored_best)

# Build column lists without duplicating row_id
bucket_cols = [
    "cat_abs_residual_q99",
    "cat_oe_q99",
    "cat_large_positive_residual",
    "cat_large_negative_residual",
    "cat_large_error_high_support",
    "cat_large_error_tail_row",
    "cat_underprediction",
    "cat_overprediction",
    "cat_high_conf_anomaly",
    "cat_cold_low_support_failure",
    "cat_hot_failure",
    "cat_cold_failure",
]

base_cols = ["row_id", "HCPCS_Cd"] + [c for c in baseline_scored_d.columns if c.startswith("bl__")]

metric_cols = [
    "HCPCS_Cd",
    "expected_cost_baseline", "expected_cost_guard",
    "oe_ratio_baseline", "oe_ratio_guard",
    "abs_residual_baseline", "abs_residual_guard",
]
metric_cols = [c for c in metric_cols if c in guard_scored_best_d.columns]

gr_cols = ["row_id", "HCPCS_Cd"] + [c for c in guard_scored_best_d.columns if c.startswith("gr__")] + metric_cols

# Ensure uniqueness in the column lists (preserve order)
def uniq(seq):
    seen = set()
    out = []
    for x in seq:
        if x not in seen:
            out.append(x)
            seen.add(x)
    return out

base_cols = uniq(base_cols)
gr_cols = uniq(gr_cols)

# Merge (row_id is unique key)
df_join = (
    baseline_scored_d[base_cols]
    .merge(
        guard_scored_best_d[gr_cols],
        on="row_id",
        how="inner",
        validate="one_to_one",
        suffixes=("", "_gr")  # should not be used, but safe
    )
)

# Transitions
base_any = df_join["bl__is_any_catastrophic"].astype(bool)
gr_any   = df_join["gr__is_any_catastrophic"].astype(bool)

improved = base_any & (~gr_any)          # catastrophic -> not catastrophic
worsened = (~base_any) & gr_any          # not catastrophic -> catastrophic
unchanged_cat = base_any & gr_any
unchanged_ok  = (~base_any) & (~gr_any)

print("A1.3.d) Transition counts (2023-only slice)")
print("  improved (cat -> ok):", int(improved.sum()))
print("  worsened (ok -> cat):", int(worsened.sum()))
print("  unchanged_cat (cat -> cat):", int(unchanged_cat.sum()))
print("  unchanged_ok (ok -> ok):", int(unchanged_ok.sum()))
print("  net delta catastrophes:", int(gr_any.sum() - base_any.sum()))

bl_buckets = [f"bl__{b}" for b in bucket_cols if f"bl__{b}" in df_join.columns]
gr_buckets = [f"gr__{b}" for b in bucket_cols if f"gr__{b}" in df_join.columns]

print("\nBaseline bucket rates within IMPROVED rows (what they were failing on):")
if improved.any() and bl_buckets:
    display((df_join.loc[improved, bl_buckets].mean().sort_values(ascending=False) * 100).to_frame("pct_true"))
else:
    print("  No improved rows or missing bucket columns.")

print("\nGuardrail bucket rates within IMPROVED rows (what remains after guardrail):")
if improved.any() and gr_buckets:
    display((df_join.loc[improved, gr_buckets].mean().sort_values(ascending=False) * 100).to_frame("pct_true"))
else:
    print("  No improved rows or missing bucket columns.")

# Inspect improved rows
cols_view = [
    "row_id", "HCPCS_Cd",
    "expected_cost_baseline", "expected_cost_guard",
    "oe_ratio_baseline", "oe_ratio_guard",
    "abs_residual_baseline", "abs_residual_guard",
    "bl__is_any_catastrophic", "gr__is_any_catastrophic",
]
cols_view = [c for c in cols_view if c in df_join.columns]

improved_view = (
    df_join.loc[improved, cols_view]
    .assign(oe_drop=lambda x: x["oe_ratio_baseline"] - x["oe_ratio_guard"])
    .sort_values("oe_drop", ascending=False)
)

print("\nTop 25 improved rows by O/E drop:")
display(improved_view.head(25))

# Quick summary stats for improved rows
needed = {"expected_cost_baseline","expected_cost_guard","oe_ratio_baseline","oe_ratio_guard"}
if improved.any() and needed.issubset(df_join.columns):
    s = pd.DataFrame({
        "metric": [
            "n_rows",
            "median_expected_baseline", "median_expected_guard",
            "median_oe_baseline", "median_oe_guard",
            "pct_expected_baseline_lt1", "pct_expected_guard_lt1",
        ],
        "value": [
            int(improved.sum()),
            df_join.loc[improved, "expected_cost_baseline"].median(),
            df_join.loc[improved, "expected_cost_guard"].median(),
            df_join.loc[improved, "oe_ratio_baseline"].median(),
            df_join.loc[improved, "oe_ratio_guard"].median(),
            (df_join.loc[improved, "expected_cost_baseline"] < 1).mean() * 100,
            (df_join.loc[improved, "expected_cost_guard"] < 1).mean() * 100,
        ]
    })
    print("\nImproved-row summary:")
    display(s)

#### A1.3.e. Soft clamp grid: cap `expected_guard` at `alpha * observed_cost`

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# A1.3.e) Soft clamp grid: cap expected_guard at alpha * observed_cost
# ============================================================

TRAIN_YEARS = [2020, 2021, 2022]
TRIGGER_LT = 1.0
ALPHAS = [0.8, 0.9, 1.0]

# floor candidates from train cold observed_cost
train_cold = read_pq("failure_df_full_v2", cols=["Year", "has_lag", "observed_cost"])
train_cold = train_cold.loc[train_cold["Year"].isin(TRAIN_YEARS) & (~train_cold["has_lag"].astype(bool))].copy()
train_cold["observed_cost"] = pd.to_numeric(train_cold["observed_cost"], errors="coerce")

floor_candidates = {
    "p25": float(train_cold["observed_cost"].quantile(0.25)),
    "median": float(train_cold["observed_cost"].median()),
    "p75": float(train_cold["observed_cost"].quantile(0.75)),
}
print("Train-cold observed_cost floor candidates:", floor_candidates)

if "baseline_scored" not in globals():
    raise ValueError("baseline_scored not found. Build it from slice_2023_only using apply_catastrophic_flags(..., prefix='bl__').")
if "slice_2023_only" not in globals():
    raise ValueError("slice_2023_only not found. Build it from failure_df_full_v2 for Year==2023 & hcpcs_stability_group=='2023_only'.")

# Baseline cat truth aligned by row_id (critical for correct new_cat math)
baseline_any_by_id = (
    baseline_scored.set_index("row_id")["bl__is_any_catastrophic"].astype(bool)
)

bucket_cols = [
    "cat_abs_residual_q99",
    "cat_oe_q99",
    "cat_large_positive_residual",
    "cat_large_negative_residual",
    "cat_large_error_high_support",
    "cat_large_error_tail_row",
    "cat_underprediction",
    "cat_overprediction",
    "cat_high_conf_anomaly",
    "cat_cold_low_support_failure",
    "cat_hot_failure",
    "cat_cold_failure",
]
baseline_bucket_rates = {b: baseline_scored[f"bl__{b}"].mean() * 100 for b in bucket_cols}
baseline_cat_rate = baseline_any_by_id.mean() * 100

work = slice_2023_only.copy()
work["expected_cost_baseline"] = pd.to_numeric(work["expected_cost"], errors="coerce")
work["observed_cost_num"] = pd.to_numeric(work["observed_cost"], errors="coerce")
work["oe_ratio_baseline"] = pd.to_numeric(work["oe_ratio"], errors="coerce")
work["abs_residual_baseline"] = pd.to_numeric(work["abs_residual"], errors="coerce")

trigger_mask = work["expected_cost_baseline"] < TRIGGER_LT

results = []
bucket_deltas_long = []
EPS = 1e-9

for floor_name, floor_val in floor_candidates.items():
    for alpha in ALPHAS:
        tmp = work.copy()

        # Step 1: floor for triggered
        tmp["expected_floor"] = tmp["expected_cost_baseline"]
        tmp.loc[trigger_mask, "expected_floor"] = np.maximum(
            tmp.loc[trigger_mask, "expected_cost_baseline"].to_numpy(),
            floor_val
        )

        # Step 2: cap at alpha * observed (triggered only)
        cap = alpha * tmp["observed_cost_num"]
        tmp["expected_guard"] = tmp["expected_floor"]
        tmp.loc[trigger_mask, "expected_guard"] = np.minimum(
            tmp.loc[trigger_mask, "expected_floor"].to_numpy(),
            cap.loc[trigger_mask].to_numpy()
        )

        # Recompute metrics
        tmp["residual_guard"] = tmp["observed_cost_num"] - tmp["expected_guard"]
        tmp["abs_residual_guard"] = tmp["residual_guard"].abs()
        tmp["oe_ratio_guard"] = tmp["observed_cost_num"] / np.maximum(tmp["expected_guard"].to_numpy(), EPS)

        # Apply baseline thresholds to guard metrics
        scored = tmp.copy()
        scored["expected_cost"] = scored["expected_guard"]
        scored["residual"] = scored["residual_guard"]
        scored["abs_residual"] = scored["abs_residual_guard"]
        scored["oe_ratio"] = scored["oe_ratio_guard"]

        scored = apply_catastrophic_flags(scored, thresholds_v2, prefix="gr__")

        guard_cat_rate = scored["gr__is_any_catastrophic"].mean() * 100
        delta_pp = guard_cat_rate - baseline_cat_rate

        # Correct new_cat computation: align baseline by row_id
        base_any_aligned = baseline_any_by_id.reindex(scored["row_id"]).to_numpy()
        new_cat = scored["gr__is_any_catastrophic"].astype(bool).to_numpy() & (~base_any_aligned)
        new_cat_rate = np.mean(new_cat) * 100

        changed = trigger_mask & (np.abs(tmp["expected_guard"] - tmp["expected_cost_baseline"]) > 1e-12)
        pct_changed = changed.mean() * 100

        results.append({
            "trigger_lt": TRIGGER_LT,
            "floor_name": floor_name,
            "floor_value": floor_val,
            "alpha_cap": alpha,
            "n_rows": len(tmp),
            "pct_rows_triggered_baseline": trigger_mask.mean() * 100,
            "pct_rows_changed": pct_changed,
            "baseline_cat_rate_%": baseline_cat_rate,
            "guard_cat_rate_%": guard_cat_rate,
            "delta_cat_rate_pp": delta_pp,
            "new_cat_rate_%": new_cat_rate,
            "median_expected_baseline": tmp["expected_cost_baseline"].median(),
            "median_expected_guard": tmp["expected_guard"].median(),
            "median_oe_baseline": tmp["oe_ratio_baseline"].median(),
            "median_oe_guard": tmp["oe_ratio_guard"].median(),
            "median_abs_resid_baseline": tmp["abs_residual_baseline"].median(),
            "median_abs_resid_guard": tmp["abs_residual_guard"].median(),
        })

        for b in bucket_cols:
            gr_rate = scored[f"gr__{b}"].mean() * 100
            bucket_deltas_long.append({
                "trigger_lt": TRIGGER_LT,
                "floor_name": floor_name,
                "alpha_cap": alpha,
                "bucket": b,
                "baseline_rate_%": baseline_bucket_rates[b],
                "guard_rate_%": gr_rate,
                "delta_pp": gr_rate - baseline_bucket_rates[b],
            })

res_df = pd.DataFrame(results).sort_values(["delta_cat_rate_pp", "new_cat_rate_%"], ascending=[True, True])
print("\nA1.3.e) Soft clamp grid results (sorted by best improvement = most negative delta_cat_rate_pp):")
display(res_df)

print("\nTop 5 configs:")
display(res_df.head(5))

best = res_df.iloc[0]
best_floor = best["floor_name"]
best_alpha = best["alpha_cap"]
best_trigger = best["trigger_lt"]

bucket_deltas = (
    pd.DataFrame(bucket_deltas_long)
    .query("trigger_lt == @best_trigger and floor_name == @best_floor and alpha_cap == @best_alpha")
    .sort_values("delta_pp")
)

print(f"\nBucket deltas for best config: trigger<{best_trigger} | floor={best_floor} | alpha={best_alpha}")
display(bucket_deltas)

#### A1.3.f. Pin the best configuration explicitly (A1c) + sanity checks

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# A1.3.f) Pin best A1c config explicitly + sanity checks
# Requires:
# - baseline_scored (bl__ flags on slice_2023_only)
# - guard_scored_A1c_best (gr__ flags + expected_cost_guard etc.)
# ============================================================

assert "baseline_scored" in globals(), "baseline_scored not found"
assert "guard_scored_A1c_best" in globals(), "guard_scored_A1c_best not found. Run materialize-best A1c cell first."

# Pin canonical names used throughout the notebook
guard_scored_best = guard_scored_A1c_best
GUARDRAIL_V0_NAME = "A1c_global_floor_bounded_by_observed (trigger<1.0, floor=train_cold_median)"

print("Pinned guardrail:", GUARDRAIL_V0_NAME)
print("guard_scored_best rows:", len(guard_scored_best))

# --- core invariants (A1c guarantees only on modified rows) ---
oc = pd.to_numeric(guard_scored_best["observed_cost"], errors="coerce")
eg = pd.to_numeric(guard_scored_best["expected_cost_guard"], errors="coerce")

# define the "modified rows" mask exactly like A1c (best config is trigger<1.0)
if "expected_cost_baseline" in guard_scored_best.columns:
    exp_base = pd.to_numeric(guard_scored_best["expected_cost_baseline"], errors="coerce")
else:
    # fallback: if baseline column missing, use current expected_cost before guardrail isn't available
    raise ValueError("expected_cost_baseline missing. Re-run A1.3.c.1 materialization to add it.")

TRIGGER_LT = 1.0  # best_row["trigger_lt"] is 1.0 in your case
mask_mod = (exp_base < TRIGGER_LT) & oc.notna() & eg.notna()

print("Rows checked for A1c invariant:", int(mask_mod.sum()))

# A1c invariant: only for modified rows, expected_guard <= observed
assert (eg[mask_mod] <= oc[mask_mod] + 1e-9).all(), "A1c invariant violated within modified rows: expected_cost_guard > observed_cost"
assert (eg[mask_mod] >= 0).all(), "Unexpected: negative expected_cost_guard within modified rows"

# --- no-new-catastrophes check (A1c property) ---
base_any = baseline_scored["bl__is_any_catastrophic"].astype(bool).to_numpy()
gr_any = guard_scored_best["gr__is_any_catastrophic"].astype(bool).to_numpy()
new_cat = gr_any & (~base_any)

print("New catastrophes created (%):", round(new_cat.mean() * 100, 6))

# --- headline metrics ---
baseline_cat_rate = base_any.mean() * 100
guard_cat_rate = gr_any.mean() * 100

print("2023-only catastrophic rate (baseline):", round(baseline_cat_rate, 6), "%")
print("2023-only catastrophic rate (A1c best):", round(guard_cat_rate, 6), "%")
print("Delta (pp):", round(guard_cat_rate - baseline_cat_rate, 6))

# --- bucket deltas ---
bucket_cols = [
    "cat_abs_residual_q99",
    "cat_oe_q99",
    "cat_large_positive_residual",
    "cat_large_negative_residual",
    "cat_large_error_high_support",
    "cat_large_error_tail_row",
    "cat_underprediction",
    "cat_overprediction",
    "cat_high_conf_anomaly",
    "cat_cold_low_support_failure",
    "cat_hot_failure",
    "cat_cold_failure",
]

bucket_compare = []
for b in bucket_cols:
    base_rate = baseline_scored[f"bl__{b}"].mean() * 100
    gr_rate = guard_scored_best[f"gr__{b}"].mean() * 100
    bucket_compare.append({"bucket": b, "baseline_rate_%": base_rate, "A1c_rate_%": gr_rate, "delta_pp": gr_rate - base_rate})

bucket_compare = pd.DataFrame(bucket_compare).sort_values("delta_pp")
display(bucket_compare)

#### A1.4. Freeze A1c as “Guardrail V0 for coverage shift” (store policy + make a reusable function)

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# A1.4) Freeze A1c as Guardrail V0 policy + reusable apply function
# ============================================================

guardrail_v0_policy = {
    "name": "A1c_global_floor_bounded_by_observed",
    "scope": "apply ONLY to Year==2023 & hcpcs_stability_group=='2023_only' & cold_start rows",
    "trigger_expected_lt": BEST_TRIGGER_LT,   # should be 1.0
    "floor_value": BEST_FLOOR_VAL,            # train-cold median (best)
    "epsilon": 1e-9,
}

print("Guardrail V0 policy:")
display(pd.DataFrame([guardrail_v0_policy]))

def apply_guardrail_v0_A1c_2023_only(df: pd.DataFrame, policy: dict) -> pd.DataFrame:
    """
    Apply Guardrail V0 (A1c) to a scored benchmarking dataframe.

    Rule (for eligible rows):
      if expected_cost < trigger:
         expected_guard = min(observed_cost, max(expected_cost, floor))
      else:
         expected_guard = expected_cost

    Eligibility:
      Year==2023 AND hcpcs_stability_group=='2023_only' AND has_lag==False

    Returns df copy with:
      expected_cost_guard, residual_guard, abs_residual_guard, oe_ratio_guard, guardrail_v0_applied
    """
    out = df.copy()

    req = ["Year", "hcpcs_stability_group", "has_lag", "observed_cost", "expected_cost"]
    missing = [c for c in req if c not in out.columns]
    if missing:
        raise ValueError(f"apply_guardrail_v0_A1c_2023_only missing columns: {missing}")

    out["observed_cost"] = pd.to_numeric(out["observed_cost"], errors="coerce")
    out["expected_cost"] = pd.to_numeric(out["expected_cost"], errors="coerce")

    obs = out["observed_cost"].to_numpy(dtype="float64")
    exp = out["expected_cost"].to_numpy(dtype="float64")

    trigger = float(policy["trigger_expected_lt"])
    floor = float(policy["floor_value"])
    EPS = float(policy.get("epsilon", 1e-9))

    eligible = (
        (out["Year"] == 2023)
        & (out["hcpcs_stability_group"] == "2023_only")
        & (~out["has_lag"].astype(bool))
        & np.isfinite(obs)
        & np.isfinite(exp)
    )

    exp_guard = exp.copy()
    mask = eligible.to_numpy() & (exp < trigger)

    raised = np.maximum(exp[mask], floor)
    exp_guard[mask] = np.minimum(obs[mask], raised)

    # safety net ONLY for modified rows (do NOT touch stable/unmodified rows)
    exp_guard[mask] = np.minimum(exp_guard[mask], obs[mask])

    residual_guard = obs - exp_guard
    abs_resid_guard = np.abs(residual_guard)
    oe_guard = obs / np.maximum(exp_guard, EPS)

    out["expected_cost_guard"] = exp_guard
    out["residual_guard"] = residual_guard
    out["abs_residual_guard"] = abs_resid_guard
    out["oe_ratio_guard"] = oe_guard
    out["guardrail_v0_applied"] = eligible

    # Optional quick diagnostics (keep or remove)
    out.attrs["guardrail_v0_mask_count"] = int(mask.sum())
    out.attrs["guardrail_v0_eligible_count"] = int(eligible.sum())

    return out

#### A1.5) Evaluate Guardrail V0 (A1c) side-by-side:

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# A1.5) Evaluate Guardrail V0 (A1c) side-by-side:
# - 2023_only vs stable_all_years (Year=2023 only)
# - Baseline labels vs V0 labels (using fixed thresholds_v2)
# Robust to guardrail_v0_policy being either:
#   - dict
#   - 1-row DataFrame
# ============================================================

CAT_BUCKETS = [
    "cat_abs_residual_q99",
    "cat_oe_q99",
    "cat_large_positive_residual",
    "cat_large_negative_residual",
    "cat_large_error_high_support",
    "cat_large_error_tail_row",
    "cat_underprediction",
    "cat_overprediction",
    "cat_high_conf_anomaly",
    "cat_cold_low_support_failure",
    "cat_hot_failure",
    "cat_cold_failure",
]

GROUPS = ["stable_all_years", "2023_only"]
YEAR_EVAL = 2023

# -----------------------------
# Preconditions
# -----------------------------
assert "thresholds_v2" in globals(), "thresholds_v2 not found"
assert "apply_catastrophic_flags" in globals(), "apply_catastrophic_flags not found"
assert "read_pq" in globals(), "read_pq not found"
assert "guardrail_v0_policy" in globals(), "guardrail_v0_policy not found (from A1.4)"

# -----------------------------
# Pull V0 settings (dict or DF)
# -----------------------------
if isinstance(guardrail_v0_policy, dict):
    p0 = guardrail_v0_policy
elif isinstance(guardrail_v0_policy, pd.DataFrame):
    assert len(guardrail_v0_policy) >= 1, "guardrail_v0_policy DataFrame is empty"
    p0 = guardrail_v0_policy.iloc[0].to_dict()
else:
    raise TypeError(f"guardrail_v0_policy must be dict or DataFrame, got {type(guardrail_v0_policy)}")

TRIGGER_LT = float(p0.get("trigger_expected_lt", 1.0))
FLOOR_VAL  = float(p0.get("floor_value"))
EPS        = float(p0.get("epsilon", 1e-9))
SCOPE_STR  = str(p0.get("scope", ""))

print("Evaluating Guardrail V0 policy:")
print("  name:", p0.get("name"))
print("  scope:", SCOPE_STR)
print("  trigger_expected_lt:", TRIGGER_LT)
print("  floor_value:", FLOOR_VAL)
print("  epsilon:", EPS)

# -----------------------------
# Load minimal Year=2023 slice for groups
# -----------------------------
cols_needed = [
    "row_id", "Year", "HCPCS_Cd", "hcpcs_stability_group",
    "has_lag", "route",
    "expected_cost_support_tier",
    "is_top_1pct_avg_mdcr_stdzd_amt",
    "high_confidence_anomaly_candidate",
    "observed_cost", "expected_cost", "residual", "abs_residual", "oe_ratio",
]

df_2023 = read_pq("failure_df_full_v2", cols=cols_needed)
df_2023 = df_2023.loc[
    (df_2023["Year"] == YEAR_EVAL) & (df_2023["hcpcs_stability_group"].isin(GROUPS))
].copy()

# Numeric coercion
for c in ["observed_cost", "expected_cost", "residual", "abs_residual", "oe_ratio"]:
    df_2023[c] = pd.to_numeric(df_2023[c], errors="coerce")

df_2023["has_lag"] = df_2023["has_lag"].astype(bool)

print("\nYear=2023 rows in scope:", len(df_2023))
display(
    df_2023["hcpcs_stability_group"]
    .value_counts(dropna=False)
    .rename_axis("hcpcs_stability_group")
    .reset_index(name="n_rows")
)

# -----------------------------
# Baseline scoring (bl__)
# -----------------------------
baseline_scored_all = apply_catastrophic_flags(df_2023.copy(), thresholds_v2, prefix="bl__")

# -----------------------------
# Apply V0 to 2023_only cold rows with expected < trigger
# A1c rule: if expected<trigger, raise toward floor but never exceed observed
# -----------------------------
work = df_2023.copy()

obs = work["observed_cost"].to_numpy(dtype="float64")
exp = work["expected_cost"].to_numpy(dtype="float64")

is_2023_only = (work["hcpcs_stability_group"] == "2023_only").to_numpy()
is_cold = (~work["has_lag"].to_numpy())
trigger = (exp < TRIGGER_LT)

mask = is_2023_only & is_cold & trigger

print("Mask share (%):", mask.mean() * 100)
print("Mask count:", int(mask.sum()))

exp_guard = exp.copy()
raised = np.maximum(exp[mask], FLOOR_VAL)
exp_guard[mask] = np.minimum(obs[mask], raised)

# safety net only within mask (do NOT touch stable rows)
exp_guard[mask] = np.minimum(exp_guard[mask], obs[mask])

# recompute under V0
residual_guard = obs - exp_guard
abs_resid_guard = np.abs(residual_guard)
oe_guard = obs / np.maximum(exp_guard, EPS)

work["expected_cost_guard_v0"] = exp_guard
work["residual_guard_v0"] = residual_guard
work["abs_residual_guard_v0"] = abs_resid_guard
work["oe_ratio_guard_v0"] = oe_guard

# Build scored df for guardrail flags
guard_scored_all = work.copy()
guard_scored_all["expected_cost"] = guard_scored_all["expected_cost_guard_v0"]
guard_scored_all["residual"] = guard_scored_all["residual_guard_v0"]
guard_scored_all["abs_residual"] = guard_scored_all["abs_residual_guard_v0"]
guard_scored_all["oe_ratio"] = guard_scored_all["oe_ratio_guard_v0"]

guard_scored_all = apply_catastrophic_flags(guard_scored_all, thresholds_v2, prefix="gr__")

# Sanity: stable_all_years expected unchanged under V0
stable_mask = (work["hcpcs_stability_group"] == "stable_all_years")
stable_unchanged = np.isclose(
    work.loc[stable_mask, "expected_cost_guard_v0"].to_numpy(dtype="float64"),
    df_2023.loc[stable_mask, "expected_cost"].to_numpy(dtype="float64"),
    atol=0, rtol=0, equal_nan=True
).all()
print("\nSanity: stable_all_years expected unchanged under V0:", bool(stable_unchanged))

# Sanity: A1c invariant for the modified slice ONLY
m_mod = mask & np.isfinite(obs) & np.isfinite(exp_guard)
viol_mod = m_mod & (exp_guard > obs + 1e-9)

print(
    "Sanity: A1c invariant holds within mask (expected_guard <= observed):",
    bool((~viol_mod).all())
)

# Optional: show how many were checked
print("Rows in mask:", int(mask.sum()), "| Violations:", int(viol_mod.sum()))

# -----------------------------
# Summaries in "Section G1" style (baseline vs V0)
# -----------------------------
def summarize_rates(df_flags: pd.DataFrame, prefix: str) -> pd.DataFrame:
    cols = [f"{prefix}is_any_catastrophic"] + [f"{prefix}{b}" for b in CAT_BUCKETS]
    out = (
        df_flags.groupby("hcpcs_stability_group", dropna=False)
        .agg(
            n_rows=("row_id", "size"),
            **{c + "_rate": (c, "mean") for c in cols}
        )
        .reset_index()
    )
    rate_cols = [c for c in out.columns if c.endswith("_rate")]
    out[rate_cols] = out[rate_cols] * 100
    return out

def summarize_counts(df_flags: pd.DataFrame, prefix: str) -> pd.DataFrame:
    cols = [f"{prefix}is_any_catastrophic"] + [f"{prefix}{b}" for b in CAT_BUCKETS]
    out = (
        df_flags.groupby("hcpcs_stability_group", dropna=False)[cols]
        .sum()
        .astype(int)
        .reset_index()
    )
    return out

rate_bl = summarize_rates(baseline_scored_all, "bl__").rename(columns=lambda c: c.replace("bl__", "baseline__"))
rate_gr = summarize_rates(guard_scored_all, "gr__").rename(columns=lambda c: c.replace("gr__", "v0__"))

count_bl = summarize_counts(baseline_scored_all, "bl__").rename(columns=lambda c: c.replace("bl__", "baseline__"))
count_gr = summarize_counts(guard_scored_all, "gr__").rename(columns=lambda c: c.replace("gr__", "v0__"))

rate_side_by_side = rate_bl.merge(rate_gr, on=["hcpcs_stability_group", "n_rows"], how="inner")
count_side_by_side = count_bl.merge(count_gr, on=["hcpcs_stability_group"], how="inner")

order = pd.CategoricalDtype(categories=["stable_all_years", "2023_only"], ordered=True)
rate_side_by_side["hcpcs_stability_group"] = rate_side_by_side["hcpcs_stability_group"].astype(order)
rate_side_by_side = rate_side_by_side.sort_values("hcpcs_stability_group")

count_side_by_side["hcpcs_stability_group"] = count_side_by_side["hcpcs_stability_group"].astype(order)
count_side_by_side = count_side_by_side.sort_values("hcpcs_stability_group")

print("\nA1.5) Rate table (%, baseline vs V0) | Year=2023")
with pd.option_context("display.max_columns", None):
    display(rate_side_by_side)

print("\nA1.5) Count table (counts, baseline vs V0) | Year=2023")
with pd.option_context("display.max_columns", None):
    display(count_side_by_side)

delta_any = (
    rate_side_by_side[[
        "hcpcs_stability_group", "n_rows",
        "baseline__is_any_catastrophic_rate",
        "v0__is_any_catastrophic_rate"
    ]]
    .assign(delta_pp=lambda x: x["v0__is_any_catastrophic_rate"] - x["baseline__is_any_catastrophic_rate"])
)
print("\nA1.5) is_any_catastrophic delta (pp) | Year=2023")
display(delta_any)

## Section B. (family-specific priors), starting with the radiopharm and misc/unclassified drug families)

## B1 Radiopharm fix

## B1.a. Apply A1c logic to radiopharm

### B1.1. Compute radiopharm family floor from TRAIN cold rows (2020–2022)

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# B1.1) Radiopharm floor prior from TRAIN years (2020–2022)
# - cold rows only (has_lag == False)
# - radiopharm family only (is_potential_radiopharm_code == True)
# - compute robust stats: p25/median/p75
# ============================================================

TRAIN_YEARS = [2020, 2021, 2022]

# Pull only what we need
train_cols = ["Year", "has_lag", "observed_cost", "is_potential_radiopharm_code"]
train = read_pq("failure_df_full_v2", cols=train_cols)

# Filter: train years + cold rows + radiopharm family
train = train.loc[
    train["Year"].isin(TRAIN_YEARS)
    & (~train["has_lag"].astype(bool))
    & (train["is_potential_radiopharm_code"].astype(bool))
].copy()

train["observed_cost"] = pd.to_numeric(train["observed_cost"], errors="coerce")

# If this ends up empty, that's a signal the flag is too sparse in train.
n_train = len(train)
print("Train cold radiopharm rows used:", n_train)

if n_train == 0:
    raise ValueError(
        "No TRAIN cold radiopharm rows found (2020–2022). "
        "Cannot compute radiopharm floor. Check is_potential_radiopharm_code coverage in train."
    )

radiopharm_floor_candidates = {
    "p25": float(train["observed_cost"].quantile(0.25)),
    "median": float(train["observed_cost"].median()),
    "p75": float(train["observed_cost"].quantile(0.75)),
}

print("Radiopharm floor candidates (train cold observed_cost):", radiopharm_floor_candidates)

# Choose default for B1 (median)
radiopharm_floor = radiopharm_floor_candidates["median"]
print("B1 default radiopharm floor (median):", radiopharm_floor)

# Also print your global floor for clarity (must exist from A1.1)
if "global_cold_floor" in globals():
    print("Global cold floor (A1):", float(global_cold_floor))
else:
    print("NOTE: global_cold_floor not found in globals(). B1.2 expects it for composable floors.")

### B1.2. Apply composable floor (max(global, radiopharm)) to radiopharm 2023-only cold rows, then evaluate

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# B1.2) Apply B1 radiopharm guardrail (composable floor) + evaluate
# Rule (eligible rows, expected < 1):
#   floor_used = max(global_cold_floor, radiopharm_floor)
#   expected_guard = min(observed_cost, max(expected_cost, floor_used))
#
# Scope (initial, safe):
#   Year==2023 & hcpcs_stability_group=='2023_only' & has_lag==False & radiopharm_flag
#
# Evaluate with fixed thresholds_v2 (apples-to-apples)
# ============================================================

assert "thresholds_v2" in globals(), "thresholds_v2 not found"
assert "apply_catastrophic_flags" in globals(), "apply_catastrophic_flags not found"
assert "global_cold_floor" in globals(), "global_cold_floor not found (from A1.1)"
assert "radiopharm_floor" in globals(), "radiopharm_floor not found (from B1.1)"

TRIGGER_LT = 1.0
EPS = 1e-9

floor_used = float(max(global_cold_floor, radiopharm_floor))
print("B1 composable floor_used = max(global_cold_floor, radiopharm_floor):", floor_used)

# --- Load needed columns from baseline artifact ---
cols_needed = [
    "row_id", "Year", "HCPCS_Cd", "hcpcs_stability_group",
    "has_lag",
    "expected_cost_support_tier",
    "is_top_1pct_avg_mdcr_stdzd_amt",
    "high_confidence_anomaly_candidate",
    "is_potential_radiopharm_code",
    "observed_cost", "expected_cost", "residual", "abs_residual", "oe_ratio",
]
df = read_pq("failure_df_full_v2", cols=cols_needed)

# Restrict to Year=2023 & 2023_only (this is your Phase 1 target universe)
df_2023_only = df.loc[
    (df["Year"] == 2023) & (df["hcpcs_stability_group"] == "2023_only")
].copy()

# Ensure numeric
for c in ["observed_cost", "expected_cost", "residual", "abs_residual", "oe_ratio"]:
    df_2023_only[c] = pd.to_numeric(df_2023_only[c], errors="coerce")

df_2023_only["has_lag"] = df_2023_only["has_lag"].astype(bool)
df_2023_only["is_potential_radiopharm_code"] = df_2023_only["is_potential_radiopharm_code"].astype(bool)

print("Year=2023 & 2023_only rows:", len(df_2023_only))
print("Radiopharm rows within 2023_only:", int(df_2023_only["is_potential_radiopharm_code"].sum()))

# --- Baseline scoring (bl__) for BOTH:
# (a) radiopharm-only 2023_only
# (b) all 2023_only
baseline_scored_all = apply_catastrophic_flags(df_2023_only.copy(), thresholds_v2, prefix="bl__")

rad_mask = df_2023_only["is_potential_radiopharm_code"].to_numpy()
baseline_scored_rad = baseline_scored_all.loc[rad_mask].copy()

# --- Apply B1 guardrail only to eligible radiopharm rows ---
work = df_2023_only.copy()

obs = work["observed_cost"].to_numpy(dtype="float64")
exp = work["expected_cost"].to_numpy(dtype="float64")

is_cold = (~work["has_lag"].to_numpy())
is_rad = work["is_potential_radiopharm_code"].to_numpy()
tiny = (exp < TRIGGER_LT)

# Eligibility: 2023_only already enforced by df_2023_only + radiopharm + cold + expected<trigger
mask = is_rad & is_cold & tiny & np.isfinite(obs) & np.isfinite(exp)

print("B1 eligible rows (radiopharm & cold & expected<1):", int(mask.sum()))
print("Eligible share within radiopharm 2023_only (%):",
      (mask.sum() / max(1, is_rad.sum())) * 100)

exp_guard = exp.copy()

raised = np.maximum(exp[mask], floor_used)
exp_guard[mask] = np.minimum(obs[mask], raised)

# Safety net within mask (extra insurance)
exp_guard[mask] = np.minimum(exp_guard[mask], obs[mask])

# Recompute metrics under guardrail
residual_guard = obs - exp_guard
abs_resid_guard = np.abs(residual_guard)
oe_guard = obs / np.maximum(exp_guard, EPS)

work["expected_cost_guard_B1"] = exp_guard
work["residual_guard_B1"] = residual_guard
work["abs_residual_guard_B1"] = abs_resid_guard
work["oe_ratio_guard_B1"] = oe_guard
work["guardrail_B1_applied"] = mask

# Build scored df
guard_scored_all = work.copy()
guard_scored_all["expected_cost"] = guard_scored_all["expected_cost_guard_B1"]
guard_scored_all["residual"] = guard_scored_all["residual_guard_B1"]
guard_scored_all["abs_residual"] = guard_scored_all["abs_residual_guard_B1"]
guard_scored_all["oe_ratio"] = guard_scored_all["oe_ratio_guard_B1"]

guard_scored_all = apply_catastrophic_flags(guard_scored_all, thresholds_v2, prefix="gr__")

guard_scored_rad = guard_scored_all.loc[rad_mask].copy()

# --- Invariant check ONLY on modified rows ---
viol = mask & (exp_guard > obs + 1e-9)
assert int(viol.sum()) == 0, "B1 invariant violated: expected_guard > observed in modified rows"
print("B1 invariant PASS (within modified rows)")

# ============================================================
# Evaluation helpers (same style as A1)
# ============================================================
bucket_cols = [
    "cat_abs_residual_q99",
    "cat_oe_q99",
    "cat_large_positive_residual",
    "cat_large_negative_residual",
    "cat_large_error_high_support",
    "cat_large_error_tail_row",
    "cat_underprediction",
    "cat_overprediction",
    "cat_high_conf_anomaly",
    "cat_cold_low_support_failure",
    "cat_hot_failure",
    "cat_cold_failure",
]

def compare_slice(baseline_df, guard_df, label):
    base_any = baseline_df["bl__is_any_catastrophic"].mean() * 100
    gr_any = guard_df["gr__is_any_catastrophic"].mean() * 100
    new_cat = (
        guard_df["gr__is_any_catastrophic"].astype(bool)
        & (~baseline_df["bl__is_any_catastrophic"].astype(bool))
    ).mean() * 100

    print(f"\n=== B1.2 results: {label} ===")
    print("baseline_cat_rate_%:", round(base_any, 6))
    print("guard_cat_rate_%   :", round(gr_any, 6))
    print("delta_pp           :", round(gr_any - base_any, 6))
    print("new_cat_%          :", round(new_cat, 6))

    bucket_compare = []
    for b in bucket_cols:
        base_rate = baseline_df[f"bl__{b}"].mean() * 100
        gr_rate = guard_df[f"gr__{b}"].mean() * 100
        bucket_compare.append({"bucket": b, "baseline_rate_%": base_rate, "B1_rate_%": gr_rate, "delta_pp": gr_rate - base_rate})

    bucket_compare = pd.DataFrame(bucket_compare).sort_values("delta_pp")
    display(bucket_compare)

# Evaluate radiopharm-only (primary)
compare_slice(baseline_scored_rad, guard_scored_rad, "2023_only radiopharm rows")

# Evaluate all 2023_only (secondary safety check)
compare_slice(baseline_scored_all, guard_scored_all, "ALL 2023_only rows (should not get worse overall)")

### Audit the results: radiopharm is likely to have tiny representatiojn in `2023_only` 

#### Quick audit cell 1 (fixed): list the radiopharm rows inside 2023_only

In [ ]:
import pandas as pd

cols = [
    "row_id","Year","HCPCS_Cd","hcpcs_desc","has_lag","route",
    "expected_cost","observed_cost","oe_ratio","abs_residual",
    "hcpcs_stability_group",
    "is_potential_radiopharm_code",
]

tmp = read_pq("failure_df_full_v2", cols=cols)

# defensive bool casting
tmp["is_potential_radiopharm_code"] = tmp["is_potential_radiopharm_code"].astype(bool)
tmp["has_lag"] = tmp["has_lag"].astype(bool)

rad_2023_only = tmp.loc[
    (tmp["Year"] == 2023)
    & (tmp["hcpcs_stability_group"] == "2023_only")
    & (tmp["is_potential_radiopharm_code"])
].copy()

print("radiopharm rows in 2023_only:", len(rad_2023_only))

if len(rad_2023_only) > 0:
    display(rad_2023_only.sort_values(["HCPCS_Cd","row_id"]).reset_index(drop=True))

    rad_2023_only["expected_cost_num"] = pd.to_numeric(rad_2023_only["expected_cost"], errors="coerce")
    rad_2023_only["observed_cost_num"] = pd.to_numeric(rad_2023_only["observed_cost"], errors="coerce")
    rad_2023_only["oe_ratio_num"] = pd.to_numeric(rad_2023_only["oe_ratio"], errors="coerce")

    display(
        rad_2023_only[["expected_cost_num","observed_cost_num","oe_ratio_num"]]
        .describe(percentiles=[.1,.25,.5,.75,.9])
    )
    print("Share expected_cost < 1:", (rad_2023_only["expected_cost_num"] < 1).mean())
else:
    print("No radiopharm rows found in 2023_only under current flag.")

#### Quick audit cell 2 (fixed): compare our flag vs “A9* prefix”

In [ ]:
cols = ["Year","HCPCS_Cd","hcpcs_stability_group","is_potential_radiopharm_code"]
tmp = read_pq("failure_df_full_v2", cols=cols)

s = tmp.loc[(tmp["Year"]==2023) & (tmp["hcpcs_stability_group"]=="2023_only")].copy()
s["is_A9_prefix"] = s["HCPCS_Cd"].astype(str).str.startswith("A9")
s["is_flag"] = s["is_potential_radiopharm_code"].astype(bool)

print("Counts in 2023_only:")
print("A9* prefix rows:", int(s["is_A9_prefix"].sum()))
print("flagged rows    :", int(s["is_flag"].sum()))
print("A9* AND flagged :", int((s["is_A9_prefix"] & s["is_flag"]).sum()))
print("A9* NOT flagged :", int((s["is_A9_prefix"] & ~s["is_flag"]).sum()))
print("flagged NOT A9* :", int((~s["is_A9_prefix"] & s["is_flag"]).sum()))

# Show examples if there are mismatches
if (s["is_A9_prefix"] & ~s["is_flag"]).any():
    print("\nA9* rows NOT flagged (first 50 distinct codes):")
    display(s.loc[s["is_A9_prefix"] & ~s["is_flag"], ["HCPCS_Cd"]].drop_duplicates().head(50))

if (~s["is_A9_prefix"] & s["is_flag"]).any():
    print("\nFlagged rows NOT A9* (first 50 distinct codes):")
    display(s.loc[~s["is_A9_prefix"] & s["is_flag"], ["HCPCS_Cd"]].drop_duplicates().head(50))

Exactly the story:

- ***Our radiopharm flag is correct** (perfect overlap with A9* in the 2023-only slice).*
- ***B1 didn’t fire because A1c’s trigger is `expected_cost < 1`, and none of the 7 radiopharm rows have `expected_cost < 1`.***

So there isn’t a bug. It’s just that **A1c’s “tiny expected” trigger is aimed at denominator-pathology rows**, and radiopharm 2023-only rows are mostly “normal expected, weird observed” (or vice versa), not “expected near zero.”

## B1.b. Adjusting the guardrails to match the needs of radiopharm errors 

- Trigger if `is_radiopharm AND oe_ratio >= oe_underprediction_cut` (3.0).
- Apply `expected_guard = min(observed, max(expected, floor_composable))`.
- This targets underprediction catastrophes (like our `A9592` row with O/E ~3.59).

### B1.b-1.1 Compute composable floor (global + radiopharm)

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# B1b-1.1) Define composable floor for radiopharm
# floor_used = max(global_cold_floor, radiopharm_floor)
# ============================================================

assert "global_cold_floor" in globals(), "global_cold_floor not found (from A1.1)"
assert "radiopharm_floor" in globals(), "radiopharm_floor not found (from B1.1)"
assert "thresholds_v2" in globals(), "thresholds_v2 not found"

floor_used_B1b = float(max(global_cold_floor, radiopharm_floor))
oe_trigger_cut = float(thresholds_v2.get("oe_underprediction_cut", 3.0))

print("B1b-1 composable floor_used = max(global_cold_floor, radiopharm_floor):", floor_used_B1b)
print("B1b-1 O/E trigger cut (oe_underprediction_cut):", oe_trigger_cut)

### B1b-1.2 Apply + evaluate on radiopharm 2023-only cold rows (and check overall 2023-only)

#### Rebuild slice_2023_only with the radiopharm flag included

In [ ]:
import pandas as pd

# Rebuild slice_2023_only with needed columns for family guardrails
cols_needed = [
    "row_id", "Year", "HCPCS_Cd", "hcpcs_desc", "hcpcs_stability_group",
    "has_lag", "route",
    "expected_cost_support_tier",
    "is_top_1pct_avg_mdcr_stdzd_amt",
    "high_confidence_anomaly_candidate",
    "observed_cost", "expected_cost", "residual", "abs_residual", "oe_ratio",
    # family flags
    "is_potential_radiopharm_code",
    "is_misc_unclassified_drug_code",
]

df = read_pq("failure_df_full_v2", cols=cols_needed)

slice_2023_only = df.loc[
    (df["Year"] == 2023) & (df["hcpcs_stability_group"] == "2023_only")
].copy()

# Rebuild baseline_scored to match the refreshed slice (important!)
baseline_scored = apply_catastrophic_flags(slice_2023_only.copy(), thresholds_v2, prefix="bl__")

print("Rebuilt slice_2023_only with columns:", [c for c in ["is_potential_radiopharm_code","is_misc_unclassified_drug_code"] if c in slice_2023_only.columns])
print("Rows:", len(slice_2023_only))
print("Radiopharm rows:", int(slice_2023_only["is_potential_radiopharm_code"].fillna(False).sum()))

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# B1b-1.2) Apply O/E-triggered floor for radiopharm (bounded by observed)
#
# Trigger:
#   is_radiopharm AND cold_start AND oe_ratio >= 3.0
#
# Guard:
#   expected_guard = min(observed_cost, max(expected_cost, floor_used_B1b))
#
# Then recompute residual/abs_residual/oe_ratio and score using FIXED thresholds_v2.
# ============================================================

assert "slice_2023_only" in globals(), "slice_2023_only not found"
assert "apply_catastrophic_flags" in globals(), "apply_catastrophic_flags not found"
assert "baseline_scored" in globals(), "baseline_scored not found (bl__ flags on slice_2023_only)"
assert "bucket_cols" in globals(), "bucket_cols not found"
assert "thresholds_v2" in globals(), "thresholds_v2 not found"
assert "floor_used_B1b" in globals(), "floor_used_B1b not found"
assert "oe_trigger_cut" in globals(), "oe_trigger_cut not found"

df = slice_2023_only.copy()

# Numeric coercion
for c in ["observed_cost", "expected_cost", "oe_ratio", "abs_residual", "residual"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df["has_lag"] = df["has_lag"].astype(bool)

# Radiopharm flag
RAD_COL = "is_potential_radiopharm_code"
assert RAD_COL in df.columns, f"{RAD_COL} not found in slice_2023_only"

is_radiopharm = df[RAD_COL].astype(bool).to_numpy()
is_cold = (~df["has_lag"].to_numpy())

obs = df["observed_cost"].to_numpy(dtype="float64")
exp = df["expected_cost"].to_numpy(dtype="float64")
oe  = df["oe_ratio"].to_numpy(dtype="float64")

# Trigger: radiopharm & cold & oe >= cut
mask = (
    is_radiopharm
    & is_cold
    & np.isfinite(oe)
    & (oe >= oe_trigger_cut)
    & np.isfinite(obs)
    & np.isfinite(exp)
)

print("Year=2023 & 2023_only rows:", len(df))
print("Radiopharm rows within 2023_only:", int(is_radiopharm.sum()))
print("B1b eligible rows (radiopharm & cold & oe>=cut):", int(mask.sum()))
if int(is_radiopharm.sum()) > 0:
    print("Eligible share within radiopharm 2023_only (%):", round(mask.sum() / is_radiopharm.sum() * 100, 6))
else:
    print("Eligible share within radiopharm 2023_only (%): n/a (no radiopharm rows)")

# Apply guardrail
exp_guard = exp.copy()
raised = np.maximum(exp[mask], float(floor_used_B1b))
exp_guard[mask] = np.minimum(obs[mask], raised)

# Safety net within mask (should be redundant)
exp_guard[mask] = np.minimum(exp_guard[mask], obs[mask])

# Recompute metrics
EPS = float(thresholds_v2.get("epsilon", 1e-9)) if isinstance(thresholds_v2, dict) else 1e-9
residual_guard = obs - exp_guard
abs_resid_guard = np.abs(residual_guard)
oe_guard = obs / np.maximum(exp_guard, EPS)

scored = df.copy()
scored["expected_cost_guard_B1b"] = exp_guard
scored["residual_guard_B1b"] = residual_guard
scored["abs_residual_guard_B1b"] = abs_resid_guard
scored["oe_ratio_guard_B1b"] = oe_guard

# Build a scored df for flagging (overwrite core metric cols)
scored_for_flags = scored.copy()
scored_for_flags["expected_cost"] = scored_for_flags["expected_cost_guard_B1b"]
scored_for_flags["residual"] = scored_for_flags["residual_guard_B1b"]
scored_for_flags["abs_residual"] = scored_for_flags["abs_residual_guard_B1b"]
scored_for_flags["oe_ratio"] = scored_for_flags["oe_ratio_guard_B1b"]

scored_for_flags = apply_catastrophic_flags(scored_for_flags, thresholds_v2, prefix="b1b__")

# Invariant check (only modified rows)
viol = mask & np.isfinite(obs) & np.isfinite(exp_guard) & (exp_guard > obs + 1e-9)
assert int(viol.sum()) == 0, "B1b invariant violated within modified rows: expected_guard > observed"
print("B1b invariant PASS (within modified rows)")

# -----------------------------
# Evaluation helper
# -----------------------------
def compare_bucket_rates(sub_df: pd.DataFrame, base_df: pd.DataFrame, prefix_new: str):
    base_any = base_df["bl__is_any_catastrophic"].astype(bool)
    new_any = sub_df[f"{prefix_new}is_any_catastrophic"].astype(bool)

    out_head = {
        "baseline_cat_rate_%": float(base_any.mean() * 100),
        "guard_cat_rate_%": float(new_any.mean() * 100),
        "delta_pp": float(new_any.mean() * 100 - base_any.mean() * 100),
        "new_cat_%": float((new_any & (~base_any)).mean() * 100),
    }

    rows = []
    for b in bucket_cols:
        base_rate = float(base_df[f"bl__{b}"].mean() * 100)
        new_rate = float(sub_df[f"{prefix_new}{b}"].mean() * 100)
        rows.append({"bucket": b, "baseline_rate_%": base_rate, "B1b_rate_%": new_rate, "delta_pp": new_rate - base_rate})
    return out_head, pd.DataFrame(rows).sort_values("delta_pp")

# -----------------------------
# Evaluate on radiopharm-only subset
# -----------------------------
rad_mask = df[RAD_COL].astype(bool)
baseline_rad = baseline_scored.loc[rad_mask].copy()
scored_rad = scored_for_flags.loc[rad_mask].copy()

head_rad, buckets_rad = compare_bucket_rates(scored_rad, baseline_rad, prefix_new="b1b__")

print("\n=== B1b-1.2 results: 2023_only radiopharm rows ===")
for k, v in head_rad.items():
    print(f"{k}: {v:.6f}" if isinstance(v, float) else f"{k}: {v}")
display(buckets_rad)

# -----------------------------
# Evaluate on ALL 2023-only (should not get worse overall)
# -----------------------------
head_all, buckets_all = compare_bucket_rates(scored_for_flags, baseline_scored, prefix_new="b1b__")

print("\n=== B1b-1.2 results: ALL 2023_only rows (sanity check) ===")
for k, v in head_all.items():
    print(f"{k}: {v:.6f}" if isinstance(v, float) else f"{k}: {v}")
display(buckets_all)

#### Quick confirmation: This will prove the eligible row’s expected didn’t move.



In [ ]:
import pandas as pd
import numpy as np

# --------------------------------------------
# 1) Find the radiopharm evaluation dataframe
# --------------------------------------------
candidates = [
    "df", "slice_2023_only", "slice_2023_only2",
    "work", "tmp", "scored", "guard_scored", "guard_scored_all",
    "guard_scored_b1b", "guard_scored_B1b", "b1b_scored", "b1b1_scored",
    "radiopharm_df", "radiopharm_rows", "radiopharm_slice",
]
found_name = None
found_df = None

for name in candidates:
    if name in globals() and isinstance(globals()[name], pd.DataFrame):
        d = globals()[name]
        # Heuristic: radiopharm flag present + hcpcs_stability_group present
        if ("is_potential_radiopharm_code" in d.columns) and ("hcpcs_stability_group" in d.columns):
            # Prefer one that already has guardrail columns from B1b
            if any(c in d.columns for c in ["expected_cost_guard", "expected_cost_guard_b1b", "expected_cost_guard_B1b", "expected_cost_guard_v0"]):
                found_name, found_df = name, d
                break
            # else keep as fallback
            if found_df is None:
                found_name, found_df = name, d

if found_df is None:
    raise NameError(
        "Couldn't find your B1b dataframe in globals(). "
        "Search your notebook for the dataframe name you used in B1b-1.2 (often `scored` or `guard_scored_*`)."
    )

print(f"Using dataframe: `{found_name}` | shape={found_df.shape}")

# --------------------------------------------
# 2) Build the radiopharm 2023_only slice
# --------------------------------------------
rad = found_df.copy()
rad = rad[(rad["Year"] == 2023) & (rad["hcpcs_stability_group"] == "2023_only") & (rad["is_potential_radiopharm_code"].astype(bool))].copy()
print("radiopharm rows in 2023_only:", len(rad))

# --------------------------------------------
# 3) Find the eligible flag column (or reconstruct it)
# --------------------------------------------
eligible_col = None
for c in ["B1b_eligible", "b1b_eligible", "eligible", "mask_eligible", "is_eligible"]:
    if c in rad.columns:
        eligible_col = c
        break

# Reconstruct eligibility if the column doesn't exist
if eligible_col is None:
    # Use the rule you described: radiopharm & cold & oe_ratio >= 3.0
    oe_cut = thresholds_v2.get("oe_underprediction_cut", 3.0) if "thresholds_v2" in globals() else 3.0
    rad["B1b_eligible_rebuilt"] = (
        rad["is_potential_radiopharm_code"].astype(bool)
        & (~rad["has_lag"].astype(bool))
        & (pd.to_numeric(rad["oe_ratio"], errors="coerce") >= oe_cut)
    )
    eligible_col = "B1b_eligible_rebuilt"

eligible = rad.loc[rad[eligible_col].astype(bool)].copy()
print(f"Eligible rows (via `{eligible_col}`):", len(eligible))

if len(eligible) == 0:
    display(rad[["row_id","HCPCS_Cd","hcpcs_desc","has_lag","expected_cost","observed_cost","oe_ratio"]])
    raise ValueError("No eligible rows found. That would mean your earlier count=1 came from a different df/definition.")

# --------------------------------------------
# 4) Show whether expected changed
# --------------------------------------------
# Figure out which columns represent baseline vs guard expected
exp_base_col = None
for c in ["expected_cost_baseline", "expected_cost_num", "expected_cost_before", "expected_cost"]:
    if c in eligible.columns:
        exp_base_col = c
        break

exp_guard_col = None
for c in ["expected_cost_guard", "expected_cost_guard_b1b", "expected_cost_guard_B1b", "expected_cost_guard_v0", "expected_cost_guardrail"]:
    if c in eligible.columns:
        exp_guard_col = c
        break

if exp_guard_col is None:
    # If you overwrote expected_cost in the guard-scored df, fall back to that
    exp_guard_col = "expected_cost"

eligible["expected_delta"] = pd.to_numeric(eligible[exp_guard_col], errors="coerce") - pd.to_numeric(eligible[exp_base_col], errors="coerce")

cols_show = [
    "row_id","HCPCS_Cd","hcpcs_desc","has_lag",
    exp_base_col, exp_guard_col, "expected_delta",
    "observed_cost","oe_ratio",
]
cols_show = [c for c in cols_show if c in eligible.columns]
display(eligible[cols_show].sort_values("expected_delta", ascending=False))

#### Quick sanity check: Before implementing anything, confirm why floors cannot help radiopharm 2023-only



In [ ]:
import pandas as pd

rad = read_pq("failure_df_full_v2", cols=[
    "Year","hcpcs_stability_group","is_potential_radiopharm_code",
    "HCPCS_Cd","HCPCS","hcpcs_cd",          # include common variants
    "expected_cost","observed_cost","oe_ratio","has_lag"
], verbose=True)

# resolve HCPCS column name
hcpcs_col = None
for c in ["HCPCS_Cd", "HCPCS", "hcpcs_cd"]:
    if c in rad.columns:
        hcpcs_col = c
        break
if hcpcs_col is None:
    raise KeyError(f"No HCPCS code column found. Available cols: {rad.columns.tolist()}")

rad = rad[
    (rad["Year"]==2023)
    & (rad["hcpcs_stability_group"]=="2023_only")
    & (rad["is_potential_radiopharm_code"].astype(bool))
    & (~rad["has_lag"].astype(bool))
].copy()

# numeric coercion
for c in ["expected_cost","observed_cost","oe_ratio"]:
    rad[c] = pd.to_numeric(rad[c], errors="coerce")

print(rad[[hcpcs_col,"expected_cost","observed_cost","oe_ratio"]].sort_values("oe_ratio", ascending=False))

print("pct expected < 1:", (rad["expected_cost"] < 1).mean())
print("pct expected < radiopharm_floor(126):", (rad["expected_cost"] < 126.14636364).mean())

In [ ]:
slice_2023_only.columns

## B2. Misc/unclassified drug fix

### B2.1 Compute misc/unclassified drug floor from TRAIN cold rows (2020–2022)

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# B2.1) Compute misc/unclassified drug floor from TRAIN cold rows (2020–2022)
# Goal: robust floor from observed_cost distribution for misc/unclassified codes
# Uses: failure_df_full_v2.parquet (train years only)
# Requires: is_misc_unclassified_drug_code column exists in failure_df_full_v2
# ============================================================

TRAIN_YEARS = [2020, 2021, 2022]

train_cols = ["Year", "has_lag", "observed_cost", "is_misc_unclassified_drug_code"]
train = read_pq("failure_df_full_v2", cols=train_cols, verbose=True).copy()

# numeric + boolean coercion
train["observed_cost"] = pd.to_numeric(train["observed_cost"], errors="coerce")
train["has_lag"] = train["has_lag"].astype(bool)
train["is_misc_unclassified_drug_code"] = train["is_misc_unclassified_drug_code"].astype(bool)

train_cold_misc = train.loc[
    train["Year"].isin(TRAIN_YEARS)
    & (~train["has_lag"])
    & (train["is_misc_unclassified_drug_code"])
    & train["observed_cost"].notna()
].copy()

n_misc = len(train_cold_misc)
print("Train cold misc/unclassified rows used:", n_misc)

if n_misc == 0:
    raise ValueError(
        "No train cold misc/unclassified rows found. "
        "Check how is_misc_unclassified_drug_code is defined in failure_df_full_v2."
    )

misc_floor_candidates = {
    "p25": float(train_cold_misc["observed_cost"].quantile(0.25)),
    "median": float(train_cold_misc["observed_cost"].median()),
    "p75": float(train_cold_misc["observed_cost"].quantile(0.75)),
}
misc_floor = misc_floor_candidates["median"]

print("Misc/unclassified floor candidates (train cold observed_cost):", misc_floor_candidates)
print("B2 default misc floor (median):", misc_floor)
print("Global cold floor (A1):", global_cold_floor)
print("B2 composable floor_used = max(global, misc):", float(max(global_cold_floor, misc_floor)))

### B2.2 Apply composable floor (max(global, misc)) to misc 2023-only cold rows, then evaluate


This starts with the same safe trigger we used for V0 (only touch expected_cost < 1) and the same safety bound (never exceed observed). That avoids the “new negative residual catastrophes” problem we saw earlier.

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# B2.2) Apply composable floor (max(global, misc)) to 2023-only misc rows
# Rule (A1c-style trigger, bounded by observed):
#   if is_misc AND cold AND expected_cost < 1:
#       expected_guard = min(observed_cost, max(expected_cost, floor_used))
#   else:
#       expected_guard = expected_cost
#
# Evaluate:
# - on misc 2023_only rows
# - and sanity check on ALL 2023_only rows (should not worsen)
#
# Requires:
# - slice_2023_only with is_misc_unclassified_drug_code present
# - baseline_scored (bl__ flags on slice_2023_only)
# - thresholds_v2, apply_catastrophic_flags
# - global_cold_floor from A1.1
# - misc_floor from B2.1
# ============================================================

assert "slice_2023_only" in globals(), "slice_2023_only not found"
assert "baseline_scored" in globals(), "baseline_scored not found (bl__ flags)"
assert "thresholds_v2" in globals(), "thresholds_v2 not found"
assert "apply_catastrophic_flags" in globals(), "apply_catastrophic_flags not found"
assert "global_cold_floor" in globals(), "global_cold_floor not found (A1.1)"
assert "misc_floor" in globals(), "misc_floor not found (B2.1)"

TRIGGER_LT = 1.0
FLOOR_USED = float(max(global_cold_floor, misc_floor))
EPS = 1e-9

print("B2 composable floor_used = max(global_cold_floor, misc_floor):", FLOOR_USED)
print("B2 trigger: expected_cost < ", TRIGGER_LT)

df = slice_2023_only.copy()

# numeric coercion
for c in ["observed_cost", "expected_cost", "oe_ratio", "abs_residual"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["has_lag"] = df["has_lag"].astype(bool)
df["is_misc_unclassified_drug_code"] = df["is_misc_unclassified_drug_code"].astype(bool)

print("Year=2023 & 2023_only rows:", len(df))
print("Misc rows within 2023_only:", int(df["is_misc_unclassified_drug_code"].sum()))

# eligibility mask
is_misc = df["is_misc_unclassified_drug_code"].to_numpy()
is_cold = (~df["has_lag"].to_numpy())
exp0 = df["expected_cost"].to_numpy(dtype="float64")
obs0 = df["observed_cost"].to_numpy(dtype="float64")

eligible = is_misc & is_cold & np.isfinite(exp0) & (exp0 < TRIGGER_LT)

print("B2 eligible rows (misc & cold & expected<1):", int(eligible.sum()))
den = int(df["is_misc_unclassified_drug_code"].sum())
print("Eligible share within misc 2023_only (%):", (eligible.sum() / den * 100) if den > 0 else 0.0)

# apply guardrail
exp_guard = exp0.copy()
raised = np.maximum(exp0[eligible], FLOOR_USED)
exp_guard[eligible] = np.minimum(obs0[eligible], raised)

# safety net within eligible only
exp_guard[eligible] = np.minimum(exp_guard[eligible], obs0[eligible])

# recompute metrics
residual_guard = obs0 - exp_guard
abs_resid_guard = np.abs(residual_guard)
oe_guard = obs0 / np.maximum(exp_guard, EPS)

scored = df.copy()
scored["expected_cost_baseline"] = exp0
scored["expected_cost_guard_B2"] = exp_guard
scored["oe_ratio_baseline"] = df["oe_ratio"]
scored["abs_residual_baseline"] = df["abs_residual"]

scored["expected_cost"] = exp_guard
scored["residual"] = residual_guard
scored["abs_residual"] = abs_resid_guard
scored["oe_ratio"] = oe_guard

scored["B2_eligible"] = eligible

# apply baseline bucket logic to guard metrics
scored = apply_catastrophic_flags(scored, thresholds_v2, prefix="grB2__")

# invariant check on modified rows only
viol = eligible & np.isfinite(obs0) & np.isfinite(exp_guard) & (exp_guard > obs0 + 1e-9)
assert int(viol.sum()) == 0, "B2 invariant violated in eligible rows: expected_guard > observed"
print("B2 invariant PASS (within modified rows)")

# -----------------------------
# Metrics helper
# -----------------------------
bucket_cols = [
    "cat_abs_residual_q99",
    "cat_oe_q99",
    "cat_large_positive_residual",
    "cat_large_negative_residual",
    "cat_large_error_high_support",
    "cat_large_error_tail_row",
    "cat_underprediction",
    "cat_overprediction",
    "cat_high_conf_anomaly",
    "cat_cold_low_support_failure",
    "cat_hot_failure",
    "cat_cold_failure",
]

def compare_subset(name, mask_subset):
    base_any = baseline_scored.loc[mask_subset, "bl__is_any_catastrophic"].astype(bool).to_numpy()
    gr_any = scored.loc[mask_subset, "grB2__is_any_catastrophic"].astype(bool).to_numpy()
    new_cat = gr_any & (~base_any)

    print(f"\n=== B2.2 results: {name} ===")
    print("baseline_cat_rate_%:", round(base_any.mean() * 100, 6))
    print("guard_cat_rate_%   :", round(gr_any.mean() * 100, 6))
    print("delta_pp           :", round((gr_any.mean() - base_any.mean()) * 100, 6))
    print("new_cat_%          :", round(new_cat.mean() * 100, 6))

    rows = []
    for b in bucket_cols:
        br = baseline_scored.loc[mask_subset, f"bl__{b}"].mean() * 100
        gr = scored.loc[mask_subset, f"grB2__{b}"].mean() * 100
        rows.append({"bucket": b, "baseline_rate_%": br, "B2_rate_%": gr, "delta_pp": gr - br})
    display(pd.DataFrame(rows).sort_values("delta_pp"))

# -----------------------------
# Evaluate on misc subset + overall 2023_only
# -----------------------------
misc_mask = scored["is_misc_unclassified_drug_code"].astype(bool)
all_mask = np.ones(len(scored), dtype=bool)

compare_subset("2023_only misc/unclassified rows", misc_mask)
compare_subset("ALL 2023_only rows (sanity check)", all_mask)

> ***There are zero rows in your Year=2023 & hcpcs_stability_group==2023_only slice where is_misc_unclassified_drug_code == True.***

#### Quick audit: confirm the population is empty and inspect what those 2023-only codes actually are.

In [ ]:
import pandas as pd

# 1) Confirm misc flag distribution inside 2023_only
tmp = slice_2023_only.copy()
tmp["is_misc_unclassified_drug_code"] = tmp["is_misc_unclassified_drug_code"].astype(bool)

print("2023_only rows:", len(tmp))
print("misc flag True count:", int(tmp["is_misc_unclassified_drug_code"].sum()))
print("misc flag True share (%):", tmp["is_misc_unclassified_drug_code"].mean() * 100)

# 2) What HCPCS codes exist in 2023_only (top 30 by frequency)
print("\nTop HCPCS codes in 2023_only:")
display(tmp["HCPCS_Cd"].value_counts().head(30).rename_axis("HCPCS_Cd").reset_index(name="n_rows"))

# 3) Sanity: do J3490/J3590/J9999 appear at all in 2023_only?
target_misc_codes = {"J3490", "J3590", "J9999"}
present = sorted(set(tmp["HCPCS_Cd"].dropna().unique()).intersection(target_misc_codes))
print("\nMisc target codes present in 2023_only:", present)

# 4) If you want to see which codes would have been flagged if the flag is code-based
tmp["is_misc_by_code"] = tmp["HCPCS_Cd"].isin(target_misc_codes)
print("is_misc_by_code True count:", int(tmp["is_misc_by_code"].sum()))

#### Find where misc codes actually live: this will tell us which slice to target for B2.

In [ ]:
import pandas as pd

cols = ["Year","hcpcs_stability_group","has_lag","HCPCS_Cd","is_misc_unclassified_drug_code","expected_cost","observed_cost","oe_ratio"]
df = read_pq("failure_df_full_v2", cols=cols, verbose=True).copy()

df["has_lag"] = df["has_lag"].astype(bool)
df["is_misc_unclassified_drug_code"] = df["is_misc_unclassified_drug_code"].astype(bool)
for c in ["expected_cost","observed_cost","oe_ratio"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

summary = (
    df[df["is_misc_unclassified_drug_code"]]
    .assign(is_cold=lambda x: ~x["has_lag"])
    .groupby(["Year","hcpcs_stability_group","is_cold"], dropna=False)
    .size()
    .rename("n_rows")
    .reset_index()
    .sort_values(["Year","hcpcs_stability_group","is_cold"])
)

display(summary)

> WE ARE STOPPING MISC/UNCLASSIFIED DRUG FIX HERE. THE 2023-ONLY HCPCS ROWS DO NOT CONTAIN THIS FLAG! THIS IS A DEAD END FOR THIS BRANCH OF INVESTIGATION. HOWEVER, WE'LL PIVOT TO TACKLE THE HIGHEST ROI FIXES! 

## Concrete step to confirm ROI and stop guessing: produce a ranked “pain leaderboard” for 2023-only.

In [ ]:
import numpy as np
import pandas as pd

# Preconditions: slice_2023_only, thresholds_v2, apply_catastrophic_flags exist
sc = apply_catastrophic_flags(slice_2023_only.copy(), thresholds_v2, prefix="bl__")

by_code = (
    sc.groupby("HCPCS_Cd", dropna=False)
    .agg(
        n_rows=("row_id","size"),
        cat_rate=("bl__is_any_catastrophic","mean"),
        oe_q99_rate=("bl__cat_oe_q99","mean"),
        abs_resid_q99_rate=("bl__cat_abs_residual_q99","mean"),
        underpred_rate=("bl__cat_underprediction","mean"),
        cold_low_support_rate=("bl__cat_cold_low_support_failure","mean"),
    )
    .reset_index()
)

# Add burden metrics
by_code["cat_rate_pct"] = by_code["cat_rate"] * 100
by_code["cat_burden_rows"] = (by_code["n_rows"] * by_code["cat_rate"]).round(2)

# Top by volume, top by burden
top_volume = by_code.sort_values("n_rows", ascending=False).head(15)
top_burden = by_code.sort_values("cat_burden_rows", ascending=False).head(15)

print("Top 15 2023-only codes by volume:")
display(top_volume[["HCPCS_Cd","n_rows","cat_rate_pct","cat_burden_rows","oe_q99_rate","abs_resid_q99_rate","underpred_rate"]])

print("Top 15 2023-only codes by catastrophic burden (n_rows * cat_rate):")
display(top_burden[["HCPCS_Cd","n_rows","cat_rate_pct","cat_burden_rows","oe_q99_rate","abs_resid_q99_rate","underpred_rate"]])

## B2.0. Drill down into the worst codes and confirm the failure mode

In [ ]:
import numpy as np
import pandas as pd

TOP_CODES = ["Q5127", "Q5126", "J1449"]  # from your leaderboard

sc = apply_catastrophic_flags(slice_2023_only.copy(), thresholds_v2, prefix="bl__")

bucket_cols = [
    "cat_abs_residual_q99",
    "cat_oe_q99",
    "cat_large_positive_residual",
    "cat_large_negative_residual",
    "cat_large_error_high_support",
    "cat_large_error_tail_row",
    "cat_underprediction",
    "cat_overprediction",
    "cat_high_conf_anomaly",
    "cat_cold_low_support_failure",
    "cat_hot_failure",
    "cat_cold_failure",
]

rows = []
for code in TOP_CODES:
    sub = sc[sc["HCPCS_Cd"] == code].copy()
    if len(sub) == 0:
        continue

    # numeric coercion
    for c in ["expected_cost", "observed_cost", "oe_ratio", "abs_residual"]:
        sub[c] = pd.to_numeric(sub[c], errors="coerce")

    r = {
        "HCPCS_Cd": code,
        "n_rows": len(sub),
        "cat_rate_%": sub["bl__is_any_catastrophic"].mean() * 100,
        "median_expected": sub["expected_cost"].median(),
        "pct_expected_lt_1": (sub["expected_cost"] < 1).mean() * 100,
        "median_observed": sub["observed_cost"].median(),
        "median_oe": sub["oe_ratio"].median(),
        "pct_oe_ge_3": (sub["oe_ratio"] >= thresholds_v2["oe_underprediction_cut"]).mean() * 100,
    }

    for b in bucket_cols:
        r[f"{b}_rate_%"] = sub[f"bl__{b}"].mean() * 100

    rows.append(r)

code_diag = pd.DataFrame(rows).sort_values("cat_rate_%", ascending=False)
with pd.option_context("display.max_columns", None):
    display(code_diag)

## B2.1 and B2.2 (first real guardrail experiment)

### B2.1. Compute a code-specific cold-start floor from train cold rows


We compute floors per-code from train cold `observed_cost` (2020–2022, has_lag==False). If the code does not exist in train (likely for 2023-only), we fallback to a “nearest proxy” floor. For now, simplest is: use the global cold floor (our A1 floor) as fallback, but we’ll structure it so later we can plug in a “family proxy.”

In [ ]:
import numpy as np
import pandas as pd

TRAIN_YEARS = [2020, 2021, 2022]
TARGET_CODES = ["Q5127", "Q5126"]

train = read_pq(
    "failure_df_full_v2",
    cols=["Year", "has_lag", "HCPCS_Cd", "observed_cost"]
).copy()

train["has_lag"] = train["has_lag"].astype(bool)
train["observed_cost"] = pd.to_numeric(train["observed_cost"], errors="coerce")

train_cold = train[(train["Year"].isin(TRAIN_YEARS)) & (~train["has_lag"])].copy()

# Code-specific floors (median observed_cost among train cold rows for that code)
code_floor = (
    train_cold[train_cold["HCPCS_Cd"].isin(TARGET_CODES)]
    .groupby("HCPCS_Cd")["observed_cost"]
    .median()
    .to_dict()
)

print("Train-cold code-specific floors found:", code_floor)
print("Global cold floor (A1):", global_cold_floor)

# Define floor lookup with fallback to global
def get_floor_for_code(hcpcs: str) -> float:
    return float(code_floor.get(hcpcs, global_cold_floor))

### B2.2. Apply code-targeted A1c-style rule, but scoped to these codes


We keep our “no new catastrophes” philosophy by bounding by observed.

Key design choice: trigger condition. For `Q5126` / `Q5127`, our bucket shows `cat_oe_q99` is 100%, so it may be better to trigger on O/E >= 3 (or on expected < 1, or both). Let’s start with a conservative AND:
- eligible if `HCPCS_Cd` in `TARGET_CODES AND cold AND (expected_cost < 1 OR oe_ratio >= 3.0)`


In [ ]:
import numpy as np
import pandas as pd

TARGET_CODES = ["Q5127", "Q5126"]
TRIGGER_EXP_LT = 1.0
TRIGGER_OE_GE = thresholds_v2["oe_underprediction_cut"]  # 3.0
EPS = 1e-9

# Baseline scored (bl__) for comparison
baseline_scored = apply_catastrophic_flags(slice_2023_only.copy(), thresholds_v2, prefix="bl__")

work = slice_2023_only.copy()
for c in ["observed_cost","expected_cost","oe_ratio","abs_residual","residual"]:
    if c in work.columns:
        work[c] = pd.to_numeric(work[c], errors="coerce")

obs = work["observed_cost"].to_numpy(dtype="float64")
exp = work["expected_cost"].to_numpy(dtype="float64")
oe  = work["oe_ratio"].to_numpy(dtype="float64")
is_cold = (~work["has_lag"].astype(bool)).to_numpy()
is_target = work["HCPCS_Cd"].isin(TARGET_CODES).to_numpy()

eligible = is_target & is_cold & ( (exp < TRIGGER_EXP_LT) | (oe >= TRIGGER_OE_GE) ) & np.isfinite(obs) & np.isfinite(exp)

print("Eligible rows total:", int(eligible.sum()))
print("Eligible by code:")
display(
    work.loc[eligible, "HCPCS_Cd"]
    .value_counts()
    .rename_axis("HCPCS_Cd")
    .reset_index(name="n_rows")
)

# Apply composable floor (currently: max(global, code_floor_or_global))
exp_guard = exp.copy()

# Per-row floor lookup (vectorized via map)
floors = np.array([get_floor_for_code(c) for c in work["HCPCS_Cd"]], dtype="float64")
floor_used = np.maximum(global_cold_floor, floors)  # composable placeholder
raised = np.maximum(exp, floor_used)

# A1c-style bounding: never exceed observed (and only apply to eligible)
exp_guard[eligible] = np.minimum(obs[eligible], raised[eligible])

# Safety net (only within eligible)
exp_guard[eligible] = np.minimum(exp_guard[eligible], obs[eligible])

# Recompute metrics
res_guard = obs - exp_guard
abs_guard = np.abs(res_guard)
oe_guard  = obs / np.maximum(exp_guard, EPS)

guard = work.copy()
guard["expected_cost_baseline"] = exp
guard["expected_cost_guard"] = exp_guard
guard["oe_ratio_baseline"] = oe
guard["oe_ratio_guard"] = oe_guard
guard["abs_residual_baseline"] = work["abs_residual"]
guard["abs_residual_guard"] = abs_guard
guard["B2_eligible"] = eligible

# Build scored df for guardrail comparison
guard_scored = guard.copy()
guard_scored["expected_cost"] = guard_scored["expected_cost_guard"]
guard_scored["residual"] = res_guard
guard_scored["abs_residual"] = abs_guard
guard_scored["oe_ratio"] = oe_guard

guard_scored = apply_catastrophic_flags(guard_scored, thresholds_v2, prefix="grB2__")

# Evaluate on target codes only + overall 2023-only sanity
def report_slice(title, mask_slice):
    base_any = baseline_scored.loc[mask_slice, "bl__is_any_catastrophic"].astype(bool)
    gr_any = guard_scored.loc[mask_slice, "grB2__is_any_catastrophic"].astype(bool)
    new_cat = gr_any & (~base_any)

    print(f"\n=== {title} ===")
    print("n_rows:", int(mask_slice.sum()))
    print("baseline_cat_rate_%:", round(base_any.mean() * 100, 6))
    print("guard_cat_rate_%   :", round(gr_any.mean() * 100, 6))
    print("delta_pp           :", round((gr_any.mean() - base_any.mean()) * 100, 6))
    print("new_cat_%          :", round(new_cat.mean() * 100, 6))

mask_target_rows = work["HCPCS_Cd"].isin(TARGET_CODES)
report_slice("B2 results: TARGET codes within 2023_only", mask_target_rows)

report_slice("B2 results: ALL 2023_only rows (sanity)", work["HCPCS_Cd"].notna())

### B2.2b. The right B2 idea for these codes: constraint-based floors

This will actually trigger for `Q5126` / `Q5127` and should reduce `cat_oe_q99` (and for `Q5127` also reduce residual-based cats when feasible).

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# B2.2b) Constraint-based guardrail for top-burden 2023-only codes (Q5127/Q5126)
# Trigger: (HCPCS in TARGET_CODES) & cold-start & (oe_ratio >= oe_q99 baseline threshold)
# Guard: expected_guard = min(obs, max(exp, floor_needed))
# where floor_needed = max(
#   global_cold_floor,
#   obs / oe_q99,
#   obs - resid_pos_q99,
#   obs - abs_resid_q99
# )
# ============================================================

TARGET_CODES = ["Q5127", "Q5126"]
EPS = 1e-9

oe_q99 = float(thresholds_v2["oe_q99"])
abs_resid_q99 = float(thresholds_v2["abs_resid_q99"])
resid_pos_q99 = float(thresholds_v2["resid_pos_q99"])

# Baseline scored for comparison
baseline_scored = apply_catastrophic_flags(slice_2023_only.copy(), thresholds_v2, prefix="bl__")

work = slice_2023_only.copy()
for c in ["observed_cost", "expected_cost", "oe_ratio", "abs_residual", "residual"]:
    work[c] = pd.to_numeric(work[c], errors="coerce")

obs = work["observed_cost"].to_numpy(dtype="float64")
exp = work["expected_cost"].to_numpy(dtype="float64")
oe  = work["oe_ratio"].to_numpy(dtype="float64")

is_cold = (~work["has_lag"].astype(bool)).to_numpy()
is_target = work["HCPCS_Cd"].isin(TARGET_CODES).to_numpy()

trigger = (oe >= oe_q99)

eligible = is_target & is_cold & trigger & np.isfinite(obs) & np.isfinite(exp) & np.isfinite(oe)

print("Eligible rows total:", int(eligible.sum()))
print("Eligible by code:")
display(
    work.loc[eligible, "HCPCS_Cd"]
    .value_counts()
    .rename_axis("HCPCS_Cd")
    .reset_index(name="n_rows")
)

# Build per-row floor requirements (only matters for eligible, but we compute vectorized)
need_for_oe = obs / max(oe_q99, EPS)
need_for_resid_pos = obs - resid_pos_q99
need_for_abs = obs - abs_resid_q99

floor_needed = np.maximum.reduce([
    np.full_like(obs, float(global_cold_floor), dtype="float64"),
    need_for_oe,
    need_for_resid_pos,
    need_for_abs,
])

# Apply guardrail only on eligible rows, and never exceed observed
exp_guard = exp.copy()
raised = np.maximum(exp, floor_needed)
exp_guard[eligible] = np.minimum(obs[eligible], raised[eligible])

# Safety net within eligible
exp_guard[eligible] = np.minimum(exp_guard[eligible], obs[eligible])

# Recompute metrics under guardrail
res_guard = obs - exp_guard
abs_guard = np.abs(res_guard)
oe_guard  = obs / np.maximum(exp_guard, EPS)

guard = work.copy()
guard["expected_cost_baseline"] = exp
guard["expected_cost_guard"] = exp_guard
guard["oe_ratio_baseline"] = oe
guard["oe_ratio_guard"] = oe_guard
guard["abs_residual_baseline"] = work["abs_residual"]
guard["abs_residual_guard"] = abs_guard
guard["B2b_eligible"] = eligible

guard_scored = guard.copy()
guard_scored["expected_cost"] = guard_scored["expected_cost_guard"]
guard_scored["residual"] = res_guard
guard_scored["abs_residual"] = abs_guard
guard_scored["oe_ratio"] = oe_guard

guard_scored = apply_catastrophic_flags(guard_scored, thresholds_v2, prefix="grB2b__")

# Invariant check within modified rows
m = eligible & np.isfinite(obs) & np.isfinite(exp_guard)
viol = m & (exp_guard > obs + 1e-9)
assert int(viol.sum()) == 0, "Invariant violated: expected_guard > observed within eligible rows."
print("Invariant PASS within eligible rows")

# --- Evaluation helpers ---
bucket_cols = [
    "cat_abs_residual_q99",
    "cat_oe_q99",
    "cat_large_positive_residual",
    "cat_large_negative_residual",
    "cat_large_error_high_support",
    "cat_large_error_tail_row",
    "cat_underprediction",
    "cat_overprediction",
    "cat_high_conf_anomaly",
    "cat_cold_low_support_failure",
    "cat_hot_failure",
    "cat_cold_failure",
]

def eval_slice(title, mask_slice):
    base_any = baseline_scored.loc[mask_slice, "bl__is_any_catastrophic"].astype(bool)
    gr_any   = guard_scored.loc[mask_slice, "grB2b__is_any_catastrophic"].astype(bool)
    new_cat  = gr_any & (~base_any)

    print(f"\n=== {title} ===")
    print("n_rows:", int(mask_slice.sum()))
    print("baseline_cat_rate_%:", round(base_any.mean() * 100, 6))
    print("guard_cat_rate_%   :", round(gr_any.mean() * 100, 6))
    print("delta_pp           :", round((gr_any.mean() - base_any.mean()) * 100, 6))
    print("new_cat_%          :", round(new_cat.mean() * 100, 6))

    bucket_compare = []
    for b in bucket_cols:
        bucket_compare.append({
            "bucket": b,
            "baseline_rate_%": baseline_scored.loc[mask_slice, f"bl__{b}"].mean() * 100,
            "guard_rate_%": guard_scored.loc[mask_slice, f"grB2b__{b}"].mean() * 100,
        })
    out = pd.DataFrame(bucket_compare)
    out["delta_pp"] = out["guard_rate_%"] - out["baseline_rate_%"]
    display(out.sort_values("delta_pp"))

mask_target = work["HCPCS_Cd"].isin(TARGET_CODES).to_numpy()
eval_slice("B2.2b results: TARGET codes (Q5126/Q5127) within 2023_only", mask_target)

eval_slice("B2.2b results: ALL 2023_only rows (sanity)", np.ones(len(work), dtype=bool))

# After B2.2b finishes
guard_scored_B2_2b = guard_scored
baseline_scored_B2_2b = baseline_scored

### Remaining-cat rows bucket breakdown (TARGET codes)

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# Transition types within TARGET codes (baseline -> guard) for B2.2b
# Requires:
#   baseline_scored_B2_2b (bl__ flags)
#   guard_scored_B2_2b    (grB2b__ flags)
#   TARGET_CODES
# ============================================================

assert "baseline_scored_B2_2b" in globals(), "baseline_scored_B2_2b not found. Run B2.2b cell and aliases."
assert "guard_scored_B2_2b" in globals(), "guard_scored_B2_2b not found. Run B2.2b cell and aliases."
assert "TARGET_CODES" in globals(), "TARGET_CODES not found (should be ['Q5127','Q5126'])."

bl = baseline_scored_B2_2b.copy()
gr = guard_scored_B2_2b.copy()

# Ensure key columns exist
for c in ["row_id", "HCPCS_Cd"]:
    assert c in bl.columns and c in gr.columns, f"Missing {c} in baseline/guard df."

# Use row_id join to align
dfj = (
    bl[["row_id", "HCPCS_Cd", "bl__is_any_catastrophic"]]
    .merge(
        gr[["row_id", "HCPCS_Cd", "grB2b__is_any_catastrophic", "B2b_eligible"]],
        on="row_id",
        how="inner",
        suffixes=("_bl", "_gr"),
        validate="many_to_one"  # row_id can repeat on baseline side, but guard should be unique per row_id typically
    )
)

# Target mask (use guard HCPCS if present)
mask_target = dfj["HCPCS_Cd_gr"].isin(TARGET_CODES) if "HCPCS_Cd_gr" in dfj.columns else dfj["HCPCS_Cd"].isin(TARGET_CODES)
df_t = dfj.loc[mask_target].copy()

base_any = df_t["bl__is_any_catastrophic"].astype(bool)
gr_any   = df_t["grB2b__is_any_catastrophic"].astype(bool)

df_t["transition"] = np.select(
    [
        base_any & (~gr_any),          # cat -> ok
        (~base_any) & gr_any,          # ok -> cat
        base_any & gr_any,             # cat -> cat
        (~base_any) & (~gr_any),       # ok -> ok
    ],
    [
        "cat->ok",
        "ok->cat",
        "cat->cat",
        "ok->ok",
    ],
    default="unknown"
)

# Summary table (counts + rates)
summary = (
    df_t.groupby("transition", dropna=False)
    .size()
    .rename("n_rows")
    .reset_index()
    .assign(pct=lambda x: x["n_rows"] / x["n_rows"].sum() * 100)
    .sort_values("n_rows", ascending=False)
)

print("Transitions within TARGET codes:", TARGET_CODES)
print("n_TARGET_rows:", len(df_t))
display(summary)

# By-code transition breakdown
by_code = (
    df_t.groupby(["HCPCS_Cd_gr", "transition"], dropna=False)
    .size()
    .rename("n_rows")
    .reset_index()
)

# Pivot to a wide table
by_code_wide = (
    by_code.pivot_table(index="HCPCS_Cd_gr", columns="transition", values="n_rows", fill_value=0, aggfunc="sum")
    .reset_index()
)

# Add totals + cat rates baseline vs guard for each code
code_rates = (
    df_t.groupby("HCPCS_Cd_gr", dropna=False)
    .agg(
        n_rows=("row_id", "size"),
        baseline_cat_rate=("bl__is_any_catastrophic", "mean"),
        guard_cat_rate=("grB2b__is_any_catastrophic", "mean"),
        eligible_rate=("B2b_eligible", lambda s: s.astype(bool).mean() if s.notna().any() else np.nan),
    )
    .reset_index()
)
code_rates["baseline_cat_rate_%"] = code_rates["baseline_cat_rate"] * 100
code_rates["guard_cat_rate_%"] = code_rates["guard_cat_rate"] * 100
code_rates["eligible_rate_%"] = code_rates["eligible_rate"] * 100

out = code_rates.merge(by_code_wide, on="HCPCS_Cd_gr", how="left").sort_values("n_rows", ascending=False)

print("\nBy-code transition breakdown + cat rates:")
with pd.option_context("display.max_columns", None):
    display(out)

# Optional: show the "ok->cat" rows if any (should be 0 if your new_cat_% was 0)
ok_to_cat = df_t.loc[df_t["transition"] == "ok->cat", ["row_id", "HCPCS_Cd_gr", "B2b_eligible"]].copy()
if len(ok_to_cat) == 0:
    print("\nNo ok->cat transitions within TARGET codes (good).")
else:
    print("\nWARNING: ok->cat transitions exist within TARGET codes:")
    display(ok_to_cat.head(50))

**1) Our guardrail only helped. It did not create any new catastrophes in the TARGET codes.**

- Transitions: **48.08% cat->ok**, **51.92% cat->cat**, **0% ok->cat**.

**2) The improvement is entirely coming from `Q5126`.**

- **`Q5126`:** went from **100% catastrophic (25/25)** to **0% catastrophic (0/25)**.
- **`Q5127`:** stayed **100% catastrophic (27/27)**.

So B2.2b is working, but only for one of the two “worst offenders”.

### Q5127 audit (compact)

In [ ]:
import numpy as np
import pandas as pd
import re

# ============================================================
# Q5127 audit (robust):
# - filters HCPCS_Cd == "Q5127"
# - bucket breakdown among remaining-cat (post-guard)
# - shows eligibility + key metrics for top 10 worst rows
#
# Requires:
#   baseline_scored_B2_2b (baseline df with bl__ flags)
#   guard_scored_B2_2b    (guard df with some *guard* flags + B2b_eligible)
#
# This cell AUTO-DETECTS the guard prefix (grB2b__/gr__/etc).
# ============================================================

assert "baseline_scored_B2_2b" in globals(), "baseline_scored_B2_2b not found."
assert "guard_scored_B2_2b" in globals(), "guard_scored_B2_2b not found."

bl = baseline_scored_B2_2b.copy()
gr = guard_scored_B2_2b.copy()

for need in ["row_id", "HCPCS_Cd"]:
    assert need in bl.columns and need in gr.columns, f"Missing {need} in baseline/guard dfs."

# -----------------------------
# 1) Detect guard prefix (e.g., 'grB2b__' or 'gr__')
# -----------------------------
guard_any_candidates = [c for c in gr.columns if c.endswith("is_any_catastrophic")]
# Prefer the one that contains '__is_any_catastrophic' (your typical pattern)
guard_any_candidates2 = [c for c in gr.columns if re.search(r"__is_any_catastrophic$", c)]
guard_any_col = None

if guard_any_candidates2:
    # If multiple, choose the longest prefix (usually most specific like grB2b__)
    guard_any_col = sorted(guard_any_candidates2, key=len, reverse=True)[0]
elif guard_any_candidates:
    guard_any_col = sorted(guard_any_candidates, key=len, reverse=True)[0]
else:
    raise AssertionError(
        "Could not find ANY guard-side is_any_catastrophic column in guard_scored_B2_2b.\n"
        "Search gr.columns for something like 'grB2b__is_any_catastrophic' or 'gr__is_any_catastrophic'."
    )

# Guard prefix is everything before 'is_any_catastrophic'
guard_prefix = guard_any_col.replace("is_any_catastrophic", "")
print("Detected guard is_any column:", guard_any_col)
print("Detected guard prefix:", repr(guard_prefix))

# -----------------------------
# 2) Define bucket base names, then build actual guard bucket columns
# -----------------------------
bucket_basenames = [
    "cat_abs_residual_q99",
    "cat_oe_q99",
    "cat_large_positive_residual",
    "cat_large_negative_residual",
    "cat_large_error_high_support",
    "cat_large_error_tail_row",
    "cat_underprediction",
    "cat_overprediction",
    "cat_high_conf_anomaly",
    "cat_cold_low_support_failure",
    "cat_hot_failure",
    "cat_cold_failure",
]

guard_bucket_cols = [f"{guard_prefix}{b}" for b in bucket_basenames if f"{guard_prefix}{b}" in gr.columns]
missing_guard_buckets = [f"{guard_prefix}{b}" for b in bucket_basenames if f"{guard_prefix}{b}" not in gr.columns]

print("Guard buckets found:", len(guard_bucket_cols))
if missing_guard_buckets:
    print("Guard buckets missing (ok if your helper didn't create all):", missing_guard_buckets[:6], ("..." if len(missing_guard_buckets) > 6 else ""))

# Eligibility column
elig_col = None
for c in ["B2b_eligible", "b2b_eligible", "eligible", "mask_eligible", "is_eligible"]:
    if c in gr.columns:
        elig_col = c
        break
print("Eligibility column:", elig_col)

# -----------------------------
# 3) Filter to Q5127 and align baseline + guard on row_id
# -----------------------------
bl_q = bl.loc[bl["HCPCS_Cd"] == "Q5127"].copy()
gr_q = gr.loc[gr["HCPCS_Cd"] == "Q5127"].copy()

print("\nQ5127 rows (baseline):", len(bl_q))
print("Q5127 rows (guard):   ", len(gr_q))

# If duplicated row_id exists, keep first for this audit
if bl_q["row_id"].duplicated().any():
    bl_q = bl_q.sort_values("row_id").drop_duplicates("row_id", keep="first")
if gr_q["row_id"].duplicated().any():
    gr_q = gr_q.sort_values("row_id").drop_duplicates("row_id", keep="first")

df = bl_q.merge(
    gr_q,
    on="row_id",
    how="inner",
    suffixes=("_bl", "_gr"),
    validate="one_to_one"
)

# After merge, guard_any_col becomes guard_any_col + "_gr" if it existed in both or only in guard
guard_any_merged = guard_any_col + "_gr" if (guard_any_col + "_gr") in df.columns else guard_any_col
assert guard_any_merged in df.columns, f"Guard is_any column not found after merge. Tried {guard_any_merged}"

remain_mask = df[guard_any_merged].astype(bool)
print("\nQ5127 remaining-cat rows (post-guard):", int(remain_mask.sum()), "/", len(df), f"({remain_mask.mean()*100:.2f}%)")

# -----------------------------
# 4) Bucket breakdown among remaining-cat rows (post-guard)
# -----------------------------
guard_bucket_cols_merged = []
for c in guard_bucket_cols:
    c_m = c + "_gr" if (c + "_gr") in df.columns else c
    if c_m in df.columns:
        guard_bucket_cols_merged.append(c_m)

if remain_mask.any() and guard_bucket_cols_merged:
    bucket_rates = (df.loc[remain_mask, guard_bucket_cols_merged].mean().sort_values(ascending=False) * 100).to_frame("pct_true_in_remaining_cat")
    print("\nBucket breakdown within remaining-cat Q5127 rows (post-guard):")
    display(bucket_rates)
else:
    print("\nNo remaining-cat rows or no guard bucket columns to summarize.")

# -----------------------------
# 5) Top 10 worst rows by abs_residual (post-guard) with key metrics
# -----------------------------
def first_present(cols):
    for c in cols:
        if c in df.columns:
            return c
    return None

# Try to locate baseline vs guard expected columns from your B2.2b construction
exp_bl_col = first_present(["expected_cost_baseline_gr", "expected_cost_baseline", "expected_cost_bl", "expected_cost"])
exp_gr_col = first_present(["expected_cost_guard_gr", "expected_cost_guard", "expected_cost_gr", "expected_cost"])

obs_col    = first_present(["observed_cost_gr", "observed_cost", "observed_cost_bl"])
oe_bl_col  = first_present(["oe_ratio_baseline_gr", "oe_ratio_baseline", "oe_ratio_bl", "oe_ratio"])
oe_gr_col  = first_present(["oe_ratio_guard_gr", "oe_ratio_guard", "oe_ratio_gr", "oe_ratio"])

# Compute guard residuals safely if not present
obs = pd.to_numeric(df[obs_col], errors="coerce") if obs_col else pd.Series(np.nan, index=df.index)
exp_gr = pd.to_numeric(df[exp_gr_col], errors="coerce") if exp_gr_col else pd.Series(np.nan, index=df.index)

res_gr = obs - exp_gr
abs_gr = res_gr.abs()

# Eligibility after merge
elig_col_m = None
if elig_col is not None:
    elig_col_m = elig_col + "_gr" if (elig_col + "_gr") in df.columns else elig_col

# Compact bucket string for each row (guard side)
def buckets_true(row):
    hits = []
    for c in guard_bucket_cols_merged:
        if bool(row.get(c, False)):
            hits.append(c.replace(guard_prefix, "").replace("_gr", ""))
    return ", ".join(hits)

view = pd.DataFrame({
    "row_id": df["row_id"],
    "HCPCS_Cd": df["HCPCS_Cd_gr"] if "HCPCS_Cd_gr" in df.columns else df.get("HCPCS_Cd", "Q5127"),
    "eligible": df[elig_col_m].astype(bool) if elig_col_m else False,
    "expected_baseline": pd.to_numeric(df[exp_bl_col], errors="coerce") if exp_bl_col else np.nan,
    "expected_guard": exp_gr,
    "observed_cost": obs,
    "residual_guard": res_gr,
    "abs_residual_guard": abs_gr,
    "oe_baseline": pd.to_numeric(df[oe_bl_col], errors="coerce") if oe_bl_col else np.nan,
    "oe_guard": pd.to_numeric(df[oe_gr_col], errors="coerce") if oe_gr_col else np.nan,
    "is_any_cat_guard": df[guard_any_merged].astype(bool),
})

if guard_bucket_cols_merged:
    view["guard_buckets_true"] = df.apply(buckets_true, axis=1)

top10 = view.sort_values("abs_residual_guard", ascending=False).head(10)

print("\nTop 10 worst Q5127 rows by abs_residual (post-guard):")
with pd.option_context("display.max_columns", None):
    display(top10)

### B2.2c `Q5127`-only residual-cap guardrail (no-new-cat safe)

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# B2.2c) Q5127-only residual-cap guardrail (no-new-cat safe)
#
# Fix remaining-cat Q5127 rows that are still catastrophic ONLY due to
# grB2b__cat_large_positive_residual by pushing expected up to cap residual.
#
# Trigger (eligible):
# - HCPCS_Cd == "Q5127"
# - cold-start (has_lag == False)
# - baseline catastrophic (bl__is_any_catastrophic == True)
# - still catastrophic after B2.2b (grB2b__is_any_catastrophic == True)
# - and specifically large positive residual under B2.2b
#
# Guard:
# - ensure residual <= resid_pos_q99 (minus small margin)
# - expected_guard_B2c = min(observed, max(expected_guard_B2b, observed - R_CAP))
#
# Safety:
# - only modify rows already baseline-cat -> cannot create new cats in full 2023_only
# ============================================================

assert "slice_2023_only" in globals(), "slice_2023_only not found"
assert "thresholds_v2" in globals(), "thresholds_v2 not found"
assert "apply_catastrophic_flags" in globals(), "apply_catastrophic_flags not found"
assert "baseline_scored_B2_2b" in globals(), "baseline_scored_B2_2b not found. Run B2.2b and save it."
assert "guard_scored_B2_2b" in globals(), "guard_scored_B2_2b not found. Run B2.2b and save it."

TARGET_CODE = "Q5127"
EPS = 1e-9
MARGIN = 1e-6

resid_pos_q99 = float(thresholds_v2["resid_pos_q99"])
R_CAP = resid_pos_q99 - MARGIN

print("B2.2c params")
print("  TARGET_CODE:", TARGET_CODE)
print("  resid_pos_q99:", resid_pos_q99)
print("  R_CAP:", R_CAP)

# --- baseline + B2.2b guard dfs ---
bl = baseline_scored_B2_2b.copy()
gr = guard_scored_B2_2b.copy()

# ---- guard expected column name (robust) ----
# Prefer the explicit guard column if present, else fall back to expected_cost (if you overwrote it)
if "expected_cost_guard" in gr.columns:
    exp_b2b_col = "expected_cost_guard"
elif "expected_cost" in gr.columns:
    exp_b2b_col = "expected_cost"
else:
    raise KeyError("guard_scored_B2_2b must contain expected_cost_guard or expected_cost")

# Basic required cols
need_bl = ["row_id", "HCPCS_Cd", "has_lag", "observed_cost", "expected_cost", "bl__is_any_catastrophic"]
need_gr = ["row_id", exp_b2b_col, "grB2b__is_any_catastrophic", "grB2b__cat_large_positive_residual"]
for c in need_bl:
    assert c in bl.columns, f"Missing in baseline_scored_B2_2b: {c}"
for c in need_gr:
    assert c in gr.columns, f"Missing in guard_scored_B2_2b: {c}"

# Merge on row_id only (stable key)
df = (
    bl[["row_id", "HCPCS_Cd", "has_lag", "observed_cost", "expected_cost", "bl__is_any_catastrophic"]]
    .merge(
        gr[["row_id", exp_b2b_col, "grB2b__is_any_catastrophic", "grB2b__cat_large_positive_residual"]],
        on="row_id",
        how="inner",
        validate="one_to_one",
        suffixes=("", "_gr")
    )
)

# Numeric coercion
for c in ["observed_cost", "expected_cost", exp_b2b_col]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["has_lag"] = df["has_lag"].astype(bool)

obs = df["observed_cost"].to_numpy(dtype="float64")
exp_base = df["expected_cost"].to_numpy(dtype="float64")
exp_b2b = df[exp_b2b_col].to_numpy(dtype="float64")

# Recompute B2.2b guard metrics (so we don't rely on residual_guard columns)
res_b2b = obs - exp_b2b
abs_b2b = np.abs(res_b2b)
oe_b2b = obs / np.maximum(exp_b2b, EPS)

# Eligibility mask
is_target = (df["HCPCS_Cd"] == TARGET_CODE).to_numpy()
is_cold = (~df["has_lag"].to_numpy())
base_cat = df["bl__is_any_catastrophic"].astype(bool).to_numpy()
b2b_cat = df["grB2b__is_any_catastrophic"].astype(bool).to_numpy()
b2b_resid_bucket = df["grB2b__cat_large_positive_residual"].astype(bool).to_numpy()

eligible = (
    is_target
    & is_cold
    & base_cat
    & b2b_cat
    & b2b_resid_bucket
    & np.isfinite(obs)
    & np.isfinite(exp_b2b)
)

print("\nB2.2c eligibility")
print("  eligible rows:", int(eligible.sum()))
print("  eligible share within Q5127 (%):", (eligible.sum() / max(1, int(is_target.sum()))) * 100)

# Apply residual cap
need_for_resid_cap = obs - R_CAP              # expected must be >= this to make residual <= R_CAP
exp_guard_c = exp_b2b.copy()
exp_guard_c[eligible] = np.maximum(exp_b2b[eligible], need_for_resid_cap[eligible])
exp_guard_c[eligible] = np.minimum(exp_guard_c[eligible], obs[eligible])  # invariant
exp_guard_c = np.minimum(exp_guard_c, obs)                                # extra safety

# Recompute metrics for B2.2c
res_c = obs - exp_guard_c
abs_c = np.abs(res_c)
oe_c = obs / np.maximum(exp_guard_c, EPS)

# Build scored df for applying baseline thresholds
b2c_scored = df.copy()
b2c_scored["expected_cost_baseline"] = exp_base
b2c_scored["expected_cost_guard_B2b"] = exp_b2b
b2c_scored["residual_guard_B2b"] = res_b2b
b2c_scored["abs_residual_guard_B2b"] = abs_b2b
b2c_scored["oe_ratio_guard_B2b"] = oe_b2b

b2c_scored["expected_cost_guard_B2c"] = exp_guard_c
b2c_scored["residual_guard_B2c"] = res_c
b2c_scored["abs_residual_guard_B2c"] = abs_c
b2c_scored["oe_ratio_guard_B2c"] = oe_c
b2c_scored["B2c_eligible"] = eligible

# Bring in auxiliary columns needed by apply_catastrophic_flags (if missing)
aux_cols = ["expected_cost_support_tier", "is_top_1pct_avg_mdcr_stdzd_amt", "high_confidence_anomaly_candidate"]
need_aux = [c for c in aux_cols if c not in b2c_scored.columns and c in slice_2023_only.columns]
if need_aux:
    aux = slice_2023_only[["row_id"] + need_aux].copy()
    b2c_scored = b2c_scored.merge(aux, on="row_id", how="left", validate="one_to_one")

# Apply catastrophic flags on B2.2c metrics using fixed thresholds
tmp = b2c_scored.copy()
tmp["expected_cost"] = tmp["expected_cost_guard_B2c"]
tmp["residual"] = tmp["residual_guard_B2c"]
tmp["abs_residual"] = tmp["abs_residual_guard_B2c"]
tmp["oe_ratio"] = tmp["oe_ratio_guard_B2c"]
tmp = apply_catastrophic_flags(tmp, thresholds_v2, prefix="grB2c__")
b2c_scored = tmp

# Invariant check within eligible rows
viol = eligible & np.isfinite(obs) & np.isfinite(exp_guard_c) & (exp_guard_c > obs + 1e-9)
assert int(viol.sum()) == 0, "Invariant violated: expected_guard_B2c > observed within eligible rows."
print("\nInvariant PASS within eligible rows")

# No-new-cat check in full 2023_only slice (by construction, but enforce)
gr_any = b2c_scored["grB2c__is_any_catastrophic"].astype(bool).to_numpy()
new_cat = gr_any & (~base_cat)
assert new_cat.mean() == 0.0, f"New catastrophes created in 2023_only slice: {new_cat.mean() * 100:.6f}%"
print("No-new-cat PASS in full 2023_only slice")

# Evaluation helper
bucket_cols = [
    "cat_abs_residual_q99","cat_oe_q99","cat_large_positive_residual","cat_large_negative_residual",
    "cat_large_error_high_support","cat_large_error_tail_row","cat_underprediction","cat_overprediction",
    "cat_high_conf_anomaly","cat_cold_low_support_failure","cat_hot_failure","cat_cold_failure",
]

def eval_mask(title, mask_eval):
    base_any_m = base_cat[mask_eval]
    gr_any_m = b2c_scored.loc[mask_eval, "grB2c__is_any_catastrophic"].astype(bool).to_numpy()
    print(f"\n=== {title} ===")
    print("n_rows:", int(mask_eval.sum()))
    print("baseline_cat_rate_%:", round(base_any_m.mean() * 100, 6))
    print("guard_cat_rate_%   :", round(gr_any_m.mean() * 100, 6))
    print("delta_pp           :", round((gr_any_m.mean() - base_any_m.mean()) * 100, 6))

    rows = []
    for b in bucket_cols:
        b_bl = bl.loc[mask_eval, f"bl__{b}"].mean() * 100
        b_gr = b2c_scored.loc[mask_eval, f"grB2c__{b}"].mean() * 100
        rows.append({"bucket": b, "baseline_rate_%": b_bl, "guard_rate_%": b_gr, "delta_pp": b_gr - b_bl})
    display(pd.DataFrame(rows).sort_values("delta_pp"))

mask_q5127 = is_target
eval_mask("B2.2c results: Q5127 within 2023_only", mask_q5127)

mask_all = np.ones(len(df), dtype=bool)
eval_mask("B2.2c results: ALL 2023_only rows (sanity)", mask_all)

# Save canonical outputs
guard_scored_B2_2c = b2c_scored
baseline_scored_B2_2c = bl

print("\nSaved: guard_scored_B2_2c, baseline_scored_B2_2c")

## B2.3 Pin winners + sanity checks

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# B2.3) Pin winners + sanity checks
# Requires:
# - baseline_scored_B2_2b, guard_scored_B2_2b
# - baseline_scored_B2_2c, guard_scored_B2_2c
# ============================================================

assert "baseline_scored_B2_2b" in globals() and "guard_scored_B2_2b" in globals(), "Missing B2.2b outputs"
assert "baseline_scored_B2_2c" in globals() and "guard_scored_B2_2c" in globals(), "Missing B2.2c outputs"

# Pin canonical names
baseline_scored_B2_best = baseline_scored_B2_2c
guard_scored_B2_2b_best = guard_scored_B2_2b
guard_scored_B2_2c_best = guard_scored_B2_2c

print("Pinned:")
print("  guard_scored_B2_2b_best:", guard_scored_B2_2b_best.shape)
print("  guard_scored_B2_2c_best:", guard_scored_B2_2c_best.shape)

def _sanity_guard(df, label, applied_col=None, exp_guard_col="expected_cost_guard"):
    # Applied mask
    if applied_col and applied_col in df.columns:
        applied = df[applied_col].astype(bool).to_numpy()
    else:
        applied = np.ones(len(df), dtype=bool)

    obs = pd.to_numeric(df["observed_cost"], errors="coerce").to_numpy(dtype="float64")
    eg  = pd.to_numeric(df[exp_guard_col], errors="coerce").to_numpy(dtype="float64")
    m = applied & np.isfinite(obs) & np.isfinite(eg)

    viol = m & (eg > obs + 1e-9)
    print(f"\n{label} sanity")
    print("  applied rows:", int(m.sum()))
    print("  invariant violations:", int(viol.sum()))
    assert int(viol.sum()) == 0, f"{label}: expected_guard > observed in applied rows"

# B2.2b invariant within eligible
if "B2b_eligible" in guard_scored_B2_2b_best.columns:
    _sanity_guard(guard_scored_B2_2b_best, "B2.2b", applied_col="B2b_eligible", exp_guard_col="expected_cost_guard")

# B2.2c invariant within eligible
if "B2c_eligible" in guard_scored_B2_2c_best.columns:
    _sanity_guard(guard_scored_B2_2c_best, "B2.2c", applied_col="B2c_eligible", exp_guard_col="expected_cost_guard")

# No-new-cat check on full 2023_only for B2.2c (strongest version)
base_any = baseline_scored_B2_best["bl__is_any_catastrophic"].astype(bool).to_numpy()
gr_any   = guard_scored_B2_2c_best["grB2c__is_any_catastrophic"].astype(bool).to_numpy()
new_cat  = gr_any & (~base_any)

print("\nNo-new-cat check (B2.2c vs baseline, full 2023_only)")
print("  new_cat_%:", round(new_cat.mean() * 100, 6))
assert new_cat.mean() == 0, "B2.2c created new catastrophes (should be 0)"
print("PASS")

## B2.4 Freeze policies + apply functions

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# B2.4) Freeze B2 policies + reusable apply functions
# ============================================================

# Pull baseline thresholds we used in B2.2b / B2.2c
OE_Q99 = float(thresholds_v2["oe_q99"])
ABS_RESID_Q99 = float(thresholds_v2["abs_resid_q99"])
RESID_POS_Q99 = float(thresholds_v2["resid_pos_q99"])

guardrail_b2_policy = {
    "name": "B2_2b_constraint_floor_Q5126_Q5127",
    "scope": "apply ONLY to Year==2023 & hcpcs_stability_group=='2023_only' & cold_start & HCPCS in {Q5126,Q5127} & oe_ratio >= oe_q99",
    "target_codes": ["Q5126", "Q5127"],
    "oe_trigger": OE_Q99,
    "global_floor": float(global_cold_floor),
    "epsilon": 1e-9,
}

guardrail_b2c_policy = {
    "name": "B2_2c_residual_cap_Q5127",
    "scope": "apply ONLY to Year==2023 & hcpcs_stability_group=='2023_only' & cold_start & HCPCS==Q5127 & baseline-cat & (post-B2.2b) cat_large_positive_residual",
    "target_code": "Q5127",
    "resid_pos_cap": float(RESID_POS_Q99) - 1e-9,  # R_CAP you used
    "epsilon": 1e-9,
}

print("B2 policies:")
display(pd.DataFrame([guardrail_b2_policy, guardrail_b2c_policy]))

def apply_guardrail_B2_2b(df: pd.DataFrame, policy: dict, thresholds: dict) -> pd.DataFrame:
    """
    B2.2b: constraint-based floor for Q5126/Q5127 when oe_ratio is extreme.
    expected_guard = min(obs, max(exp, floor_needed))
    floor_needed = max(global_floor, obs/oe_q99, obs-resid_pos_q99, obs-abs_resid_q99)
    """
    out = df.copy()
    req = ["Year","hcpcs_stability_group","has_lag","HCPCS_Cd","observed_cost","expected_cost","oe_ratio"]
    missing = [c for c in req if c not in out.columns]
    if missing:
        raise ValueError(f"apply_guardrail_B2_2b missing columns: {missing}")

    EPS = float(policy.get("epsilon", 1e-9))
    oe_q99 = float(thresholds["oe_q99"])
    abs_q99 = float(thresholds["abs_resid_q99"])
    resid_pos_q99 = float(thresholds["resid_pos_q99"])
    global_floor = float(policy["global_floor"])
    targets = set(policy["target_codes"])

    out["observed_cost"] = pd.to_numeric(out["observed_cost"], errors="coerce")
    out["expected_cost"] = pd.to_numeric(out["expected_cost"], errors="coerce")
    out["oe_ratio"] = pd.to_numeric(out["oe_ratio"], errors="coerce")

    obs = out["observed_cost"].to_numpy(dtype="float64")
    exp = out["expected_cost"].to_numpy(dtype="float64")
    oe  = out["oe_ratio"].to_numpy(dtype="float64")

    eligible = (
        (out["Year"] == 2023)
        & (out["hcpcs_stability_group"] == "2023_only")
        & (~out["has_lag"].astype(bool))
        & (out["HCPCS_Cd"].isin(targets))
        & (oe >= oe_q99)
        & np.isfinite(obs) & np.isfinite(exp) & np.isfinite(oe)
    ).to_numpy()

    need_for_oe = obs / max(oe_q99, EPS)
    need_for_resid_pos = obs - resid_pos_q99
    need_for_abs = obs - abs_q99

    floor_needed = np.maximum.reduce([
        np.full_like(obs, global_floor, dtype="float64"),
        need_for_oe,
        need_for_resid_pos,
        need_for_abs,
    ])

    exp_guard = exp.copy()
    raised = np.maximum(exp, floor_needed)
    exp_guard[eligible] = np.minimum(obs[eligible], raised[eligible])
    exp_guard[eligible] = np.minimum(exp_guard[eligible], obs[eligible])

    res_guard = obs - exp_guard
    abs_guard = np.abs(res_guard)
    oe_guard  = obs / np.maximum(exp_guard, EPS)

    out["expected_cost_guard_b2"] = exp_guard
    out["residual_guard_b2"] = res_guard
    out["abs_residual_guard_b2"] = abs_guard
    out["oe_ratio_guard_b2"] = oe_guard
    out["b2_applied"] = eligible

    return out

def apply_guardrail_B2_2c(df: pd.DataFrame, policy: dict, thresholds: dict) -> pd.DataFrame:
    """
    B2.2c: residual-cap for Q5127 to eliminate remaining cat_large_positive_residual.
    We enforce residual_guard <= resid_pos_q99 (cap), while keeping expected <= observed.
    """
    out = df.copy()
    req = ["Year","hcpcs_stability_group","has_lag","HCPCS_Cd","observed_cost","expected_cost"]
    missing = [c for c in req if c not in out.columns]
    if missing:
        raise ValueError(f"apply_guardrail_B2_2c missing columns: {missing}")

    EPS = float(policy.get("epsilon", 1e-9))
    resid_cap = float(policy["resid_pos_cap"])
    target = policy["target_code"]

    out["observed_cost"] = pd.to_numeric(out["observed_cost"], errors="coerce")
    out["expected_cost"] = pd.to_numeric(out["expected_cost"], errors="coerce")

    obs = out["observed_cost"].to_numpy(dtype="float64")
    exp = out["expected_cost"].to_numpy(dtype="float64")

    eligible = (
        (out["Year"] == 2023)
        & (out["hcpcs_stability_group"] == "2023_only")
        & (~out["has_lag"].astype(bool))
        & (out["HCPCS_Cd"] == target)
        & np.isfinite(obs) & np.isfinite(exp)
    ).to_numpy()

    # Need expected >= obs - resid_cap (so residual <= resid_cap)
    need_exp = obs - resid_cap

    exp_guard = exp.copy()
    exp_guard[eligible] = np.maximum(exp_guard[eligible], need_exp[eligible])
    exp_guard[eligible] = np.minimum(exp_guard[eligible], obs[eligible])  # cap at observed

    res_guard = obs - exp_guard
    abs_guard = np.abs(res_guard)
    oe_guard  = obs / np.maximum(exp_guard, EPS)

    out["expected_cost_guard_b2c"] = exp_guard
    out["residual_guard_b2c"] = res_guard
    out["abs_residual_guard_b2c"] = abs_guard
    out["oe_ratio_guard_b2c"] = oe_guard
    out["b2c_applied"] = eligible

    return out

## B2.4. Standardized apply functions

### B2.4.0 Standard utilities for guardrails

In [ ]:
import numpy as np
import pandas as pd

def _require_cols(df: pd.DataFrame, cols, fn_name: str):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{fn_name}: missing required columns: {missing}")

def _to_f64(df: pd.DataFrame, col: str) -> np.ndarray:
    return pd.to_numeric(df[col], errors="coerce").to_numpy(dtype="float64")

def _promote_guard_as_current(df: pd.DataFrame) -> pd.DataFrame:
    """
    Promote *_guard columns to the working columns expected_cost/residual/abs_residual/oe_ratio.
    This is how we compose stages deterministically.
    """
    out = df.copy()
    for src, dst in [
        ("expected_cost_guard", "expected_cost"),
        ("residual_guard", "residual"),
        ("abs_residual_guard", "abs_residual"),
        ("oe_ratio_guard", "oe_ratio"),
    ]:
        if src not in out.columns:
            raise ValueError(f"_promote_guard_as_current: missing {src}")
        out[dst] = out[src]
    return out

### B2.4.1 Standardized B2.2b apply function


In [ ]:
def apply_guardrail_B2_2b(
    df: pd.DataFrame,
    policy: dict,
    thresholds: dict
) -> pd.DataFrame:
    """
    B2.2b constraint-based floor for top-burden 2023-only codes (Q5126/Q5127).

    Expected policy keys:
      - name
      - scope (string)
      - target_codes (list[str])
      - oe_trigger (float)  # usually thresholds['oe_q99']
      - global_floor (float)  # usually global_cold_floor
      - epsilon (float)

    Writes standardized outputs:
      expected_cost_guard, residual_guard, abs_residual_guard, oe_ratio_guard,
      guardrail_applied, guardrail_name
    """
    fn = "apply_guardrail_B2_2b"
    _require_cols(df, ["Year","hcpcs_stability_group","HCPCS_Cd","has_lag","observed_cost","expected_cost","oe_ratio"], fn)

    out = df.copy()

    # numeric arrays
    obs = _to_f64(out, "observed_cost")
    exp = _to_f64(out, "expected_cost")
    oe  = _to_f64(out, "oe_ratio")

    EPS = float(policy.get("epsilon", 1e-9))

    # thresholds needed for floor constraints
    oe_q99 = float(policy.get("oe_trigger", thresholds["oe_q99"]))
    abs_resid_q99 = float(thresholds["abs_resid_q99"])
    resid_pos_q99 = float(thresholds["resid_pos_q99"])
    global_floor = float(policy["global_floor"])
    target_codes = set(policy["target_codes"])

    is_eligible = (
        (out["Year"] == 2023)
        & (out["hcpcs_stability_group"] == "2023_only")
        & (~out["has_lag"].astype(bool))
        & (out["HCPCS_Cd"].isin(target_codes))
        & np.isfinite(obs) & np.isfinite(exp) & np.isfinite(oe)
        & (oe >= oe_q99)
    ).to_numpy()

    # Per-row needed floors (vectorized)
    need_for_oe        = obs / max(oe_q99, EPS)
    need_for_resid_pos = obs - resid_pos_q99
    need_for_abs       = obs - abs_resid_q99

    floor_needed = np.maximum.reduce([
        np.full_like(obs, global_floor, dtype="float64"),
        need_for_oe,
        need_for_resid_pos,
        need_for_abs,
    ])

    exp_guard = exp.copy()
    exp_guard[is_eligible] = np.minimum(
        obs[is_eligible],
        np.maximum(exp[is_eligible], floor_needed[is_eligible])
    )

    # safety net within eligible
    exp_guard[is_eligible] = np.minimum(exp_guard[is_eligible], obs[is_eligible])

    res_guard = obs - exp_guard
    abs_guard = np.abs(res_guard)
    oe_guard  = obs / np.maximum(exp_guard, EPS)

    out["expected_cost_guard"] = exp_guard
    out["residual_guard"] = res_guard
    out["abs_residual_guard"] = abs_guard
    out["oe_ratio_guard"] = oe_guard
    out["guardrail_applied"] = is_eligible
    out["guardrail_name"] = str(policy.get("name", "B2_2b"))

    return out

### B2.4.2 Standardized B2.2c apply function


In [ ]:
def apply_guardrail_B2_2c(
    df: pd.DataFrame,
    policy: dict,
    thresholds: dict
) -> pd.DataFrame:
    """
    B2.2c residual-cap guardrail for Q5127.
    Idea: if a row is still in large_positive_residual territory, force residual <= cap,
    by raising expected to observed - cap (bounded by observed).

    Expected policy keys:
      - name
      - target_code (str)  # "Q5127"
      - resid_pos_cap (float)  # e.g. thresholds['resid_pos_q99'] - tiny_eps
      - epsilon (float)

    Writes standardized outputs:
      expected_cost_guard, residual_guard, abs_residual_guard, oe_ratio_guard,
      guardrail_applied, guardrail_name
    """
    fn = "apply_guardrail_B2_2c"
    _require_cols(df, ["Year","hcpcs_stability_group","HCPCS_Cd","has_lag","observed_cost","expected_cost"], fn)

    out = df.copy()

    obs = _to_f64(out, "observed_cost")
    exp = _to_f64(out, "expected_cost")
    EPS = float(policy.get("epsilon", 1e-9))

    target_code = str(policy["target_code"])
    cap = float(policy["resid_pos_cap"])

    # Eligibility: scoped exactly to where you intended this to apply
    # (you can add additional trigger logic if desired, but keep it explicit)
    eligible = (
        (out["Year"] == 2023)
        & (out["hcpcs_stability_group"] == "2023_only")
        & (~out["has_lag"].astype(bool))
        & (out["HCPCS_Cd"] == target_code)
        & np.isfinite(obs) & np.isfinite(exp)
    ).to_numpy()

    # Residual cap means: residual = obs - expected <= cap  -> expected >= obs - cap
    min_expected = obs - cap

    exp_guard = exp.copy()
    exp_guard[eligible] = np.maximum(exp_guard[eligible], min_expected[eligible])

    # Never exceed observed (keeps expected<=obs invariant for modified rows)
    exp_guard[eligible] = np.minimum(exp_guard[eligible], obs[eligible])

    res_guard = obs - exp_guard
    abs_guard = np.abs(res_guard)
    oe_guard  = obs / np.maximum(exp_guard, EPS)

    out["expected_cost_guard"] = exp_guard
    out["residual_guard"] = res_guard
    out["abs_residual_guard"] = abs_guard
    out["oe_ratio_guard"] = oe_guard
    out["guardrail_applied"] = eligible
    out["guardrail_name"] = str(policy.get("name", "B2_2c"))

    return out

### B2.4.3 Mini “standardization sanity check” cell

In [ ]:
# Quick sanity: functions exist and standardized columns match
for fn_name in ["apply_guardrail_B2_2b", "apply_guardrail_B2_2c"]:
    assert fn_name in globals() and callable(globals()[fn_name]), f"Missing {fn_name}"

print("OK: standardized B2 apply functions defined.")

# Optional: check required output columns on a tiny sample (no-op is fine)
sample = read_pq("failure_df_full_v2", cols=[
    "row_id","Year","HCPCS_Cd","hcpcs_stability_group","has_lag","observed_cost","expected_cost","oe_ratio"
]).query("Year==2023").head(5).copy()

# minimal thresholds/policies should already exist in your notebook
# This just verifies output columns exist
tmp = apply_guardrail_B2_2b(sample, guardrail_b2_policy, thresholds_v2)
need = ["expected_cost_guard","residual_guard","abs_residual_guard","oe_ratio_guard","guardrail_applied","guardrail_name"]
assert all(c in tmp.columns for c in need), f"B2_2b missing outputs: {[c for c in need if c not in tmp.columns]}"

tmp2 = apply_guardrail_B2_2c(sample, guardrail_b2c_policy, thresholds_v2)
assert all(c in tmp2.columns for c in need), f"B2_2c missing outputs: {[c for c in need if c not in tmp2.columns]}"

print("OK: both B2 apply functions write standardized guard columns.")

## B2.5) Compose guardrails: A1c (V0) + B2.2b + B2.2c

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# B2.5) Compose guardrails: A1c (V0) -> B2.2b -> B2.2c
# Evaluate side-by-side like A1.5 (Year=2023):
#   - stable_all_years vs 2023_only
#   - baseline vs composed (using fixed thresholds_v2)
# ============================================================

CAT_BUCKETS = [
    "cat_abs_residual_q99",
    "cat_oe_q99",
    "cat_large_positive_residual",
    "cat_large_negative_residual",
    "cat_large_error_high_support",
    "cat_large_error_tail_row",
    "cat_underprediction",
    "cat_overprediction",
    "cat_high_conf_anomaly",
    "cat_cold_low_support_failure",
    "cat_hot_failure",
    "cat_cold_failure",
]
GROUPS = ["stable_all_years", "2023_only"]
YEAR_EVAL = 2023

# -----------------------------
# Preconditions
# -----------------------------
assert "thresholds_v2" in globals()
assert "apply_catastrophic_flags" in globals()
assert "read_pq" in globals()

assert "guardrail_v0_policy" in globals()
assert "apply_guardrail_v0_A1c_2023_only" in globals()

assert "guardrail_b2_policy" in globals()
assert "guardrail_b2c_policy" in globals()
assert "apply_guardrail_B2_2b" in globals()
assert "apply_guardrail_B2_2c" in globals()
assert "_promote_guard_as_current" in globals()

print("B2.5 compose order:")
print("  1) A1c:", guardrail_v0_policy.get("name"))
print("  2) B2.2b:", guardrail_b2_policy.get("name"))
print("  3) B2.2c:", guardrail_b2c_policy.get("name"))

# -----------------------------
# Load Year=2023 slice
# -----------------------------
cols_needed = [
    "row_id", "Year", "HCPCS_Cd", "hcpcs_stability_group",
    "has_lag", "route",
    "expected_cost_support_tier",
    "is_top_1pct_avg_mdcr_stdzd_amt",
    "high_confidence_anomaly_candidate",
    "observed_cost", "expected_cost", "residual", "abs_residual", "oe_ratio",
]
df_2023 = read_pq("failure_df_full_v2", cols=cols_needed)
df_2023 = df_2023.loc[
    (df_2023["Year"] == YEAR_EVAL) & (df_2023["hcpcs_stability_group"].isin(GROUPS))
].copy()

for c in ["observed_cost", "expected_cost", "residual", "abs_residual", "oe_ratio"]:
    df_2023[c] = pd.to_numeric(df_2023[c], errors="coerce")
df_2023["has_lag"] = df_2023["has_lag"].astype(bool)

print("\nYear=2023 rows in scope:", len(df_2023))
display(
    df_2023["hcpcs_stability_group"]
    .value_counts(dropna=False)
    .rename_axis("hcpcs_stability_group")
    .reset_index(name="n_rows")
)

# -----------------------------
# Baseline scoring (bl__)
# -----------------------------
baseline_scored_all = apply_catastrophic_flags(df_2023.copy(), thresholds_v2, prefix="bl__")

# -----------------------------
# Compose guardrails (A1c -> B2.2b -> B2.2c)
# Each stage writes standardized *_guard columns.
# We "promote" those guard columns into expected_cost/residual/... before the next stage.
# -----------------------------
w0 = apply_guardrail_v0_A1c_2023_only(df_2023.copy(), guardrail_v0_policy)
w1 = _promote_guard_as_current(w0)

w2 = apply_guardrail_B2_2b(w1, guardrail_b2_policy, thresholds_v2)
w3 = _promote_guard_as_current(w2)

w4 = apply_guardrail_B2_2c(w3, guardrail_b2c_policy, thresholds_v2)
w5 = _promote_guard_as_current(w4)

composed_scored_all = apply_catastrophic_flags(w5.copy(), thresholds_v2, prefix="gr__")

# Keep a clear final guard column for audits
composed_scored_all["expected_cost_guard_final"] = composed_scored_all["expected_cost"]
composed_scored_all["residual_guard_final"] = composed_scored_all["residual"]
composed_scored_all["abs_residual_guard_final"] = composed_scored_all["abs_residual"]
composed_scored_all["oe_ratio_guard_final"] = composed_scored_all["oe_ratio"]

# -----------------------------
# Sanity checks
# -----------------------------
stable_mask = (df_2023["hcpcs_stability_group"] == "stable_all_years").to_numpy()
stable_unchanged = np.isclose(
    composed_scored_all.loc[stable_mask, "expected_cost_guard_final"].to_numpy(dtype="float64"),
    df_2023.loc[stable_mask, "expected_cost"].to_numpy(dtype="float64"),
    atol=0, rtol=0, equal_nan=True
).all()
print("\nSanity: stable_all_years expected unchanged under COMPOSED:", bool(stable_unchanged))

mask_2023_only = (df_2023["hcpcs_stability_group"] == "2023_only").to_numpy()
base_any = baseline_scored_all.loc[mask_2023_only, "bl__is_any_catastrophic"].astype(bool).to_numpy()
gr_any   = composed_scored_all.loc[mask_2023_only, "gr__is_any_catastrophic"].astype(bool).to_numpy()
new_cat = gr_any & (~base_any)
print("No-new-cat check (2023_only): new_cat_% =", round(new_cat.mean() * 100, 6))

# -----------------------------
# A1.5-style summaries
# -----------------------------
def summarize_rates(df_flags: pd.DataFrame, prefix: str) -> pd.DataFrame:
    cols = [f"{prefix}is_any_catastrophic"] + [f"{prefix}{b}" for b in CAT_BUCKETS]
    out = (
        df_flags.groupby("hcpcs_stability_group", dropna=False)
        .agg(
            n_rows=("row_id", "size"),
            **{c + "_rate": (c, "mean") for c in cols}
        )
        .reset_index()
    )
    rate_cols = [c for c in out.columns if c.endswith("_rate")]
    out[rate_cols] = out[rate_cols] * 100
    return out

def summarize_counts(df_flags: pd.DataFrame, prefix: str) -> pd.DataFrame:
    cols = [f"{prefix}is_any_catastrophic"] + [f"{prefix}{b}" for b in CAT_BUCKETS]
    out = (
        df_flags.groupby("hcpcs_stability_group", dropna=False)[cols]
        .sum()
        .astype(int)
        .reset_index()
    )
    return out

rate_bl = summarize_rates(baseline_scored_all, "bl__").rename(columns=lambda c: c.replace("bl__", "baseline__"))
rate_gr = summarize_rates(composed_scored_all, "gr__").rename(columns=lambda c: c.replace("gr__", "composed__"))

count_bl = summarize_counts(baseline_scored_all, "bl__").rename(columns=lambda c: c.replace("bl__", "baseline__"))
count_gr = summarize_counts(composed_scored_all, "gr__").rename(columns=lambda c: c.replace("gr__", "composed__"))

rate_side_by_side = rate_bl.merge(rate_gr, on=["hcpcs_stability_group", "n_rows"], how="inner")
count_side_by_side = count_bl.merge(count_gr, on=["hcpcs_stability_group"], how="inner")

order = pd.CategoricalDtype(categories=["stable_all_years", "2023_only"], ordered=True)
rate_side_by_side["hcpcs_stability_group"] = rate_side_by_side["hcpcs_stability_group"].astype(order)
rate_side_by_side = rate_side_by_side.sort_values("hcpcs_stability_group")

count_side_by_side["hcpcs_stability_group"] = count_side_by_side["hcpcs_stability_group"].astype(order)
count_side_by_side = count_side_by_side.sort_values("hcpcs_stability_group")

print("\nB2.5) Rate table (%, baseline vs COMPOSED) | Year=2023")
with pd.option_context("display.max_columns", None):
    display(rate_side_by_side)

print("\nB2.5) Count table (counts, baseline vs COMPOSED) | Year=2023")
with pd.option_context("display.max_columns", None):
    display(count_side_by_side)

delta_any = (
    rate_side_by_side[[
        "hcpcs_stability_group", "n_rows",
        "baseline__is_any_catastrophic_rate",
        "composed__is_any_catastrophic_rate"
    ]]
    .assign(delta_pp=lambda x: x["composed__is_any_catastrophic_rate"] - x["baseline__is_any_catastrophic_rate"])
)
print("\nB2.5) is_any_catastrophic delta (pp) | Year=2023")
display(delta_any)

# Save canonical outputs
baseline_scored_B2_5 = baseline_scored_all
guard_scored_B2_5 = composed_scored_all
print("\nSaved: baseline_scored_B2_5, guard_scored_B2_5")

## B2.6 Pin composed policy set as V1 (Coverage shift + top-burden code guardrails)

In [ ]:
# ============================================================
# B2.6) Pin composed policy set as V1 (Coverage shift + top-burden code guardrails)
# Requires existing objects from your notebook:
#   - guardrail_v0_policy (A1c) + apply_guardrail_v0_A1c_2023_only
#   - guardrail_b2_policy (B2.2b) + apply_guardrail_B2_2b
#   - guardrail_b2c_policy (B2.2c) + apply_guardrail_B2_2c
# ============================================================
import pandas as pd

import inspect
print("A1c sig:", inspect.signature(apply_guardrail_v0_A1c_2023_only))
print("B2.2b sig:", inspect.signature(apply_guardrail_B2_2b))
print("B2.2c sig:", inspect.signature(apply_guardrail_B2_2c))

required = [
    "guardrail_v0_policy",
    "guardrail_b2_policy",
    "guardrail_b2c_policy",
    "apply_guardrail_v0_A1c_2023_only",
    "apply_guardrail_B2_2b",
    "apply_guardrail_B2_2c",
]
missing = [x for x in required if x not in globals()]
if missing:
    raise NameError(f"Missing required objects for V1 pinning: {missing}")

guardrail_v1 = {
    "name": "V1_coverage_shift_plus_top_burden_codes",
    "components": [
        {"name": "A1c_V0", "policy": guardrail_v0_policy, "apply_fn": apply_guardrail_v0_A1c_2023_only},
        {"name": "B2_2b",  "policy": guardrail_b2_policy, "apply_fn": apply_guardrail_B2_2b},
        {"name": "B2_2c",  "policy": guardrail_b2c_policy, "apply_fn": apply_guardrail_B2_2c},
    ],
}

print("Pinned guardrail set:", guardrail_v1["name"])


display(pd.DataFrame([{
    "set_name": guardrail_v1["name"],
    "component": c["name"],
    "policy_name": c["policy"].get("name"),
    "scope": c["policy"].get("scope"),
} for c in guardrail_v1["components"]]))

### B2.6.a. A single helper that applies V1 in order and handles the signature difference automatically

In [ ]:
import inspect
import numpy as np
import pandas as pd

def apply_guardrail_set_v1(df: pd.DataFrame, guardrail_set: dict, thresholds: dict) -> pd.DataFrame:
    """
    Apply a pinned guardrail set (like guardrail_v1) in order.
    Handles components with either signature:
      - (df, policy)
      - (df, policy, thresholds)
    Returns a df where expected_cost/residual/abs_residual/oe_ratio reflect the final guarded metrics.
    """
    out = df.copy()

    for comp in guardrail_set["components"]:
        fn = comp["apply_fn"]
        pol = comp["policy"]

        sig = inspect.signature(fn)
        if len(sig.parameters) == 2:
            out = fn(out, pol)
        else:
            out = fn(out, pol, thresholds)

        # Standardize “current scored view” after each component:
        # if the component wrote expected_cost_guard*, promote it to expected_cost for the next stage.
        if "expected_cost_guard" in out.columns:
            out["expected_cost"] = out["expected_cost_guard"]
        if "residual_guard" in out.columns:
            out["residual"] = out["residual_guard"]
        if "abs_residual_guard" in out.columns:
            out["abs_residual"] = out["abs_residual_guard"]
        if "oe_ratio_guard" in out.columns:
            out["oe_ratio"] = out["oe_ratio_guard"]

    return out

# Evaluate V1 performance

## B2.7.a. Evaluate V1 across all years (Year × stability_group)

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# B2.7.a) Evaluate V1 across ALL years: Year x hcpcs_stability_group
# - Uses fixed thresholds_v2 (no moving goalposts)
# - Applies composed guardrail set V1 via apply_guardrail_set_v1
# - Produces: rate table + count table + delta table
# - Includes "no changes outside intended scopes" sanity checks
# ============================================================

assert "guardrail_v1" in globals(), "guardrail_v1 not found (run B2.6)"
assert "apply_guardrail_set_v1" in globals(), "apply_guardrail_set_v1 not found (run B2.6.a)"
assert "thresholds_v2" in globals(), "thresholds_v2 not found"
assert "apply_catastrophic_flags" in globals(), "apply_catastrophic_flags not found"
assert "read_pq" in globals(), "read_pq not found"

CAT_BUCKETS = [
    "cat_abs_residual_q99",
    "cat_oe_q99",
    "cat_large_positive_residual",
    "cat_large_negative_residual",
    "cat_large_error_high_support",
    "cat_large_error_tail_row",
    "cat_underprediction",
    "cat_overprediction",
    "cat_high_conf_anomaly",
    "cat_cold_low_support_failure",
    "cat_hot_failure",
    "cat_cold_failure",
]

# -----------------------------
# 1) Load minimal full dataset
# -----------------------------
cols_needed = [
    "row_id", "Year", "HCPCS_Cd", "hcpcs_stability_group",
    "has_lag", "route",
    "expected_cost_support_tier",
    "is_top_1pct_avg_mdcr_stdzd_amt",
    "high_confidence_anomaly_candidate",
    "observed_cost", "expected_cost", "residual", "abs_residual", "oe_ratio",
]
df_all = read_pq("failure_df_full_v2", cols=cols_needed)

# numeric coercion
for c in ["observed_cost", "expected_cost", "residual", "abs_residual", "oe_ratio"]:
    df_all[c] = pd.to_numeric(df_all[c], errors="coerce")
df_all["has_lag"] = df_all["has_lag"].astype(bool)

print("Rows loaded:", len(df_all))
display(
    df_all.groupby(["Year", "hcpcs_stability_group"], dropna=False)
    .size().rename("n_rows").reset_index()
    .sort_values(["Year", "hcpcs_stability_group"])
)

# -----------------------------
# 2) Baseline scoring (bl__)
# -----------------------------
baseline_scored = apply_catastrophic_flags(df_all.copy(), thresholds_v2, prefix="bl__")

# -----------------------------
# 3) Apply composed V1 guardrails
# -----------------------------
guarded = apply_guardrail_set_v1(df_all.copy(), guardrail_v1, thresholds_v2)

# Guarded scoring (gr__)
guard_scored = apply_catastrophic_flags(guarded.copy(), thresholds_v2, prefix="gr__")

# -----------------------------
# 4) Sanity checks: no changes outside intended scopes
# Intended scopes (V1):
#   - A1c: Year==2023 & hcpcs_stability_group=='2023_only' & cold & expected<1
#   - B2.2b: Year==2023 & hcpcs_stability_group=='2023_only' & cold & HCPCS in {Q5126,Q5127} & oe>=oe_q99
#   - B2.2c: Year==2023 & hcpcs_stability_group=='2023_only' & cold & HCPCS=='Q5127'
# So: ANY row outside (Year==2023 & hcpcs_stability_group=='2023_only' & cold) should be unchanged.
# -----------------------------
base_exp = pd.to_numeric(df_all["expected_cost"], errors="coerce").to_numpy(dtype="float64")
gr_exp   = pd.to_numeric(guarded["expected_cost"], errors="coerce").to_numpy(dtype="float64")

mask_cold = (~df_all["has_lag"].to_numpy())
mask_2023_only_cold = (df_all["Year"].to_numpy() == 2023) & (df_all["hcpcs_stability_group"].to_numpy() == "2023_only") & mask_cold
mask_outside = ~mask_2023_only_cold

unchanged_outside = np.isclose(
    gr_exp[mask_outside],
    base_exp[mask_outside],
    atol=0, rtol=0, equal_nan=True
).all()

print("\nSanity: expected_cost unchanged OUTSIDE (Year==2023 & 2023_only & cold):", bool(unchanged_outside))

# Also check "no-new-cat" inside 2023_only slice (your preferred safety property)
mask_2023_only = (df_all["Year"].to_numpy() == 2023) & (df_all["hcpcs_stability_group"].to_numpy() == "2023_only")
base_any_2023_only = baseline_scored.loc[mask_2023_only, "bl__is_any_catastrophic"].astype(bool).to_numpy()
gr_any_2023_only   = guard_scored.loc[mask_2023_only, "gr__is_any_catastrophic"].astype(bool).to_numpy()
new_cat = gr_any_2023_only & (~base_any_2023_only)
print("No-new-cat check (Year=2023 & 2023_only): new_cat_% =", round(new_cat.mean() * 100, 6))

# -----------------------------
# 5) Summaries (Year x stability_group)
# -----------------------------
def summarize_rates_year(df_flags: pd.DataFrame, prefix: str) -> pd.DataFrame:
    cols = [f"{prefix}is_any_catastrophic"] + [f"{prefix}{b}" for b in CAT_BUCKETS]
    out = (
        df_flags.groupby(["Year", "hcpcs_stability_group"], dropna=False)
        .agg(
            n_rows=("row_id", "size"),
            **{c + "_rate": (c, "mean") for c in cols}
        )
        .reset_index()
    )
    rate_cols = [c for c in out.columns if c.endswith("_rate")]
    out[rate_cols] = out[rate_cols] * 100
    return out

def summarize_counts_year(df_flags: pd.DataFrame, prefix: str) -> pd.DataFrame:
    cols = [f"{prefix}is_any_catastrophic"] + [f"{prefix}{b}" for b in CAT_BUCKETS]
    out = (
        df_flags.groupby(["Year", "hcpcs_stability_group"], dropna=False)[cols]
        .sum()
        .astype(int)
        .reset_index()
    )
    return out

rate_bl = summarize_rates_year(baseline_scored, "bl__").rename(columns=lambda c: c.replace("bl__", "baseline__"))
rate_gr = summarize_rates_year(guard_scored, "gr__").rename(columns=lambda c: c.replace("gr__", "v1__"))

count_bl = summarize_counts_year(baseline_scored, "bl__").rename(columns=lambda c: c.replace("bl__", "baseline__"))
count_gr = summarize_counts_year(guard_scored, "gr__").rename(columns=lambda c: c.replace("gr__", "v1__"))

rate_side = rate_bl.merge(rate_gr, on=["Year", "hcpcs_stability_group", "n_rows"], how="inner")
count_side = count_bl.merge(count_gr, on=["Year", "hcpcs_stability_group"], how="inner")

rate_side = rate_side.sort_values(["Year", "hcpcs_stability_group"]).reset_index(drop=True)
count_side = count_side.sort_values(["Year", "hcpcs_stability_group"]).reset_index(drop=True)

print("\nB2.7.a) Rate table (%, baseline vs V1) | Year x stability_group")
with pd.option_context("display.max_columns", None):
    display(rate_side)

print("\nB2.7.a) Count table (counts, baseline vs V1) | Year x stability_group")
with pd.option_context("display.max_columns", None):
    display(count_side)

delta_any = (
    rate_side[[
        "Year", "hcpcs_stability_group", "n_rows",
        "baseline__is_any_catastrophic_rate",
        "v1__is_any_catastrophic_rate"
    ]]
    .assign(delta_pp=lambda x: x["v1__is_any_catastrophic_rate"] - x["baseline__is_any_catastrophic_rate"])
    .sort_values(["Year", "hcpcs_stability_group"])
    .reset_index(drop=True)
)

print("\nB2.7.a) is_any_catastrophic delta (pp) | Year x stability_group")
display(delta_any)

# Save canonical outputs
baseline_scored_B2_7a = baseline_scored
guard_scored_B2_7a = guard_scored
print("\nSaved: baseline_scored_B2_7a, guard_scored_B2_7a")

## B2.7.b. Evaluate V1 across all years (stability_group within each Year)


In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# B2.7.b) Stability_group within each Year (presentation-friendly)
# - Uses outputs from B2.7.a if present, otherwise recomputes quickly
# ============================================================

# Reuse if available
if "baseline_scored_B2_7a" in globals() and "guard_scored_B2_7a" in globals():
    baseline_scored = baseline_scored_B2_7a.copy()
    guard_scored = guard_scored_B2_7a.copy()
else:
    # fallback: run minimal recompute
    cols_needed = [
        "row_id", "Year", "HCPCS_Cd", "hcpcs_stability_group",
        "has_lag", "route",
        "expected_cost_support_tier",
        "is_top_1pct_avg_mdcr_stdzd_amt",
        "high_confidence_anomaly_candidate",
        "observed_cost", "expected_cost", "residual", "abs_residual", "oe_ratio",
    ]
    df_all = read_pq("failure_df_full_v2", cols=cols_needed)
    for c in ["observed_cost", "expected_cost", "residual", "abs_residual", "oe_ratio"]:
        df_all[c] = pd.to_numeric(df_all[c], errors="coerce")
    df_all["has_lag"] = df_all["has_lag"].astype(bool)

    baseline_scored = apply_catastrophic_flags(df_all.copy(), thresholds_v2, prefix="bl__")
    guarded = apply_guardrail_set_v1(df_all.copy(), guardrail_v1, thresholds_v2)
    guard_scored = apply_catastrophic_flags(guarded.copy(), thresholds_v2, prefix="gr__")

CAT_BUCKETS = [
    "cat_abs_residual_q99",
    "cat_oe_q99",
    "cat_large_positive_residual",
    "cat_large_negative_residual",
    "cat_large_error_high_support",
    "cat_large_error_tail_row",
    "cat_underprediction",
    "cat_overprediction",
    "cat_high_conf_anomaly",
    "cat_cold_low_support_failure",
    "cat_hot_failure",
    "cat_cold_failure",
]

def year_block(year: int):
    bl = baseline_scored.loc[baseline_scored["Year"] == year].copy()
    gr = guard_scored.loc[guard_scored["Year"] == year].copy()

    def summarize(df_flags, prefix):
        cols = [f"{prefix}is_any_catastrophic"] + [f"{prefix}{b}" for b in CAT_BUCKETS]
        out = (
            df_flags.groupby("hcpcs_stability_group", dropna=False)
            .agg(
                n_rows=("row_id", "size"),
                **{c + "_rate": (c, "mean") for c in cols}
            )
            .reset_index()
        )
        rate_cols = [c for c in out.columns if c.endswith("_rate")]
        out[rate_cols] = out[rate_cols] * 100
        return out

    rate_bl = summarize(bl, "bl__").rename(columns=lambda c: c.replace("bl__", "baseline__"))
    rate_gr = summarize(gr, "gr__").rename(columns=lambda c: c.replace("gr__", "v1__"))
    out = rate_bl.merge(rate_gr, on=["hcpcs_stability_group", "n_rows"], how="inner")

    # add delta
    out["delta_pp_is_any"] = out["v1__is_any_catastrophic_rate"] - out["baseline__is_any_catastrophic_rate"]

    # stable ordering if present
    order = pd.CategoricalDtype(categories=["stable_all_years", "other", "2023_only"], ordered=True)
    out["hcpcs_stability_group"] = out["hcpcs_stability_group"].astype(order)
    out = out.sort_values("hcpcs_stability_group")

    return out

years = sorted(pd.unique(baseline_scored["Year"].dropna()).tolist())

for y in years:
    print(f"\nB2.7.b) Year={y} (rates %, baseline vs V1)")
    with pd.option_context("display.max_columns", None):
        display(year_block(int(y)))

# Lock V1 as our current “winner” for the coverage-shift phase

## V1.0 Pin V1 winner (coverage-shift phase)

In [ ]:
import pandas as pd

# ============================================================
# V1.0) Lock V1 as the current winner for the coverage-shift phase
# Requires:
#   - guardrail_v0_policy + apply_guardrail_v0_A1c_2023_only
#   - guardrail_b2_policy + apply_guardrail_B2_2b
#   - guardrail_b2c_policy + apply_guardrail_B2_2c
#   - thresholds_v2
# ============================================================

required = [
    "thresholds_v2",
    "guardrail_v0_policy",
    "guardrail_b2_policy",
    "guardrail_b2c_policy",
    "apply_guardrail_v0_A1c_2023_only",
    "apply_guardrail_B2_2b",
    "apply_guardrail_B2_2c",
]
missing = [x for x in required if x not in globals()]
if missing:
    raise NameError(f"Missing required objects to pin V1: {missing}")

# ============================================================
# V1.0) Lock V1 as the current winner for the coverage-shift phase
# Canonical object: guardrail_v1
# ============================================================

assert "guardrail_v1" in globals(), "guardrail_v1 not found. Run B2.6 first."

# Optional: keep this alias for readability/back-compat, but do not use it elsewhere.
guardrail_winner = guardrail_v1

print("LOCKED winner:", guardrail_v1["name"])

display(pd.DataFrame([{
    "set_name": guardrail_v1["name"],
    "component": c["name"],
    "policy_name": c["policy"].get("name"),
    "scope": c["policy"].get("scope"),
} for c in guardrail_v1["components"]]))

## V1.1 One stable apply helper for V1

> We already wrote a helper above bamed `apply_guardrail_set_v1`. We will use that one in the future. 

## V1.2 Quick “locked winner” sanity checks (scope + no-new-cat)


In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# V1.2) Sanity checks for locked V1 winner
# - Verify scope containment (expected unchanged outside intended scope)
# - Verify no-new-cat on 2023_only slice (apples-to-apples)
# ============================================================

assert "guardrail_v1" in globals(), "guardrail_v1 not found (run B2.6)"
assert "apply_guardrail_set_v1" in globals(), "apply_guardrail_set_v1 not found"
assert "thresholds_v2" in globals(), "thresholds_v2 not found"

YEAR_EVAL = 2023
GROUPS = ["stable_all_years", "2023_only"]

cols_needed = [
    "row_id", "Year", "HCPCS_Cd", "hcpcs_stability_group",
    "has_lag", "route",
    "expected_cost_support_tier",
    "is_top_1pct_avg_mdcr_stdzd_amt",
    "high_confidence_anomaly_candidate",
    "observed_cost", "expected_cost", "residual", "abs_residual", "oe_ratio",
]
df_2023 = read_pq("failure_df_full_v2", cols=cols_needed).copy()
df_2023 = df_2023.loc[(df_2023["Year"] == YEAR_EVAL) & (df_2023["hcpcs_stability_group"].isin(GROUPS))].copy()

for c in ["observed_cost","expected_cost","residual","abs_residual","oe_ratio"]:
    df_2023[c] = pd.to_numeric(df_2023[c], errors="coerce")
df_2023["has_lag"] = df_2023["has_lag"].astype(bool)

# Baseline flags on the same slice
baseline = apply_catastrophic_flags(df_2023.copy(), thresholds_v2, prefix="bl__")

# Apply locked winner
guarded = apply_guardrail_set_v1(df_2023.copy(), guardrail_v1, thresholds_v2)
guarded = apply_catastrophic_flags(guarded, thresholds_v2, prefix="gr__")

# Intended scope for V1 changes (union of components as currently defined)
scope_mask = (
    (df_2023["Year"] == 2023)
    & (df_2023["hcpcs_stability_group"] == "2023_only")
    & (~df_2023["has_lag"])
).to_numpy()

# 1) Scope containment: expected should not change outside intended scope
exp0 = df_2023["expected_cost"].to_numpy(dtype="float64")
exp1 = guarded["expected_cost"].to_numpy(dtype="float64")
unchanged_outside = np.isclose(exp0[~scope_mask], exp1[~scope_mask], atol=0, rtol=0, equal_nan=True).all()
print("Sanity: expected_cost unchanged outside intended scope:", bool(unchanged_outside))

# 2) No-new-cat on 2023_only slice
mask_2023_only = (df_2023["hcpcs_stability_group"] == "2023_only").to_numpy()
base_any = baseline.loc[mask_2023_only, "bl__is_any_catastrophic"].astype(bool).to_numpy()
gr_any   = guarded.loc[mask_2023_only, "gr__is_any_catastrophic"].astype(bool).to_numpy()
new_cat = gr_any & (~base_any)
print("No-new-cat check (2023_only): new_cat_% =", round(new_cat.mean() * 100, 6))

# Optional: headline delta
print("2023_only baseline cat rate (%):", round(base_any.mean() * 100, 6))
print("2023_only V1 cat rate (%):      ", round(gr_any.mean() * 100, 6))
print("delta_pp:", round((gr_any.mean() - base_any.mean()) * 100, 6))

## B2.8) Evaluate V1 for ALL years (A1.5-style table), with "no changes outside scope" checks

In [ ]:
# ============================================================
# B2.8) Evaluate V1 for ALL years (A1.5-style table), with "no changes outside scope" checks
# - Uses failure_df_full_v2 as the baseline scored source
# - Applies V1, then applies catastrophic flags using FIXED thresholds_v2
#
# Outputs:
#   - rate_table_all_years
#   - count_table_all_years
#   - delta_any_all_years
# ============================================================

import numpy as np
import pandas as pd

assert "thresholds_v2" in globals(), "thresholds_v2 not found"
assert "apply_catastrophic_flags" in globals(), "apply_catastrophic_flags not found"
assert "read_pq" in globals(), "read_pq not found"
assert "guardrail_v1" in globals(), "guardrail_v1 not found (run B2.6)"
assert "apply_guardrail_set_v1" in globals(), "apply_guardrail_set_v1 not found (run B2.7)"
assert "guardrail_v0_policy" in globals(), "guardrail_v0_policy not found (A1.4)"

CAT_BUCKETS = [
    "cat_abs_residual_q99",
    "cat_oe_q99",
    "cat_large_positive_residual",
    "cat_large_negative_residual",
    "cat_large_error_high_support",
    "cat_large_error_tail_row",
    "cat_underprediction",
    "cat_overprediction",
    "cat_high_conf_anomaly",
    "cat_cold_low_support_failure",
    "cat_hot_failure",
    "cat_cold_failure",
]

cols_needed = [
    "row_id", "Year", "HCPCS_Cd", "hcpcs_stability_group",
    "has_lag", "route",
    "expected_cost_support_tier",
    "is_top_1pct_avg_mdcr_stdzd_amt",
    "high_confidence_anomaly_candidate",
    "observed_cost", "expected_cost", "residual", "abs_residual", "oe_ratio",
]

df_all = read_pq("failure_df_full_v2", cols=cols_needed)

# Numeric + bool coercions
for c in ["observed_cost", "expected_cost", "residual", "abs_residual", "oe_ratio"]:
    df_all[c] = pd.to_numeric(df_all[c], errors="coerce")
df_all["has_lag"] = df_all["has_lag"].astype(bool)

print("Loaded failure_df_full_v2 for all-years eval. rows:", len(df_all))

# Baseline flags
baseline_all = apply_catastrophic_flags(df_all.copy(), thresholds_v2, prefix="bl__")

# Apply V1
guarded_all = apply_guardrail_set_v1(df_all.copy(), guardrail_v1, thresholds_v2)

# Guarded flags
guarded_all = apply_catastrophic_flags(guarded_all, thresholds_v2, prefix="gr__")

# -----------------------------
# Sanity check: No-new-cat within 2023_only slice (the promise you kept so far)
# -----------------------------
mask_2023_only = (df_all["Year"] == 2023) & (df_all["hcpcs_stability_group"] == "2023_only")
base_any_2023_only = baseline_all.loc[mask_2023_only, "bl__is_any_catastrophic"].astype(bool).to_numpy()
gr_any_2023_only   = guarded_all.loc[mask_2023_only, "gr__is_any_catastrophic"].astype(bool).to_numpy()
new_cat_2023_only = gr_any_2023_only & (~base_any_2023_only)
print("No-new-cat check (Year=2023 & 2023_only): new_cat_% =", round(new_cat_2023_only.mean() * 100, 6))

# -----------------------------
# Sanity check: No expected changes outside intended scopes
# We check "expected_cost_guard" == baseline expected_cost for rows that should never be touched.
#
# You can make this stricter later by enumerating exact scopes per component, but here is a
# practical first pass that catches most accidental bleed:
#   - stable_all_years rows in ANY year should be unchanged by these coverage-shift/code guardrails
# -----------------------------
eps = float(guardrail_v0_policy.get("epsilon", 1e-9))
stable_mask = (df_all["hcpcs_stability_group"] == "stable_all_years").to_numpy()

exp_base = pd.to_numeric(df_all["expected_cost"], errors="coerce").to_numpy(dtype="float64")
exp_guard = pd.to_numeric(guarded_all["expected_cost"], errors="coerce").to_numpy(dtype="float64")

stable_unchanged = np.isclose(exp_guard[stable_mask], exp_base[stable_mask], atol=0, rtol=0, equal_nan=True).all()
print("Sanity: stable_all_years expected unchanged (all years):", bool(stable_unchanged))

# -----------------------------
# A1.5-style summaries by (Year, hcpcs_stability_group)
# -----------------------------
def summarize_rates(df_flags: pd.DataFrame, prefix: str) -> pd.DataFrame:
    cols = [f"{prefix}is_any_catastrophic"] + [f"{prefix}{b}" for b in CAT_BUCKETS]
    out = (
        df_flags.groupby(["Year", "hcpcs_stability_group"], dropna=False)
        .agg(
            n_rows=("row_id", "size"),
            **{c + "_rate": (c, "mean") for c in cols}
        )
        .reset_index()
    )
    rate_cols = [c for c in out.columns if c.endswith("_rate")]
    out[rate_cols] = out[rate_cols] * 100
    return out

def summarize_counts(df_flags: pd.DataFrame, prefix: str) -> pd.DataFrame:
    cols = [f"{prefix}is_any_catastrophic"] + [f"{prefix}{b}" for b in CAT_BUCKETS]
    out = (
        df_flags.groupby(["Year", "hcpcs_stability_group"], dropna=False)[cols]
        .sum()
        .astype(int)
        .reset_index()
    )
    return out

rate_bl = summarize_rates(baseline_all, "bl__").rename(columns=lambda c: c.replace("bl__", "baseline__"))
rate_gr = summarize_rates(guarded_all, "gr__").rename(columns=lambda c: c.replace("gr__", "v1__"))
rate_table_all_years = rate_bl.merge(rate_gr, on=["Year", "hcpcs_stability_group", "n_rows"], how="inner")

count_bl = summarize_counts(baseline_all, "bl__").rename(columns=lambda c: c.replace("bl__", "baseline__"))
count_gr = summarize_counts(guarded_all, "gr__").rename(columns=lambda c: c.replace("gr__", "v1__"))
count_table_all_years = count_bl.merge(count_gr, on=["Year", "hcpcs_stability_group"], how="inner")

print("\nB2.8) Rate table (%, baseline vs V1) | ALL years")
with pd.option_context("display.max_columns", None):
    display(rate_table_all_years.sort_values(["Year", "hcpcs_stability_group"]))

print("\nB2.8) Count table (counts, baseline vs V1) | ALL years")
with pd.option_context("display.max_columns", None):
    display(count_table_all_years.sort_values(["Year", "hcpcs_stability_group"]))

delta_any_all_years = (
    rate_table_all_years[[
        "Year", "hcpcs_stability_group", "n_rows",
        "baseline__is_any_catastrophic_rate",
        "v1__is_any_catastrophic_rate"
    ]]
    .assign(delta_pp=lambda x: x["v1__is_any_catastrophic_rate"] - x["baseline__is_any_catastrophic_rate"])
    .sort_values(["Year", "hcpcs_stability_group"])
)

print("\nB2.8) is_any_catastrophic delta (pp) | ALL years")
display(delta_any_all_years)

# Save canonical
baseline_scored_V1_all_years = baseline_all
guard_scored_V1_all_years = guarded_all
print("\nSaved: baseline_scored_V1_all_years, guard_scored_V1_all_years")

## V1 guardrail set. Coverage-shift phase winner

### Goal
Reduce catastrophic flags caused by the **2023 coverage shift** in the `hcpcs_stability_group == "2023_only"` slice, while guaranteeing:

- **Scope containment**: no changes outside the intended eligibility slices.
- **No-new-cat** within `Year==2023 & hcpcs_stability_group=="2023_only"` (apples-to-apples vs baseline labels computed with fixed `thresholds_v2`).

### Baseline context and definitions
**Dataset**: `failure_df_full_v2` (Parquet, loaded via `read_pq(...)`).

**Key slices**
- `stable_all_years`: codes with stable HCPCS mapping across years.
- `2023_only`: codes that appear only in 2023 (coverage shift).
- `cold_start`: rows with `has_lag == False` (no lag history).

**Baseline scoring**
- Baseline catastrophic flags are computed with a fixed thresholds dictionary `thresholds_v2` using:
  - `apply_catastrophic_flags(df, thresholds_v2, prefix="bl__")`
- All guardrail evaluations reuse the same `thresholds_v2` so comparisons are apples-to-apples.

**Core invariants used throughout**
- Guardrails must not violate: `expected_cost_guard <= observed_cost` for modified rows (within tolerance).
- Guardrails must not introduce new catastrophes in the protected scope, checked via:
  - `new_cat = guard_any & (~baseline_any)` and `new_cat.mean() == 0`.

---

## V1 component 1. A1c (Guardrail V0). Global cold-start floor bounded by observed

### Motivation
In `2023_only` cold-start rows, a small subset had extremely low `expected_cost` values (often `< 1`), which produced extreme O/E ratios and triggered `cat_oe_q99`. A naive global floor can reduce O/E, but must not create negative residual pathologies by forcing expected above observed.

### Policy definition
- **Scope**: apply only to rows satisfying:
  - `Year == 2023`
  - `hcpcs_stability_group == "2023_only"`
  - `has_lag == False` (cold-start)
- **Trigger**: `expected_cost < 1.0`
- **Floor value**: computed from training cold rows:
  - Training years: `2020–2022`
  - Cold: `has_lag == False`
  - `global_cold_floor = median(observed_cost)` on train cold rows
  - In this notebook run: `global_cold_floor ≈ 51.3611976745`
- **Guard rule** (bounded by observed):
  - If triggered, set:
    - `expected_cost_guard = min(observed_cost, max(expected_cost, global_cold_floor))`
  - Else:
    - `expected_cost_guard = expected_cost`

This rule guarantees for modified rows:
- `expected_cost_guard <= observed_cost`
- `expected_cost_guard >= expected_cost`

### Validation and outcome
- Grid search (optional validation) confirmed the best configuration:
  - `trigger_lt = 1.0`
  - `floor = train_cold_median`
  - No new catastrophes produced.
- Impact on `Year==2023 & 2023_only`:
  - Baseline catastrophic rate: `17.100372%`
  - A1c catastrophic rate: `15.892193%`
  - Improvement: `-1.208178 pp`
- Major mover: reduced `cat_oe_q99` only (others unchanged).

### Frozen implementation
- Stored as `guardrail_v0_policy` with keys:
  - `name`, `scope`, `trigger_expected_lt`, `floor_value`, `epsilon`
- Reusable apply function:
  - `apply_guardrail_v0_A1c_2023_only(df, guardrail_v0_policy)`
  - Writes guard columns:
    - `expected_cost_guard`, `residual_guard`, `abs_residual_guard`, `oe_ratio_guard`
  - Marks rows touched with a boolean (e.g. `guardrail_v0_applied`).

---

## V1 component 2. B2.2b. Constraint-based floor for Q5126 and Q5127

### Motivation
After focusing on actual burden inside `2023_only`, the top pain drivers were specific codes:

- `Q5127`: 27 rows, 100% catastrophic
- `Q5126`: 25 rows, 100% catastrophic

These were not “tiny expected” rows. Instead, they were failing hard against global thresholds (O/E and residual constraints). A simple floor triggered on `expected < 1` did nothing because these rows had expected values well above 1.

### Design principle
Use a **constraint-based floor** computed per row so that, when the row is already in extreme territory, we raise expected just enough to satisfy the relevant global thresholds, while staying bounded by observed.

### Policy definition
- **Scope**: apply only to rows satisfying:
  - `Year == 2023`
  - `hcpcs_stability_group == "2023_only"`
  - `has_lag == False` (cold-start)
  - `HCPCS_Cd in {"Q5126", "Q5127"}`
- **Trigger**: `oe_ratio >= thresholds_v2["oe_q99"]` (extreme O/E tail)
- **Guard rule**
  - Compute per-row `floor_needed` as the max of several constraints:
    - `global_cold_floor` (from A1c)
    - `observed_cost / oe_q99` (ensures O/E not above oe_q99 once raised)
    - `observed_cost - resid_pos_q99` (caps positive residual tail)
    - `observed_cost - abs_resid_q99` (caps absolute residual tail)
  - Apply:
    - `expected_cost_guard = min(observed_cost, max(expected_cost, floor_needed))`

### Outcome
- Target slice (Q5126/Q5127 within 2023_only, 52 rows):
  - Baseline cat rate: `100%`
  - Guard cat rate: `51.923077%`
  - Improvement: `-48.076923 pp`
  - No new catastrophes.

- Whole `2023_only` slice (1076 rows):
  - Baseline cat rate: `17.100372%`
  - After B2.2b: `14.776952%`
  - Improvement: `-2.323420 pp`
  - No new catastrophes.

### Key diagnostic finding
`Q5126` reached **0%** catastrophic after B2.2b, but `Q5127` remained **100%** catastrophic due to a remaining bucket:
- `cat_large_positive_residual` was still binding for all Q5127 rows post-B2.2b.

---

## V1 component 3. B2.2c. Residual-cap for Q5127

### Motivation
For `Q5127`, even after constraint floors removed O/E failures, the rows still violated the **positive residual cap**. The “remaining-cat” audit showed:

- All 27 Q5127 rows stayed catastrophic solely because `cat_large_positive_residual` remained true.

### Policy definition
- **Scope**: apply only to rows satisfying:
  - `Year == 2023`
  - `hcpcs_stability_group == "2023_only"`
  - `has_lag == False` (cold-start)
  - `HCPCS_Cd == "Q5127"`
- **Trigger**: eligible rows are those still in the relevant residual tail for this code slice (implementation kept explicit and safe).
- **Residual cap**:
  - `cap = thresholds_v2["resid_pos_q99"] - tiny_epsilon`
- **Guard rule**
  - Enforce `residual_guard <= cap` by raising expected:
    - `expected_cost_guard = max(expected_cost, observed_cost - cap)`
  - Bound by observed:
    - `expected_cost_guard = min(expected_cost_guard, observed_cost)`

### Outcome
- Q5127 slice:
  - Baseline cat rate: `100%`
  - Guard cat rate: `0%`
  - Improvement: `-100 pp`
  - No new catastrophes.
- Whole `2023_only` slice:
  - Baseline: `17.100372%`
  - After B2.2c: `10.687732%` (in the standalone test)
  - When composed with A1c and B2.2b in V1, the final `2023_only` rate became `11.059480%` (see composition below).

---

## V1 composition and evaluation

### Composition order
V1 is applied as a **composed guardrail set** in this fixed order:

1. **A1c V0** (global floor bounded by observed for tiny expected)
2. **B2.2b** (constraint-based floor for Q5126/Q5127)
3. **B2.2c** (residual cap for Q5127)

Composition is implemented by promoting each component’s guarded metrics into the canonical columns (`expected_cost`, `residual`, `abs_residual`, `oe_ratio`) before the next component runs.

### Apply helper
A stable helper was introduced to apply the set in order and handle signature differences:

- `apply_guardrail_set_v1(df, guardrail_set, thresholds_v2)`

It supports components whose apply functions accept either:
- `(df, policy)` or
- `(df, policy, thresholds)`

After each component, it promotes standardized guard columns into canonical columns for the next stage.

### Locked winner object
The pinned guardrail set is stored as a dict (commonly `guardrail_v1` and later also pinned as the winner for the coverage-shift phase):

- `guardrail_v1 = { name, components=[{name, policy, apply_fn}, ...] }`

### V1 headline results (Year=2023)
All-years scope was loaded but changes were confirmed to occur only within intended eligibility.

- `stable_all_years` (2023): unchanged
- `2023_only`:
  - Baseline catastrophic rate: `17.100372%`
  - V1 catastrophic rate: `11.059480%`
  - Improvement: `-6.040892 pp`
- **No-new-cat** on `Year==2023 & 2023_only`: `0.0%`
- **Scope containment**:
  - `expected_cost` unchanged outside the intended scope union.

---

# Section C. Guardrails beyond 2023-only predictions

## C.1. Cold-start hardening diagnostic

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# C.1) Cold-start hardening diagnostic:
# Identify cold-start pain BEYOND the 2023_only slice already handled by V1.
# Leaderboard by Year x support_tier x route.
# ============================================================

assert "baseline_scored_V1_all_years" in globals(), "baseline_scored_V1_all_years not found (run B2.8)"
sc_all = baseline_scored_V1_all_years.copy()

need_cols = [
    "Year", "hcpcs_stability_group", "has_lag",
    "expected_cost_support_tier", "route",
    "row_id",
    "bl__is_any_catastrophic",
    "bl__cat_oe_q99",
    "bl__cat_abs_residual_q99",
    "bl__cat_large_positive_residual",
    "bl__cat_large_negative_residual",
]
missing = [c for c in need_cols if c not in sc_all.columns]
assert not missing, f"C.1 missing columns in baseline_scored_V1_all_years: {missing}"

sc_all["is_cold"] = (~sc_all["has_lag"].astype(bool))

# Cold rows, excluding the 2023_only slice you already handled in V1
exclude_2023_only = (sc_all["Year"] == 2023) & (sc_all["hcpcs_stability_group"] == "2023_only")
cold_non_2023_only = sc_all.loc[sc_all["is_cold"] & (~exclude_2023_only)].copy()

leader = (
    cold_non_2023_only
    .groupby(["Year", "expected_cost_support_tier", "route"], dropna=False)
    .agg(
        n_rows=("row_id", "size"),
        cat_rate=("bl__is_any_catastrophic", "mean"),
        oe_q99_rate=("bl__cat_oe_q99", "mean"),
        abs_resid_q99_rate=("bl__cat_abs_residual_q99", "mean"),
        pos_resid_rate=("bl__cat_large_positive_residual", "mean"),
        neg_resid_rate=("bl__cat_large_negative_residual", "mean"),
    )
    .reset_index()
)

leader["cat_rate_%"] = leader["cat_rate"] * 100
leader["cat_burden_rows"] = (leader["n_rows"] * leader["cat_rate"]).round(2)

print("C.1) Cold-start (non-2023_only) leaderboard by burden (top 30):")
display(
    leader.sort_values("cat_burden_rows", ascending=False)
    .head(30)[
        ["Year","expected_cost_support_tier","route",
         "n_rows","cat_rate_%","cat_burden_rows",
         "oe_q99_rate","abs_resid_q99_rate","pos_resid_rate","neg_resid_rate"]
    ]
)

print("C.1) Cold-start (non-2023_only) leaderboard by rate (top 30, min n>=50):")
display(
    leader.loc[leader["n_rows"] >= 50]
    .sort_values(["cat_rate", "cat_burden_rows"], ascending=[False, False])
    .head(30)[
        ["Year","expected_cost_support_tier","route",
         "n_rows","cat_rate_%","cat_burden_rows",
         "oe_q99_rate","abs_resid_q99_rate","pos_resid_rate","neg_resid_rate"]
    ]
)

## C.2. Drill down into worst cold-start stratum

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# C.2) Drill down into worst cold-start stratum
# Default target: expected_cost_support_tier == "medium_high" AND cold_start (has_lag==False),
# excluding the 2023_only slice already handled by V1.
#
# Answers:
# 1) Which HCPCS codes contribute most catastrophic BURDEN?
# 2) Are failures concentrated in a few routes or spread?
# 3) Failure-mode mix: under/over vs residual vs OE buckets (rates + burden)
#
# Requires:
# - baseline_scored_V1_all_years (or equivalent baseline flags for all years) with:
#     Year, HCPCS_Cd, route, has_lag, expected_cost_support_tier, hcpcs_stability_group
#     bl__is_any_catastrophic + bl__ bucket flags
# ============================================================

# --------- pick baseline all-years df robustly ----------
BASELINE_CANDIDATES = ["baseline_scored_V1_all_years", "baseline_all", "sc_all", "baseline_scored_all_years"]
sc_all = None
for nm in BASELINE_CANDIDATES:
    if nm in globals() and isinstance(globals()[nm], pd.DataFrame):
        sc_all = globals()[nm].copy()
        print(f"Using baseline all-years df: `{nm}` | shape={sc_all.shape}")
        break
if sc_all is None:
    raise NameError(
        "Could not find a baseline all-years scored df. "
        "Expected something like `baseline_scored_V1_all_years` from B2.8."
    )

# --------- ensure required columns ----------
need_cols = [
    "row_id", "Year", "HCPCS_Cd", "route", "has_lag",
    "expected_cost_support_tier", "hcpcs_stability_group",
    "bl__is_any_catastrophic",
]
missing = [c for c in need_cols if c not in sc_all.columns]
if missing:
    raise KeyError(f"Missing required columns in baseline df: {missing}")

sc_all["has_lag"] = sc_all["has_lag"].astype(bool)
sc_all["is_cold"] = ~sc_all["has_lag"]

# --------- define the target stratum ----------
TARGET_TIER = "medium_high"
TARGET_ROUTE = None  # set to a string to narrow (e.g., "cold_start") if route is informative in your data

# Exclude the slice you already handled in V1:
# (Year==2023 & hcpcs_stability_group=="2023_only")
exclude_2023_only = (sc_all["Year"] == 2023) & (sc_all["hcpcs_stability_group"] == "2023_only")

mask_stratum = (
    sc_all["is_cold"]
    & (sc_all["expected_cost_support_tier"] == TARGET_TIER)
    & (~exclude_2023_only)
)

if TARGET_ROUTE is not None and "route" in sc_all.columns:
    mask_stratum &= (sc_all["route"] == TARGET_ROUTE)

stratum = sc_all.loc[mask_stratum].copy()
print("\nC.2) Target stratum:")
print("  tier:", TARGET_TIER)
print("  route filter:", TARGET_ROUTE if TARGET_ROUTE is not None else "(none)")
print("  rows:", len(stratum))
print("  years:", sorted(stratum["Year"].dropna().unique().tolist())[:10], "...")

if len(stratum) == 0:
    raise ValueError("Target stratum is empty. Check tier/route values and baseline df contents.")

# --------- convenience fields ----------
stratum["is_cat"] = stratum["bl__is_any_catastrophic"].astype(bool)
stratum["cat_burden_row"] = stratum["is_cat"].astype(int)

# ============================================================
# 1) HCPCS code burden leaderboard (overall + by year)
# ============================================================
by_code = (
    stratum.groupby("HCPCS_Cd", dropna=False)
    .agg(
        n_rows=("row_id", "size"),
        cat_rate=("is_cat", "mean"),
        cat_burden=("cat_burden_row", "sum"),
    )
    .reset_index()
)
by_code["cat_rate_%"] = by_code["cat_rate"] * 100
by_code = by_code.sort_values(["cat_burden", "n_rows"], ascending=[False, False])

print("\nC.2.1) Top 25 HCPCS codes by catastrophic BURDEN (in stratum):")
display(by_code.head(25)[["HCPCS_Cd", "n_rows", "cat_rate_%", "cat_burden"]])

by_code_year = (
    stratum.groupby(["Year", "HCPCS_Cd"], dropna=False)
    .agg(
        n_rows=("row_id", "size"),
        cat_rate=("is_cat", "mean"),
        cat_burden=("cat_burden_row", "sum"),
    )
    .reset_index()
)
by_code_year["cat_rate_%"] = by_code_year["cat_rate"] * 100
by_code_year = by_code_year.sort_values(["Year", "cat_burden", "n_rows"], ascending=[True, False, False])

print("\nC.2.1b) Top 10 codes by burden WITHIN each year (in stratum):")
display(
    by_code_year.groupby("Year", dropna=False)
    .head(10)[["Year","HCPCS_Cd","n_rows","cat_rate_%","cat_burden"]]
)

# ============================================================
# 2) Route concentration (overall + by year)
# ============================================================
by_route = (
    stratum.groupby("route", dropna=False)
    .agg(
        n_rows=("row_id","size"),
        cat_rate=("is_cat","mean"),
        cat_burden=("cat_burden_row","sum"),
    )
    .reset_index()
)
by_route["cat_rate_%"] = by_route["cat_rate"] * 100
by_route = by_route.sort_values(["cat_burden","n_rows"], ascending=[False, False])

print("\nC.2.2) Route breakdown by BURDEN (in stratum):")
display(by_route)

by_route_year = (
    stratum.groupby(["Year","route"], dropna=False)
    .agg(
        n_rows=("row_id","size"),
        cat_rate=("is_cat","mean"),
        cat_burden=("cat_burden_row","sum"),
    )
    .reset_index()
)
by_route_year["cat_rate_%"] = by_route_year["cat_rate"] * 100
by_route_year = by_route_year.sort_values(["Year","cat_burden"], ascending=[True, False])

print("\nC.2.2b) Route breakdown within each year (in stratum):")
display(by_route_year)

# ============================================================
# 3) Failure-mode mix: under/over vs residual vs OE buckets
#    (rates and burden, overall + optional by year)
# ============================================================
# These should exist from apply_catastrophic_flags thresholds_v2
BUCKETS_FOCUS = [
    "cat_underprediction",
    "cat_overprediction",
    "cat_large_positive_residual",
    "cat_large_negative_residual",
    "cat_abs_residual_q99",
    "cat_oe_q99",
    "cat_large_error_tail_row",
    "cat_large_error_high_support",
    "cat_high_conf_anomaly",
    "cat_cold_low_support_failure",
    "cat_hot_failure",
    "cat_cold_failure",
]
bucket_cols = [f"bl__{b}" for b in BUCKETS_FOCUS if f"bl__{b}" in stratum.columns]
missing_buckets = [b for b in BUCKETS_FOCUS if f"bl__{b}" not in stratum.columns]
if missing_buckets:
    print("\nNOTE: Missing these bucket columns in baseline df (skipping):", missing_buckets)

# overall mix within the stratum
mix_rows = []
for b in BUCKETS_FOCUS:
    col = f"bl__{b}"
    if col not in stratum.columns:
        continue
    rate = stratum[col].mean()
    burden = int(stratum[col].sum())
    mix_rows.append({"bucket": b, "rate_%": rate * 100, "burden_rows": burden})

mix = pd.DataFrame(mix_rows).sort_values(["burden_rows","rate_%"], ascending=[False, False])
print("\nC.2.3) Failure-mode mix (overall in stratum):")
display(mix)

# mix restricted to catastrophic rows (what's driving cats)
cat_only = stratum.loc[stratum["is_cat"]].copy()
mix_cat_rows = []
for b in BUCKETS_FOCUS:
    col = f"bl__{b}"
    if col not in cat_only.columns:
        continue
    rate = cat_only[col].mean()
    mix_cat_rows.append({"bucket": b, "pct_of_cat_rows_%": rate * 100, "n_cat_rows": int(cat_only[col].sum())})

mix_cat = pd.DataFrame(mix_cat_rows).sort_values(["pct_of_cat_rows_%","n_cat_rows"], ascending=[False, False])
print("\nC.2.3b) Failure-mode mix within catastrophic rows only (in stratum):")
display(mix_cat)

# Optional: by year (top 5 buckets by burden per year)
mix_year_rows = []
for yr, g in stratum.groupby("Year", dropna=False):
    for b in BUCKETS_FOCUS:
        col = f"bl__{b}"
        if col not in g.columns:
            continue
        mix_year_rows.append({
            "Year": yr,
            "bucket": b,
            "rate_%": g[col].mean() * 100,
            "burden_rows": int(g[col].sum())
        })
mix_year = pd.DataFrame(mix_year_rows).sort_values(["Year","burden_rows"], ascending=[True, False])

print("\nC.2.3c) Top 8 buckets by burden within each year (in stratum):")
display(mix_year.groupby("Year", dropna=False).head(8))

# ============================================================
# (Optional) Save these tables for downstream cells
# ============================================================
c2_stratum_df = stratum
c2_by_code = by_code
c2_by_code_year = by_code_year
c2_by_route = by_route
c2_by_route_year = by_route_year
c2_mix = mix
c2_mix_cat = mix_cat
c2_mix_year = mix_year

print("\nSaved: c2_stratum_df, c2_by_code, c2_by_code_year, c2_by_route, c2_by_route_year, c2_mix, c2_mix_cat, c2_mix_year")

## C.3. J-codes: J2469 and J2505 and radiation planning cluster as a “family”

## C.3.a. Drilldown: “monster” J-codes (J2469, J2505)

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# C.3.a) Drilldown: “monster” J-codes (J2469, J2505)
# - Baseline-only diagnostics (no guardrails applied)
# - For each code: distributions + bucket rates
# - Split by Year (and optionally focus on medium_high x cold_start)
#
# Outputs:
#   - c3a_df_code: filtered baseline df for these codes
#   - c3a_summary_by_year: per (Year, HCPCS_Cd) numeric dist + bucket rates
#   - c3a_bucket_mix_by_year: long table of bucket rates
# ============================================================

assert "baseline_scored_V1_all_years" in globals(), "baseline_scored_V1_all_years not found (run B2.8)"
assert "thresholds_v2" in globals(), "thresholds_v2 not found"

TARGET_CODES = ["J2469", "J2505"]

# Optional scope filter to match your “worst stratum” (medium_high x cold_start)
# Set to False if you want "all rows for these codes"
FOCUS_STRATUM_ONLY = True
TARGET_TIER = "medium_high"
TARGET_ROUTE = "cold_start"

BUCKETS = [
    "cat_abs_residual_q99",
    "cat_oe_q99",
    "cat_large_positive_residual",
    "cat_large_negative_residual",
    "cat_large_error_high_support",
    "cat_large_error_tail_row",
    "cat_underprediction",
    "cat_overprediction",
    "cat_high_conf_anomaly",
    "cat_cold_low_support_failure",
    "cat_hot_failure",
    "cat_cold_failure",
]

# -----------------------------
# 1) Pull baseline scored df and filter to the two codes
# -----------------------------
df0 = baseline_scored_V1_all_years.copy()

need_cols = [
    "row_id","Year","HCPCS_Cd","has_lag","route","expected_cost_support_tier",
    "expected_cost","observed_cost","residual","abs_residual","oe_ratio",
    "bl__is_any_catastrophic",
] + [f"bl__{b}" for b in BUCKETS]

missing = [c for c in need_cols if c not in df0.columns]
if missing:
    raise ValueError(f"C.3.a missing required cols in baseline_scored_V1_all_years: {missing}")

# numeric coercion (safe even if already numeric)
for c in ["expected_cost","observed_cost","residual","abs_residual","oe_ratio"]:
    df0[c] = pd.to_numeric(df0[c], errors="coerce")
df0["has_lag"] = df0["has_lag"].astype(bool)

mask_code = df0["HCPCS_Cd"].isin(TARGET_CODES)

if FOCUS_STRATUM_ONLY:
    mask_stratum = (
        (df0["expected_cost_support_tier"] == TARGET_TIER)
        & (df0["route"] == TARGET_ROUTE)
        & (~df0["has_lag"])
    )
    c3a_df_code = df0.loc[mask_code & mask_stratum].copy()
else:
    c3a_df_code = df0.loc[mask_code].copy()

print("C.3.a) Rows selected:", len(c3a_df_code))
print("Codes:", TARGET_CODES)
if FOCUS_STRATUM_ONLY:
    print("Focused stratum:", f"{TARGET_TIER} x {TARGET_ROUTE} x cold(has_lag==False)")

display(
    c3a_df_code.groupby(["Year","HCPCS_Cd"], dropna=False)
    .size().rename("n_rows").reset_index()
    .sort_values(["Year","HCPCS_Cd"])
)

# -----------------------------
# 2) Helper: numeric distribution summary
# -----------------------------
def _dist_summary(s: pd.Series) -> dict:
    s = pd.to_numeric(s, errors="coerce")
    return {
        "count": int(s.notna().sum()),
        "mean": float(s.mean()),
        "p10": float(s.quantile(0.10)),
        "p25": float(s.quantile(0.25)),
        "median": float(s.median()),
        "p75": float(s.quantile(0.75)),
        "p90": float(s.quantile(0.90)),
        "max": float(s.max()),
    }

# -----------------------------
# 3) Per (Year, Code) summary: distributions + “failure mode” indicators
# -----------------------------
group_keys = ["Year","HCPCS_Cd"]

rows = []
for (yy, code), g in c3a_df_code.groupby(group_keys, dropna=False):
    r = {"Year": yy, "HCPCS_Cd": code, "n_rows": int(len(g))}
    r["cat_rate_%"] = float(g["bl__is_any_catastrophic"].mean() * 100)

    # core metrics
    for metric in ["expected_cost","observed_cost","residual","oe_ratio"]:
        d = _dist_summary(g[metric])
        for k, v in d.items():
            r[f"{metric}__{k}"] = v

    # quick “which direction” indicators
    # OE spikes (extreme underprediction) if oe_ratio >= oe_q99 (baseline threshold)
    oe_q99 = float(thresholds_v2["oe_q99"])
    r["pct_oe_ge_oe_q99_%"] = float((g["oe_ratio"] >= oe_q99).mean() * 100)

    # large positive/negative residual rates (as % of rows)
    r["pct_large_pos_resid_%"] = float(g["bl__cat_large_positive_residual"].mean() * 100)
    r["pct_large_neg_resid_%"] = float(g["bl__cat_large_negative_residual"].mean() * 100)

    # abs residual q99
    r["pct_abs_resid_q99_%"] = float(g["bl__cat_abs_residual_q99"].mean() * 100)

    rows.append(r)

c3a_summary_by_year = (
    pd.DataFrame(rows)
    .sort_values(["HCPCS_Cd","Year"])
    .reset_index(drop=True)
)

print("\nC.3.a) Per-year distributions + directional indicators")
with pd.option_context("display.max_columns", None):
    display(c3a_summary_by_year)

# -----------------------------
# 4) Bucket rates (long table + wide view)
# -----------------------------
bucket_long = []
for (yy, code), g in c3a_df_code.groupby(group_keys, dropna=False):
    for b in BUCKETS:
        bucket_long.append({
            "Year": yy,
            "HCPCS_Cd": code,
            "bucket": b,
            "rate_%": float(g[f"bl__{b}"].mean() * 100),
            "n_rows": int(len(g)),
        })

c3a_bucket_mix_by_year = (
    pd.DataFrame(bucket_long)
    .sort_values(["HCPCS_Cd","Year","rate_%"], ascending=[True, True, False])
    .reset_index(drop=True)
)

print("\nC.3.a) Bucket mix by year (long). Top buckets first within each (Year, Code)")
display(
    c3a_bucket_mix_by_year
    .groupby(["HCPCS_Cd","Year"], dropna=False)
    .head(8)   # show top 8 buckets per group
    .reset_index(drop=True)
)

# Wide view: bucket rates per (Year, Code)
c3a_bucket_mix_wide = (
    c3a_bucket_mix_by_year.pivot_table(
        index=["Year","HCPCS_Cd","n_rows"],
        columns="bucket",
        values="rate_%",
        aggfunc="first"
    )
    .reset_index()
    .sort_values(["HCPCS_Cd","Year"])
)

print("\nC.3.a) Bucket mix by year (wide)")
with pd.option_context("display.max_columns", None):
    display(c3a_bucket_mix_wide)

print("\nSaved: c3a_df_code, c3a_summary_by_year, c3a_bucket_mix_by_year, c3a_bucket_mix_wide")

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# C.3.b) Monster J-code diagnostics: confirm J2469 is a 2020 shock,
#        and quantify which residual thresholds bind for J2505.
#
# Requires:
#   - c3a_df_code from your C.3.a cell (filtered to medium_high x cold_start x cold)
#     OR, if missing, it will rebuild from baseline_scored_V1_all_years.
#   - thresholds_v2
# ============================================================

assert "thresholds_v2" in globals(), "thresholds_v2 not found"

# --- Resolve source df (prefer your saved C.3.a slice) ---
src = None
if "c3a_df_code" in globals() and isinstance(globals()["c3a_df_code"], pd.DataFrame):
    src = globals()["c3a_df_code"].copy()
else:
    # fallback: rebuild from baseline_scored_V1_all_years if present
    assert "baseline_scored_V1_all_years" in globals(), (
        "Need c3a_df_code or baseline_scored_V1_all_years to rebuild."
    )
    sc = baseline_scored_V1_all_years.copy()
    sc["has_lag"] = sc["has_lag"].astype(bool)

    # Focused stratum you used in C.3.a
    src = sc.loc[
        (sc["expected_cost_support_tier"] == "medium_high")
        & (sc["route"] == "cold_start")
        & (~sc["has_lag"])
        & (sc["HCPCS_Cd"].isin(["J2469", "J2505"]))
    ].copy()

# --- Numeric coercion ---
for c in ["expected_cost", "observed_cost", "residual", "abs_residual", "oe_ratio"]:
    if c in src.columns:
        src[c] = pd.to_numeric(src[c], errors="coerce")

# --- Thresholds ---
ABS_Q99 = float(thresholds_v2.get("abs_resid_q99"))
RESID_POS_Q99 = float(thresholds_v2.get("resid_pos_q99"))

# resid_neg_q99 may or may not exist in your thresholds dict
RESID_NEG_Q99 = thresholds_v2.get("resid_neg_q99", None)
RESID_NEG_Q99 = float(RESID_NEG_Q99) if RESID_NEG_Q99 is not None else None

print("C.3.b thresholds:")
print("  abs_resid_q99:", ABS_Q99)
print("  resid_pos_q99:", RESID_POS_Q99)
print("  resid_neg_q99:", RESID_NEG_Q99, "(None means we approximate via p1 tail)")

# ============================================================
# 1) J2469: confirm 2020 shock (OE + observed/expected medians by year)
# ============================================================
j2469 = src.loc[src["HCPCS_Cd"] == "J2469"].copy()
j2469["obs_over_exp"] = j2469["observed_cost"] / np.maximum(j2469["expected_cost"].to_numpy(dtype="float64"), 1e-12)

j2469_year = (
    j2469.groupby("Year", dropna=False)
    .agg(
        n_rows=("row_id", "size") if "row_id" in j2469.columns else ("expected_cost", "size"),
        median_oe=("oe_ratio", "median"),
        median_obs_over_exp=("obs_over_exp", "median"),
        p25_obs_over_exp=("obs_over_exp", lambda x: np.nanpercentile(x, 25)),
        p75_obs_over_exp=("obs_over_exp", lambda x: np.nanpercentile(x, 75)),
    )
    .reset_index()
    .sort_values("Year")
)

print("\nC.3.b.1) J2469 shock check (by year):")
display(j2469_year)

# Compact bucket mix for J2469 by year: only buckets with >0%
bucket_cols = [
    "cat_abs_residual_q99",
    "cat_oe_q99",
    "cat_large_positive_residual",
    "cat_large_negative_residual",
    "cat_large_error_high_support",
    "cat_large_error_tail_row",
    "cat_underprediction",
    "cat_overprediction",
    "cat_high_conf_anomaly",
    "cat_cold_low_support_failure",
    "cat_hot_failure",
    "cat_cold_failure",
]
bl_bucket_cols = [f"bl__{b}" for b in bucket_cols if f"bl__{b}" in src.columns]

if bl_bucket_cols:
    mix2469 = (
        src.loc[src["HCPCS_Cd"] == "J2469"]
        .groupby(["Year", "HCPCS_Cd"], dropna=False)[bl_bucket_cols]
        .mean()
        .mul(100)
        .reset_index()
        .melt(id_vars=["Year", "HCPCS_Cd"], var_name="bucket", value_name="rate_%")
    )
    mix2469 = mix2469[mix2469["rate_%"] > 0].copy()
    mix2469["bucket"] = mix2469["bucket"].str.replace("bl__", "", regex=False)
    mix2469 = mix2469.sort_values(["Year", "rate_%"], ascending=[True, False])

    print("\nC.3.b.1b) J2469 bucket mix by year (only buckets > 0%):")
    display(mix2469)
else:
    print("\nC.3.b.1b) Skipped J2469 bucket mix: baseline bucket flags (bl__*) not found in src.")

# ============================================================
# 2) J2505: quantify binding thresholds by year
# ============================================================
j2505 = src.loc[src["HCPCS_Cd"] == "J2505"].copy()

# Tail approximation if resid_neg_q99 not present:
# We'll report pct(residual <= p01_residual) as a "negative-tail pressure" proxy.
def p01(x):
    x = pd.to_numeric(x, errors="coerce")
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan
    return float(np.nanpercentile(x, 1))

j2505_year = (
    j2505.groupby("Year", dropna=False)
    .agg(
        n_rows=("row_id", "size") if "row_id" in j2505.columns else ("expected_cost", "size"),
        pct_abs_ge_abs_q99=("abs_residual", lambda x: float(np.nanmean(pd.to_numeric(x, errors="coerce") >= ABS_Q99) * 100)),
        pct_resid_ge_pos_q99=("residual", lambda x: float(np.nanmean(pd.to_numeric(x, errors="coerce") >= RESID_POS_Q99) * 100)),
        resid_p01=("residual", p01),
        resid_p50=("residual", "median"),
        resid_p99=("residual", lambda x: float(np.nanpercentile(pd.to_numeric(x, errors="coerce"), 99))),
    )
    .reset_index()
    .sort_values("Year")
)

# Add negative-threshold metric
if RESID_NEG_Q99 is not None:
    j2505_year["pct_resid_le_neg_q99"] = j2505.groupby("Year")["residual"].apply(
        lambda x: float(np.nanmean(pd.to_numeric(x, errors="coerce") <= RESID_NEG_Q99) * 100)
    ).values
else:
    # proxy: pct(residual <= per-year p01)
    # This answers: how many are in the extreme negative tail, even without a fixed neg threshold.
    per_year_p01 = j2505.groupby("Year")["residual"].apply(p01).to_dict()
    j2505_year["pct_resid_le_p01_proxy"] = j2505.apply(
        lambda r: np.nan, axis=1
    )  # placeholder, overwritten below
    vals = []
    for y, grp in j2505.groupby("Year"):
        thr = per_year_p01.get(y, np.nan)
        v = float(np.nanmean(pd.to_numeric(grp["residual"], errors="coerce") <= thr) * 100) if np.isfinite(thr) else np.nan
        vals.append((y, v))
    vals = dict(vals)
    j2505_year["pct_resid_le_p01_proxy"] = j2505_year["Year"].map(vals)

print("\nC.3.b.2) J2505 threshold binding by year:")
display(j2505_year)

# Compact bucket mix for J2505 by year: only buckets with >0%
if bl_bucket_cols:
    mix2505 = (
        src.loc[src["HCPCS_Cd"] == "J2505"]
        .groupby(["Year", "HCPCS_Cd"], dropna=False)[bl_bucket_cols]
        .mean()
        .mul(100)
        .reset_index()
        .melt(id_vars=["Year", "HCPCS_Cd"], var_name="bucket", value_name="rate_%")
    )
    mix2505 = mix2505[mix2505["rate_%"] > 0].copy()
    mix2505["bucket"] = mix2505["bucket"].str.replace("bl__", "", regex=False)
    mix2505 = mix2505.sort_values(["Year", "rate_%"], ascending=[True, False])

    print("\nC.3.b.2b) J2505 bucket mix by year (only buckets > 0%):")
    display(mix2505)

# ============================================================
# 3) Print a short "fix recipe" based on what we just computed
# ============================================================
print("\nC.3.b.3) Suggested fix recipe (diagnostic-driven):")

# J2469: if 2020 median_obs_over_exp is materially > 1 (and much higher than other years), call it a shock
if not j2469_year.empty:
    med_2020 = j2469_year.loc[j2469_year["Year"] == 2020, "median_obs_over_exp"]
    med_others = j2469_year.loc[j2469_year["Year"] != 2020, "median_obs_over_exp"]
    if len(med_2020) == 1 and len(med_others) > 0:
        ratio = float(med_2020.values[0] / np.nanmedian(med_others.values))
        print(f"- J2469: 2020 median(observed/expected) is ~{med_2020.values[0]:.3f}; "
              f"vs other years median ~{np.nanmedian(med_others.values):.3f} (ratio ~{ratio:.2f}).")
        print("  => Treat as a 2020-specific scale shock. Consider a multiplicative uplift for 2020 cold-start J2469,")
        print("     with safety bound expected_guard <= observed if you want no-new-cat behavior.")
    else:
        print("- J2469: insufficient year coverage to compute shock ratio cleanly, but use the table above.")

# J2505: if abs threshold binding is huge and sign flips, recommend two-sided clamp
if not j2505_year.empty:
    print("- J2505: failures are residual-driven (abs_residual and positive/negative residual buckets), not OE-driven.")
    if RESID_NEG_Q99 is None:
        print("  resid_neg_q99 not available in thresholds_v2, so negative-tail is shown via per-year p01 proxy.")
    print("  => Implement a two-sided residual/abs-residual constraint (clamp) on eligible rows,")
    print("     targeting abs_residual_q99 and both residual tails, while preserving expected_guard <= observed where required.")

## C.4. (J2469 uplift) and (J2505 residual clamp)

### C.4.0 Preconditions + shared config

In [ ]:
import numpy as np
import pandas as pd

assert "thresholds_v2" in globals(), "thresholds_v2 not found"
assert "apply_catastrophic_flags" in globals(), "apply_catastrophic_flags not found"
assert "read_pq" in globals(), "read_pq not found"
assert "apply_guardrail_set_v1" in globals(), "apply_guardrail_set_v1 not found"
assert "guardrail_v1" in globals(), "guardrail_v1 not found"

EPS = float(thresholds_v2.get("epsilon", 1e-9)) if isinstance(thresholds_v2, dict) else 1e-9
oe_q99 = float(thresholds_v2["oe_q99"])
abs_resid_q99 = float(thresholds_v2["abs_resid_q99"])
resid_pos_q99 = float(thresholds_v2["resid_pos_q99"])

print("C.4 thresholds:")
print("  oe_q99:", oe_q99)
print("  abs_resid_q99:", abs_resid_q99)
print("  resid_pos_q99:", resid_pos_q99)
print("  EPS:", EPS)

### C.4.a (J2469 uplift)

#### C.4.a.1 Freeze policy for J2469 uplift (2020 shock)

We use the **diagnostic-driven uplift**: uplift_factor = median(obs/exp) for **J2469 in 2020** (we measured ~2.010610).

In [ ]:
# ============================================================
# C.4.a.1) Policy: J2469 2020 multiplicative uplift (shock fix)
# - Scope: Year==2020, HCPCS_Cd==J2469, medium_high x cold_start x cold(has_lag==False)
# - Trigger: oe_ratio >= oe_q99 (targets OE spikes, avoids touching clean rows)
# - Guard: expected_cost_guard = min(observed_cost, expected_cost * uplift_factor)
# - Keeps expected_guard <= observed (no negative residual created by this rule)
# ============================================================

guardrail_c4a_policy = {
    "name": "C4a_J2469_2020_multiplicative_uplift_bounded_by_observed",
    "scope": "apply ONLY to Year==2020 & HCPCS_Cd=='J2469' & tier=='medium_high' & route=='cold_start' & cold(has_lag==False) & (oe_ratio>=oe_q99)",
    "target_code": "J2469",
    "year": 2020,
    "expected_cost_support_tier": "medium_high",
    "route": "cold_start",
    "oe_trigger": float(oe_q99),
    "uplift_factor": 2.010610,  # from C.3.b median(obs/exp) for 2020 J2469
    "epsilon": float(EPS),
    "hcpcs_stability_group": "other"
}

print("C.4.a policy:")
display(pd.DataFrame([guardrail_c4a_policy]))

#### C.4.a.2 Apply function + evaluate (C4a)

##### C.4.a.2.1 Define apply function: `apply_guardrail_C4a_J2469_uplift`

In [ ]:
import numpy as np
import pandas as pd

def apply_guardrail_C4a_J2469_uplift(df: pd.DataFrame, policy: dict, thresholds: dict) -> pd.DataFrame:
    """
    C4a: multiplicative uplift for J2469 2020 shock (medium_high x cold_start x cold),
    triggered on oe_ratio >= oe_trigger, bounded by observed.

    Writes standardized outputs:
      expected_cost_guard, residual_guard, abs_residual_guard, oe_ratio_guard,
      guardrail_applied, guardrail_name
    """
    out = df.copy()

    req = ["Year","HCPCS_Cd","expected_cost_support_tier","route","has_lag","observed_cost","expected_cost","oe_ratio"]
    missing = [c for c in req if c not in out.columns]
    if missing:
        raise ValueError(f"apply_guardrail_C4a_J2469_uplift missing columns: {missing}")

    EPS = float(policy.get("epsilon", 1e-9))
    target = str(policy["target_code"])
    year = int(policy["year"])
    tier = str(policy["expected_cost_support_tier"])
    route = str(policy["route"])
    oe_trigger = float(policy["oe_trigger"])
    uplift = float(policy["uplift_factor"])

    out["has_lag"] = out["has_lag"].astype(bool)
    obs = pd.to_numeric(out["observed_cost"], errors="coerce").to_numpy(dtype="float64")
    exp = pd.to_numeric(out["expected_cost"], errors="coerce").to_numpy(dtype="float64")
    oe  = pd.to_numeric(out["oe_ratio"], errors="coerce").to_numpy(dtype="float64")

    group = str(policy.get("hcpcs_stability_group", "other"))

    eligible = (
        (out["Year"] == year)
        & (out["hcpcs_stability_group"] == group)
        & (out["HCPCS_Cd"] == target)
        & (out["expected_cost_support_tier"] == tier)
        & (out["route"] == route)
        & (~out["has_lag"])
        & np.isfinite(obs) & np.isfinite(exp) & np.isfinite(oe)
        & (oe >= oe_trigger)
    ).to_numpy()

    exp_guard = exp.copy()
    exp_guard[eligible] = np.minimum(obs[eligible], exp[eligible] * uplift)
    exp_guard[eligible] = np.minimum(exp_guard[eligible], obs[eligible])  # safety

    res_guard = obs - exp_guard
    abs_guard = np.abs(res_guard)
    oe_guard  = obs / np.maximum(exp_guard, EPS)

    out["expected_cost_guard"] = exp_guard
    out["residual_guard"] = res_guard
    out["abs_residual_guard"] = abs_guard
    out["oe_ratio_guard"] = oe_guard
    out["guardrail_applied"] = eligible
    out["guardrail_name"] = str(policy.get("name", "C4a_J2469_uplift"))

    return out

print("Defined apply_guardrail_C4a_J2469_uplift with signature:", apply_guardrail_C4a_J2469_uplift.__name__)

In [ ]:
import inspect
print(inspect.signature(apply_guardrail_C4a_J2469_uplift))

##### C.4.a.2.2 Evaluate C.4.a policy

In [ ]:
import numpy as np
import pandas as pd

assert "apply_guardrail_C4a_J2469_uplift" in globals(), "Run the cell that defines apply_guardrail_C4a_J2469_uplift first."

# -----------------------------
# C.4.a.2) Apply + evaluate (C4a) on focused stratum
# -----------------------------
cols_needed = [
    "row_id","Year","hcpcs_stability_group","HCPCS_Cd","has_lag","route","expected_cost_support_tier",
    "observed_cost","expected_cost","residual","abs_residual","oe_ratio",
    "is_top_1pct_avg_mdcr_stdzd_amt","high_confidence_anomaly_candidate"
]
df_all = read_pq("failure_df_full_v2", cols=cols_needed).copy()

# de-dupe columns defensively
if df_all.columns.duplicated().any():
    dup_cols = df_all.columns[df_all.columns.duplicated()].tolist()
    print("Dropping duplicate columns:", dup_cols)
    df_all = df_all.loc[:, ~df_all.columns.duplicated()].copy()

# Numeric + bool coercions
for c in ["observed_cost","expected_cost","residual","abs_residual","oe_ratio"]:
    df_all[c] = pd.to_numeric(df_all[c], errors="coerce")
df_all["has_lag"] = df_all["has_lag"].astype(bool)

# Focused slice: medium_high x cold_start x cold
mask_focus = (
    (df_all["expected_cost_support_tier"] == "medium_high")
    & (df_all["route"] == "cold_start")
    & (~df_all["has_lag"])
).to_numpy()

focus = df_all.loc[mask_focus].copy()

# Baseline flags on focused slice
baseline_focus = apply_catastrophic_flags(focus.copy(), thresholds_v2, prefix="bl__")

# Apply C4a guardrail (writes *_guard columns)
guarded_focus = apply_guardrail_C4a_J2469_uplift(focus.copy(), guardrail_c4a_policy, thresholds_v2)

# --- quick sanity: did it actually apply to J2469 2020? ---
applied = guarded_focus["guardrail_applied"].to_numpy(dtype=bool)
print("C4a applied rows (total in focus):", int(applied.sum()))

mask_j2469_2020 = (guarded_focus["Year"] == 2020) & (guarded_focus["HCPCS_Cd"] == "J2469")
print("C4a applied rows (J2469 2020):", int((applied & mask_j2469_2020.to_numpy()).sum()))

# Invariant within applied rows: expected_guard <= observed
obs = pd.to_numeric(guarded_focus["observed_cost"], errors="coerce").to_numpy(dtype="float64")
eg  = pd.to_numeric(guarded_focus["expected_cost_guard"], errors="coerce").to_numpy(dtype="float64")
m = applied & np.isfinite(obs) & np.isfinite(eg)
viol = m & (eg > obs + 1e-9)
assert int(viol.sum()) == 0, "C4a invariant violated: expected_guard > observed within applied rows."
print("C4a invariant PASS within applied rows")

# Promote guarded metrics into canonical columns BEFORE scoring
guarded_scored_focus = guarded_focus.copy()
guarded_scored_focus["expected_cost"] = guarded_scored_focus["expected_cost_guard"]
guarded_scored_focus["residual"]      = guarded_scored_focus["residual_guard"]
guarded_scored_focus["abs_residual"]  = guarded_scored_focus["abs_residual_guard"]
guarded_scored_focus["oe_ratio"]      = guarded_scored_focus["oe_ratio_guard"]

# Score on guarded view
guarded_scored_focus = apply_catastrophic_flags(guarded_scored_focus, thresholds_v2, prefix="grC4a__")

# Evaluate: J2469 2020 only
base_any = baseline_focus.loc[mask_j2469_2020, "bl__is_any_catastrophic"].astype(bool).to_numpy()
gr_any   = guarded_scored_focus.loc[mask_j2469_2020, "grC4a__is_any_catastrophic"].astype(bool).to_numpy()
new_cat  = gr_any & (~base_any)

print("\n=== C4a results: J2469 2020 within medium_high cold_start cold ===")
print("n_rows:", int(mask_j2469_2020.sum()))
print("baseline_cat_rate_%:", round(base_any.mean() * 100, 6))
print("guard_cat_rate_%   :", round(gr_any.mean() * 100, 6))
print("delta_pp           :", round((gr_any.mean() - base_any.mean()) * 100, 6))
print("new_cat_%          :", round(new_cat.mean() * 100, 6))

# Optional: show median OE shift (should move strongly if uplift is active)
tmp = guarded_scored_focus.loc[mask_j2469_2020, ["oe_ratio_baseline","oe_ratio_guard"]].copy() if "oe_ratio_baseline" in guarded_scored_focus.columns else None
if tmp is not None:
    print("\nJ2469 2020 median OE baseline:", float(pd.to_numeric(tmp["oe_ratio_baseline"], errors="coerce").median()))
    print("J2469 2020 median OE guard   :", float(pd.to_numeric(tmp["oe_ratio_guard"], errors="coerce").median()))
else:
    # fallback if your apply fn didn't store oe_ratio_baseline
    print("\nJ2469 2020 median OE (guard):", float(pd.to_numeric(guarded_scored_focus.loc[mask_j2469_2020, "oe_ratio"], errors="coerce").median()))

mask_j2469_2020 = (guarded_scored_focus["Year"] == 2020) & (guarded_scored_focus["HCPCS_Cd"] == "J2469")

print("\nJ2469 2020 median OE baseline:",
      float(pd.to_numeric(baseline_focus.loc[mask_j2469_2020, "oe_ratio"], errors="coerce").median()))
print("J2469 2020 median OE guard   :",
      float(pd.to_numeric(guarded_scored_focus.loc[mask_j2469_2020, "oe_ratio"], errors="coerce").median()))

# Save for later pinning
baseline_scored_C4a = baseline_focus
guard_scored_C4a = guarded_scored_focus
print("\nSaved: baseline_scored_C4a, guard_scored_C4a")

### C.4.b J2505 residual clamp

#### C.4.b.1 Freeze policy for J2505 residual clamp (2020–2021)

In [ ]:
import pandas as pd

# ============================================================
# C.4.b.1) Policy: J2505 residual clamp for 2020–2021
# - Scope: Year in {2020, 2021}, HCPCS_Cd==J2505, medium_high x cold_start x cold(has_lag==False)
# - Trigger: abs_residual >= abs_resid_q99 OR residual >= resid_pos_q99 OR residual <= -abs_resid_q99 (proxy)
# - Guard: expected_guard = min(obs, max(exp, obs - cap))
#   where cap = resid_pos_q99 - tiny_eps
# ============================================================


assert "thresholds_v2" in globals(), "thresholds_v2 not found"

EPS = float(thresholds_v2.get("epsilon", 1e-9)) if isinstance(thresholds_v2, dict) else 1e-9

guardrail_c4b_policy = {
    "name": "C4b_J2505_2020_2021_residual_cap_bounded_by_observed",
    "scope": (
        "apply ONLY to Year in {2020,2021} & HCPCS_Cd=='J2505' "
        "& tier=='medium_high' & route=='cold_start' & cold(has_lag==False) "
        "& tail-triggered (abs_resid>=abs_q99 OR resid>=pos_q99 OR resid<=-abs_q99)"
    ),
    "target_code": "J2505",
    "years": [2020, 2021],
    "expected_cost_support_tier": "medium_high",
    "route": "cold_start",

    # The residual cap. Start with the global resid_pos_q99 (same logic as Q5127 cap),
    # optionally subtract a tiny epsilon if you want strict "<" instead of "<=".
    "cap": float(thresholds_v2["resid_pos_q99"]) - 1e-9,

    # carry thresholds explicitly for transparency (your apply fn reads these)
    "abs_resid_q99": float(thresholds_v2["abs_resid_q99"]),
    "resid_pos_q99": float(thresholds_v2["resid_pos_q99"]),
    "epsilon": float(EPS),
    "hcpcs_stability_group": "other"
}

print("C.4.b policy:")
display(pd.DataFrame([guardrail_c4b_policy]))

#### C.4.b.2 Apply function + evaluate (C4b)

##### C.4.b.2.1 define `apply_guardrail_C4b_J2505_resid_cap` function

In [ ]:
import numpy as np
import pandas as pd

def apply_guardrail_C4b_J2505_resid_cap(
    df: pd.DataFrame,
    policy: dict,
    thresholds: dict,
) -> pd.DataFrame:
    """
    C4b: residual clamp for J2505 (2020–2021), implemented safely with expected<=observed.
    Writes standardized outputs:
      expected_cost_guard, residual_guard, abs_residual_guard, oe_ratio_guard,
      guardrail_applied, guardrail_name
    """
    out = df.copy()

    req = ["Year","hcpcs_stability_group","HCPCS_Cd","has_lag","route","expected_cost_support_tier",
           "observed_cost","expected_cost","residual","abs_residual","oe_ratio"]
    missing = [c for c in req if c not in out.columns]
    if missing:
        raise ValueError(f"apply_guardrail_C4b_J2505_resid_cap missing cols: {missing}")

    for c in ["observed_cost","expected_cost","residual","abs_residual","oe_ratio"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    obs = out["observed_cost"].to_numpy(dtype="float64")
    exp = out["expected_cost"].to_numpy(dtype="float64")
    resid = out["residual"].to_numpy(dtype="float64")
    absr  = out["abs_residual"].to_numpy(dtype="float64")
    oe    = out["oe_ratio"].to_numpy(dtype="float64")

    years = set(int(x) for x in policy["years"])
    code = str(policy["target_code"])
    tier = str(policy["expected_cost_support_tier"])
    route = str(policy["route"])

    cap = float(policy["cap"])
    EPS = float(policy.get("epsilon", 1e-9))
    abs_q99 = float(policy.get("abs_resid_q99", thresholds["abs_resid_q99"]))
    pos_q99 = float(policy.get("resid_pos_q99", thresholds["resid_pos_q99"]))

    group = str(policy.get("hcpcs_stability_group", "other"))

    eligible_scope = (
        (out["Year"].isin(years))
        & (out["hcpcs_stability_group"] == group)
        & (out["HCPCS_Cd"] == code)
        & (out["expected_cost_support_tier"] == tier)
        & (out["route"] == route)
        & (~out["has_lag"].astype(bool))
        & np.isfinite(obs) & np.isfinite(exp) & np.isfinite(resid) & np.isfinite(absr) & np.isfinite(oe)
    ).to_numpy()

    # tail trigger
    trigger_tail = (
        (absr >= abs_q99) | (resid >= pos_q99) | (resid <= -abs_q99)
    )

    eligible = eligible_scope & trigger_tail

    # Two-sided behavior via one safe formula:
    # - if exp > obs, min(obs, ...) clamps down to obs (kills big negative residual)
    # - if obs-exp > cap, max(exp, obs-cap) raises expected to obs-cap (caps positive residual)
    exp_guard = exp.copy()
    exp_guard[eligible] = np.minimum(
        obs[eligible],
        np.maximum(exp[eligible], (obs[eligible] - cap))
    )

    # safety net
    exp_guard[eligible] = np.minimum(exp_guard[eligible], obs[eligible])

    res_guard = obs - exp_guard
    abs_guard = np.abs(res_guard)
    oe_guard  = obs / np.maximum(exp_guard, EPS)

    out["expected_cost_guard"] = exp_guard
    out["residual_guard"] = res_guard
    out["abs_residual_guard"] = abs_guard
    out["oe_ratio_guard"] = oe_guard
    out["guardrail_applied"] = eligible
    out["guardrail_name"] = str(policy.get("name", "C4b"))

    return out

print("Defined apply_guardrail_C4b_J2505_resid_cap with signature:", apply_guardrail_C4b_J2505_resid_cap.__name__)

In [ ]:
import inspect
print(inspect.signature(apply_guardrail_C4b_J2505_resid_cap))

##### C.4.b.2.2 Evaluate C.4.b policy

In [ ]:
import numpy as np
import pandas as pd

assert "apply_guardrail_C4b_J2505_resid_cap" in globals(), "Define apply_guardrail_C4b_J2505_resid_cap first."
assert "guardrail_c4b_policy" in globals(), "Define guardrail_c4b_policy first."
assert "thresholds_v2" in globals()
assert "apply_catastrophic_flags" in globals()
assert "read_pq" in globals()

# -----------------------------
# Build / reuse focused slice
# -----------------------------
if "df_all" not in globals():
    cols_needed = [
        "row_id","Year","HCPCS_Cd","has_lag","route","expected_cost_support_tier",
        "observed_cost","expected_cost","residual","abs_residual","oe_ratio",
        "is_top_1pct_avg_mdcr_stdzd_amt","high_confidence_anomaly_candidate"
    ]
    df_all = read_pq("failure_df_full_v2", cols=cols_needed).copy()
    for c in ["observed_cost","expected_cost","residual","abs_residual","oe_ratio"]:
        df_all[c] = pd.to_numeric(df_all[c], errors="coerce")
    df_all["has_lag"] = df_all["has_lag"].astype(bool)

mask_focus = (
    (df_all["expected_cost_support_tier"] == "medium_high")
    & (df_all["route"] == "cold_start")
    & (~df_all["has_lag"])
).to_numpy()

focus = df_all.loc[mask_focus].copy()

# Baseline scoring on focused slice
baseline_focus = apply_catastrophic_flags(focus.copy(), thresholds_v2, prefix="bl__")

# Apply guardrail (writes *_guard columns)
guarded_focus_raw = apply_guardrail_C4b_J2505_resid_cap(focus.copy(), guardrail_c4b_policy, thresholds_v2)

# Invariant within applied rows: expected_guard <= observed
applied = guarded_focus_raw["guardrail_applied"].to_numpy(dtype=bool)
obs = pd.to_numeric(guarded_focus_raw["observed_cost"], errors="coerce").to_numpy(dtype="float64")
eg  = pd.to_numeric(guarded_focus_raw["expected_cost_guard"], errors="coerce").to_numpy(dtype="float64")
m = applied & np.isfinite(obs) & np.isfinite(eg)
viol = m & (eg > obs + 1e-9)
assert int(viol.sum()) == 0, "C4b invariant violated: expected_guard > observed within applied rows."
print("C4b invariant PASS within applied rows")
print("C4b applied rows (in focus):", int(applied.sum()))

# Promote guarded metrics into canonical columns BEFORE scoring
guarded_focus = guarded_focus_raw.copy()
guarded_focus["expected_cost"] = guarded_focus["expected_cost_guard"]
guarded_focus["residual"]      = guarded_focus["residual_guard"]
guarded_focus["abs_residual"]  = guarded_focus["abs_residual_guard"]
guarded_focus["oe_ratio"]      = guarded_focus["oe_ratio_guard"]

# Score on guarded view
guarded_focus = apply_catastrophic_flags(guarded_focus, thresholds_v2, prefix="grC4b__")

# Evaluate: J2505 2020 and 2021 separately
for yr in [2020, 2021]:
    mask = (baseline_focus["Year"] == yr) & (baseline_focus["HCPCS_Cd"] == "J2505")
    base_any = baseline_focus.loc[mask, "bl__is_any_catastrophic"].astype(bool).to_numpy()
    gr_any   = guarded_focus.loc[mask, "grC4b__is_any_catastrophic"].astype(bool).to_numpy()
    new_cat = gr_any & (~base_any)

    print(f"\n=== C4b results: J2505 {yr} within medium_high cold_start cold ===")
    print("n_rows:", int(mask.sum()))
    print("baseline_cat_rate_%:", round(base_any.mean() * 100, 6))
    print("guard_cat_rate_%   :", round(gr_any.mean() * 100, 6))
    print("delta_pp           :", round((gr_any.mean() - base_any.mean()) * 100, 6))
    print("new_cat_%          :", round(new_cat.mean() * 100, 6))

# Save for later pinning
baseline_scored_C4b = baseline_focus
guard_scored_C4b = guarded_focus
print("\nSaved: baseline_scored_C4b, guard_scored_C4b")

### C4.3 Pin winners + sanity checks

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# C4.3) Pin C4a + C4b winners + sanity checks
# Requires:
#   - baseline_scored_C4a, guard_scored_C4a
#   - baseline_scored_C4b, guard_scored_C4b
#   - guardrail_c4a_policy, guardrail_c4b_policy
# ============================================================

required = [
    "baseline_scored_C4a", "guard_scored_C4a",
    "baseline_scored_C4b", "guard_scored_C4b",
    "guardrail_c4a_policy", "guardrail_c4b_policy",
]
missing = [x for x in required if x not in globals()]
if missing:
    raise NameError(f"Missing required objects for C4.3 pinning: {missing}")

# Canonical pins
baseline_scored_C4a_best = baseline_scored_C4a
guard_scored_C4a_best    = guard_scored_C4a

baseline_scored_C4b_best = baseline_scored_C4b
guard_scored_C4b_best    = guard_scored_C4b

print("Pinned:")
print("  guard_scored_C4a_best:", guard_scored_C4a_best.shape)
print("  guard_scored_C4b_best:", guard_scored_C4b_best.shape)

def _inv_viol(df, applied_col="guardrail_applied", tol=1e-9):
    applied = df[applied_col].astype(bool).to_numpy()
    oc = pd.to_numeric(df["observed_cost"], errors="coerce").to_numpy(dtype="float64")
    eg = pd.to_numeric(df["expected_cost_guard"], errors="coerce").to_numpy(dtype="float64")
    m = applied & np.isfinite(oc) & np.isfinite(eg)
    return int((m & (eg > oc + tol)).sum()), int(m.sum())

# C4a sanity
n_viol_a, n_chk_a = _inv_viol(guard_scored_C4a_best)
print("\nC4a sanity")
print("  applied rows:", int(pd.to_numeric(guard_scored_C4a_best["guardrail_applied"], errors="coerce").fillna(False).astype(bool).sum()))
print("  invariant violations:", n_viol_a, "| checked:", n_chk_a)

# C4b sanity
n_viol_b, n_chk_b = _inv_viol(guard_scored_C4b_best)
print("\nC4b sanity")
print("  applied rows:", int(pd.to_numeric(guard_scored_C4b_best["guardrail_applied"], errors="coerce").fillna(False).astype(bool).sum()))
print("  invariant violations:", n_viol_b, "| checked:", n_chk_b)

assert n_viol_a == 0, "C4a invariant violated: expected_cost_guard > observed_cost in applied rows"
assert n_viol_b == 0, "C4b invariant violated: expected_cost_guard > observed_cost in applied rows"

# No-new-cat checks (focused slices)
def _no_new_cat(baseline_df, guard_df, base_any_col, guard_any_col):
    base_any = baseline_df[base_any_col].astype(bool).to_numpy()
    gr_any   = guard_df[guard_any_col].astype(bool).to_numpy()
    new_cat  = gr_any & (~base_any)
    return float(new_cat.mean() * 100)

if "bl__is_any_catastrophic" in baseline_scored_C4a_best.columns and "grC4a__is_any_catastrophic" in guard_scored_C4a_best.columns:
    print("\nC4a no-new-cat (%):", round(_no_new_cat(baseline_scored_C4a_best, guard_scored_C4a_best, "bl__is_any_catastrophic", "grC4a__is_any_catastrophic"), 6))
if "bl__is_any_catastrophic" in baseline_scored_C4b_best.columns and "grC4b__is_any_catastrophic" in guard_scored_C4b_best.columns:
    print("C4b no-new-cat (%):", round(_no_new_cat(baseline_scored_C4b_best, guard_scored_C4b_best, "bl__is_any_catastrophic", "grC4b__is_any_catastrophic"), 6))

print("\nPASS: C4a/C4b pinned + invariants OK.")

### C4.4 Freeze policies + apply functions (C4)


In [ ]:
import pandas as pd

# ============================================================
# C4.4) Freeze C4 policies + confirm apply functions exist
# ============================================================

required = [
    "guardrail_c4a_policy", "guardrail_c4b_policy",
    "apply_guardrail_C4a_J2469_uplift", "apply_guardrail_C4b_J2505_resid_cap",
]
missing = [x for x in required if x not in globals()]
if missing:
    raise NameError(f"Missing required objects for C4.4: {missing}")

guardrail_c4_policies = pd.DataFrame([guardrail_c4a_policy, guardrail_c4b_policy])
print("C4 policies:")
display(guardrail_c4_policies)

print("C4 apply fns:")
print("  C4a fn:", apply_guardrail_C4a_J2469_uplift.__name__)
print("  C4b fn:", apply_guardrail_C4b_J2505_resid_cap.__name__)

### C4.5 Pin composed winner set as V2 (V1 + C4)


In [ ]:
import pandas as pd

# ============================================================
# C4.5) Pin composed set as V2 = V1 + C4a + C4b
# Requires:
#   - guardrail_v1 (B2.6)
#   - apply_guardrail_set_v1
#   - guardrail_c4a_policy, guardrail_c4b_policy
#   - apply_guardrail_C4a_J2469_uplift, apply_guardrail_C4b_J2505_resid_cap
# ============================================================

required = [
    "guardrail_v1", "apply_guardrail_set_v1", "thresholds_v2",
    "guardrail_c4a_policy", "guardrail_c4b_policy",
    "apply_guardrail_C4a_J2469_uplift", "apply_guardrail_C4b_J2505_resid_cap",
]
missing = [x for x in required if x not in globals()]
if missing:
    raise NameError(f"Missing required objects for C4.5: {missing}")

guardrail_v2 = {
    "name": "V2_coverage_shift_plus_top_burden_codes_plus_monster_J_fixes",
    "phase": "cold_start_hardening",
    "components": (
        guardrail_v1["components"]
        + [
            {"name": "C4a_J2469_uplift", "policy": guardrail_c4a_policy, "apply_fn": apply_guardrail_C4a_J2469_uplift},
            {"name": "C4b_J2505_resid_cap", "policy": guardrail_c4b_policy, "apply_fn": apply_guardrail_C4b_J2505_resid_cap},
        ]
    ),
}

print("Pinned guardrail set:", guardrail_v2["name"])
display(pd.DataFrame([{
    "set_name": guardrail_v2["name"],
    "component": c["name"],
    "policy_name": c["policy"].get("name"),
    "scope": c["policy"].get("scope"),
} for c in guardrail_v2["components"]]))

### C4.6 Evaluate V2 across all years (Year × stability_group), with scope checks


In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# C4.6) Evaluate V2 across all years (Year × stability_group)
# Baseline vs V2 using FIXED thresholds_v2
# Outputs:
#   - baseline_scored_C4_6
#   - guard_scored_C4_6
# ============================================================

assert "guardrail_v2" in globals(), "guardrail_v2 not found (run C4.5)"
assert "apply_guardrail_set_v1" in globals(), "apply_guardrail_set_v1 not found"
assert "thresholds_v2" in globals(), "thresholds_v2 not found"
assert "apply_catastrophic_flags" in globals(), "apply_catastrophic_flags not found"
assert "read_pq" in globals(), "read_pq not found"
assert "guardrail_v0_policy" in globals(), "guardrail_v0_policy not found"

CAT_BUCKETS = [
    "cat_abs_residual_q99","cat_oe_q99","cat_large_positive_residual","cat_large_negative_residual",
    "cat_large_error_high_support","cat_large_error_tail_row","cat_underprediction","cat_overprediction",
    "cat_high_conf_anomaly","cat_cold_low_support_failure","cat_hot_failure","cat_cold_failure",
]

cols_needed = [
    "row_id","Year","HCPCS_Cd","hcpcs_stability_group","has_lag","route",
    "expected_cost_support_tier","is_top_1pct_avg_mdcr_stdzd_amt","high_confidence_anomaly_candidate",
    "observed_cost","expected_cost","residual","abs_residual","oe_ratio",
]
df_all = read_pq("failure_df_full_v2", cols=cols_needed).copy()

for c in ["observed_cost","expected_cost","residual","abs_residual","oe_ratio"]:
    df_all[c] = pd.to_numeric(df_all[c], errors="coerce")
df_all["has_lag"] = df_all["has_lag"].astype(bool)

print("Rows loaded:", len(df_all))
display(
    df_all.groupby(["Year","hcpcs_stability_group"], dropna=False)
    .size().rename("n_rows").reset_index().sort_values(["Year","hcpcs_stability_group"])
)

# Baseline flags
baseline_all = apply_catastrophic_flags(df_all.copy(), thresholds_v2, prefix="bl__")

# Apply V2
guarded_all = apply_guardrail_set_v1(df_all.copy(), guardrail_v2, thresholds_v2)
guarded_all = apply_catastrophic_flags(guarded_all, thresholds_v2, prefix="gr__")

# -----------------------------
# Sanity 1: stable_all_years expected unchanged (all years)
# -----------------------------
stable_mask = (df_all["hcpcs_stability_group"] == "stable_all_years").to_numpy()
exp0 = pd.to_numeric(df_all["expected_cost"], errors="coerce").to_numpy(dtype="float64")
exp1 = pd.to_numeric(guarded_all["expected_cost"], errors="coerce").to_numpy(dtype="float64")
stable_unchanged = np.isclose(exp0[stable_mask], exp1[stable_mask], atol=0, rtol=0, equal_nan=True).all()
print("\nSanity: stable_all_years expected unchanged (all years):", bool(stable_unchanged))

# -----------------------------
# Sanity 2: no-new-cat promise on Year=2023 & 2023_only
# -----------------------------
mask_2023_only = ((df_all["Year"] == 2023) & (df_all["hcpcs_stability_group"] == "2023_only")).to_numpy()
base_any = baseline_all.loc[mask_2023_only, "bl__is_any_catastrophic"].astype(bool).to_numpy()
gr_any   = guarded_all.loc[mask_2023_only, "gr__is_any_catastrophic"].astype(bool).to_numpy()
new_cat = gr_any & (~base_any)
print("No-new-cat check (Year=2023 & 2023_only): new_cat_% =", round(new_cat.mean() * 100, 6))

# -----------------------------
# A1.5-style summaries by (Year, hcpcs_stability_group)
# -----------------------------
def summarize_rates(df_flags: pd.DataFrame, prefix: str) -> pd.DataFrame:
    cols = [f"{prefix}is_any_catastrophic"] + [f"{prefix}{b}" for b in CAT_BUCKETS]
    out = (
        df_flags.groupby(["Year","hcpcs_stability_group"], dropna=False)
        .agg(n_rows=("row_id","size"), **{c + "_rate": (c, "mean") for c in cols})
        .reset_index()
    )
    rate_cols = [c for c in out.columns if c.endswith("_rate")]
    out[rate_cols] = out[rate_cols] * 100
    return out

def summarize_counts(df_flags: pd.DataFrame, prefix: str) -> pd.DataFrame:
    cols = [f"{prefix}is_any_catastrophic"] + [f"{prefix}{b}" for b in CAT_BUCKETS]
    out = (
        df_flags.groupby(["Year","hcpcs_stability_group"], dropna=False)[cols]
        .sum().astype(int).reset_index()
    )
    return out

rate_bl = summarize_rates(baseline_all, "bl__").rename(columns=lambda c: c.replace("bl__", "baseline__"))
rate_gr = summarize_rates(guarded_all, "gr__").rename(columns=lambda c: c.replace("gr__", "v2__"))
rate_table = rate_bl.merge(rate_gr, on=["Year","hcpcs_stability_group","n_rows"], how="inner")

count_bl = summarize_counts(baseline_all, "bl__").rename(columns=lambda c: c.replace("bl__", "baseline__"))
count_gr = summarize_counts(guarded_all, "gr__").rename(columns=lambda c: c.replace("gr__", "v2__"))
count_table = count_bl.merge(count_gr, on=["Year","hcpcs_stability_group"], how="inner")

print("\nC4.6) Rate table (%, baseline vs V2) | ALL years")
with pd.option_context("display.max_columns", None):
    display(rate_table.sort_values(["Year","hcpcs_stability_group"]))

print("\nC4.6) Count table (counts, baseline vs V2) | ALL years")
with pd.option_context("display.max_columns", None):
    display(count_table.sort_values(["Year","hcpcs_stability_group"]))

delta_any = (
    rate_table[["Year","hcpcs_stability_group","n_rows","baseline__is_any_catastrophic_rate","v2__is_any_catastrophic_rate"]]
    .assign(delta_pp=lambda x: x["v2__is_any_catastrophic_rate"] - x["baseline__is_any_catastrophic_rate"])
    .sort_values(["Year","hcpcs_stability_group"])
)
print("\nC4.6) is_any_catastrophic delta (pp) | ALL years")
display(delta_any)

# Save canonical
baseline_scored_C4_6 = baseline_all
guard_scored_C4_6 = guarded_all
print("\nSaved: baseline_scored_C4_6, guard_scored_C4_6")

### C4.7 All-years eval cell, “B2.8 style” but for V2


In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# C4.7) Evaluate V2 for ALL years (B2.8-style), with scope checks
# Outputs:
#   - rate_table_V2_all_years
#   - count_table_V2_all_years
#   - delta_any_V2_all_years
#   - baseline_scored_V2_all_years
#   - guard_scored_V2_all_years
# ============================================================

assert "thresholds_v2" in globals(), "thresholds_v2 not found"
assert "apply_catastrophic_flags" in globals(), "apply_catastrophic_flags not found"
assert "read_pq" in globals(), "read_pq not found"
assert "guardrail_v2" in globals(), "guardrail_v2 not found (run C4.5)"
assert "apply_guardrail_set_v1" in globals(), "apply_guardrail_set_v1 not found"
assert "guardrail_v0_policy" in globals(), "guardrail_v0_policy not found (A1.4)"

CAT_BUCKETS = [
    "cat_abs_residual_q99","cat_oe_q99","cat_large_positive_residual","cat_large_negative_residual",
    "cat_large_error_high_support","cat_large_error_tail_row","cat_underprediction","cat_overprediction",
    "cat_high_conf_anomaly","cat_cold_low_support_failure","cat_hot_failure","cat_cold_failure",
]

cols_needed = [
    "row_id","Year","HCPCS_Cd","hcpcs_stability_group","has_lag","route",
    "expected_cost_support_tier","is_top_1pct_avg_mdcr_stdzd_amt","high_confidence_anomaly_candidate",
    "observed_cost","expected_cost","residual","abs_residual","oe_ratio",
]
df_all = read_pq("failure_df_full_v2", cols=cols_needed).copy()

for c in ["observed_cost","expected_cost","residual","abs_residual","oe_ratio"]:
    df_all[c] = pd.to_numeric(df_all[c], errors="coerce")
df_all["has_lag"] = df_all["has_lag"].astype(bool)

print("Loaded failure_df_full_v2 for all-years eval. rows:", len(df_all))

baseline_all = apply_catastrophic_flags(df_all.copy(), thresholds_v2, prefix="bl__")
guarded_all  = apply_guardrail_set_v1(df_all.copy(), guardrail_v2, thresholds_v2)
guarded_all  = apply_catastrophic_flags(guarded_all, thresholds_v2, prefix="gr__")

# No-new-cat on 2023_only
mask_2023_only = ((df_all["Year"] == 2023) & (df_all["hcpcs_stability_group"] == "2023_only")).to_numpy()
base_any = baseline_all.loc[mask_2023_only, "bl__is_any_catastrophic"].astype(bool).to_numpy()
gr_any   = guarded_all.loc[mask_2023_only, "gr__is_any_catastrophic"].astype(bool).to_numpy()
new_cat = gr_any & (~base_any)
print("No-new-cat check (Year=2023 & 2023_only): new_cat_% =", round(new_cat.mean() * 100, 6))

# stable_all_years expected unchanged
stable_mask = (df_all["hcpcs_stability_group"] == "stable_all_years").to_numpy()
exp0 = pd.to_numeric(df_all["expected_cost"], errors="coerce").to_numpy(dtype="float64")
exp1 = pd.to_numeric(guarded_all["expected_cost"], errors="coerce").to_numpy(dtype="float64")
stable_unchanged = np.isclose(exp0[stable_mask], exp1[stable_mask], atol=0, rtol=0, equal_nan=True).all()
print("Sanity: stable_all_years expected unchanged (all years):", bool(stable_unchanged))

def summarize_rates(df_flags: pd.DataFrame, prefix: str) -> pd.DataFrame:
    cols = [f"{prefix}is_any_catastrophic"] + [f"{prefix}{b}" for b in CAT_BUCKETS]
    out = (
        df_flags.groupby(["Year","hcpcs_stability_group"], dropna=False)
        .agg(n_rows=("row_id","size"), **{c + "_rate": (c, "mean") for c in cols})
        .reset_index()
    )
    rate_cols = [c for c in out.columns if c.endswith("_rate")]
    out[rate_cols] = out[rate_cols] * 100
    return out

def summarize_counts(df_flags: pd.DataFrame, prefix: str) -> pd.DataFrame:
    cols = [f"{prefix}is_any_catastrophic"] + [f"{prefix}{b}" for b in CAT_BUCKETS]
    out = (
        df_flags.groupby(["Year","hcpcs_stability_group"], dropna=False)[cols]
        .sum().astype(int).reset_index()
    )
    return out

rate_bl = summarize_rates(baseline_all, "bl__").rename(columns=lambda c: c.replace("bl__", "baseline__"))
rate_gr = summarize_rates(guarded_all, "gr__").rename(columns=lambda c: c.replace("gr__", "v2__"))
rate_table_V2_all_years = rate_bl.merge(rate_gr, on=["Year","hcpcs_stability_group","n_rows"], how="inner")

count_bl = summarize_counts(baseline_all, "bl__").rename(columns=lambda c: c.replace("bl__", "baseline__"))
count_gr = summarize_counts(guarded_all, "gr__").rename(columns=lambda c: c.replace("gr__", "v2__"))
count_table_V2_all_years = count_bl.merge(count_gr, on=["Year","hcpcs_stability_group"], how="inner")

print("\nC4.7) Rate table (%, baseline vs V2) | ALL years")
with pd.option_context("display.max_columns", None):
    display(rate_table_V2_all_years.sort_values(["Year","hcpcs_stability_group"]))

print("\nC4.7) Count table (counts, baseline vs V2) | ALL years")
with pd.option_context("display.max_columns", None):
    display(count_table_V2_all_years.sort_values(["Year","hcpcs_stability_group"]))

delta_any_V2_all_years = (
    rate_table_V2_all_years[["Year","hcpcs_stability_group","n_rows","baseline__is_any_catastrophic_rate","v2__is_any_catastrophic_rate"]]
    .assign(delta_pp=lambda x: x["v2__is_any_catastrophic_rate"] - x["baseline__is_any_catastrophic_rate"])
    .sort_values(["Year","hcpcs_stability_group"])
)
print("\nC4.7) is_any_catastrophic delta (pp) | ALL years")
display(delta_any_V2_all_years)

baseline_scored_V2_all_years = baseline_all
guard_scored_V2_all_years = guarded_all
print("\nSaved: baseline_scored_V2_all_years, guard_scored_V2_all_years")

## V2 guardrail set. V1 plus “monster J-code” fixes (cold-start hardening)

### Goal
Extend the cold-start hardening approach to the largest non-2023_only cold-start burden stratum:

- Focus: `expected_cost_support_tier == "medium_high"`, `route == "cold_start"`, `has_lag == False`
- Largest burden was in `other` stability group, with extreme failures concentrated in:
  - `J2469` (mostly 2020, OE shock)
  - `J2505` (2020–2021, residual-driven tails)

V2 preserves V1 guarantees while adding targeted fixes:
- Maintain **stable_all_years unchanged**.
- Maintain **no-new-cat on 2023_only**.
- Keep changes tightly scoped to explicitly intended strata.

---

### V2 component 4. C4a. Multiplicative uplift for J2469 (2020 shock)

#### Diagnosis summary (from C.3.a and C.3.b)
- `J2469` in 2020 had a clear **multiplicative shock**:
  - 2020 median observed/expected ≈ `2.01`
  - Other years median observed/expected ≈ `0.56–1.06`
- Bucket mix: almost entirely `cat_oe_q99` in 2020.
- This suggested a 2020-only multiplicative uplift.

#### Policy definition
- **Scope**:
  - `Year == 2020`
  - `HCPCS_Cd == "J2469"`
  - `expected_cost_support_tier == "medium_high"`
  - `route == "cold_start"`
  - `has_lag == False`
  - `hcpcs_stability_group == "other"` (added after leakage discovery)
- **Trigger**:
  - `oe_ratio >= thresholds_v2["oe_q99"]`
- **Guard rule**:
  - `expected_cost_guard = min(observed_cost, expected_cost * uplift_factor)`
  - `uplift_factor = 2.010610` (empirically from 2020 median obs/exp)
- **Invariant**:
  - bounded by observed ensures `expected_cost_guard <= observed_cost`.

#### Outcome on the intended slice
- Applied rows: `2270` (a subset of J2469 2020 rows meeting the OE trigger)
- J2469 2020 catastrophic rate:
  - Baseline: `87.916344%`
  - Guarded: `0.0%`
  - Improvement: `-87.916344 pp`
  - No new catastrophes.

---

### V2 component 5. C4b. Residual clamp for J2505 (2020–2021)

#### Diagnosis summary (from C.3.a and C.3.b)
`J2505` failures were not primarily OE-driven. They were residual-driven:
- 2020: very high positive residual tail binding
- 2021: strong negative residual tail binding (expected too high relative to observed)
- Buckets: dominated by `cat_abs_residual_q99` and residual buckets.

#### Policy definition
- **Scope**:
  - `Year in {2020, 2021}`
  - `HCPCS_Cd == "J2505"`
  - `expected_cost_support_tier == "medium_high"`
  - `route == "cold_start"`
  - `has_lag == False`
  - `hcpcs_stability_group == "other"` (added after leakage discovery)
- **Trigger**: tail-triggered row if any:
  - `abs_residual >= abs_resid_q99`
  - `residual >= resid_pos_q99`
  - `residual <= -abs_resid_q99` (proxy for negative tail when resid_neg_q99 not present)
- **Cap value**:
  - `cap = thresholds_v2["resid_pos_q99"] - 1e-9`
- **Guard rule** (single safe formula that handles both sides):
  - `expected_cost_guard = min(observed_cost, max(expected_cost, observed_cost - cap))`

Interpretation:
- If expected is too low and residual is too positive, raising to `observed - cap` caps the residual.
- If expected is too high (expected > observed), `min(observed, ...)` clamps expected down to observed (eliminating negative residual spikes).
- Always bounded by observed ensures `expected_cost_guard <= observed_cost`.

#### Outcome on the intended slices
- Applied rows in the focused slice: `1547`
- J2505 2020:
  - Baseline cat: `98.119777%`
  - Guard cat: `1.392758%`
  - Improvement: `-96.727019 pp`
  - No new catastrophes.
- J2505 2021:
  - Baseline cat: `95.882353%`
  - Guard cat: `4.117647%`
  - Improvement: `-91.764706 pp`
  - No new catastrophes.

---

### V2 leakage discovery and fix (important)

#### Problem observed
Initial all-years V2 evaluation reported:

- `Sanity: stable_all_years expected unchanged (all years): False`

The diagnostic “which component is leaking” showed that C4 components were modifying `stable_all_years` rows because their eligibility masks did not explicitly constrain stability group.

#### Fix applied
Both C4 apply functions were patched to include stability group in scope:

- In each apply function, read:
  - `group = str(policy.get("hcpcs_stability_group", "other"))`
- Add to `eligible` / `eligible_scope` masks:
  - `& (out["hcpcs_stability_group"] == group)`
- Add explicit policy key:
  - `"hcpcs_stability_group": "other"`

After this fix:
- `Sanity: stable_all_years expected unchanged (all years): True`
- Per-component change audit confirmed `0` stable rows changed at each step.

---

### V2 composition and evaluation

#### Composition order
V2 is the V1 set plus C4 fixes, applied in order:

1. A1c V0 (2023_only tiny expected)
2. B2.2b (Q5126/Q5127 constraint floor)
3. B2.2c (Q5127 residual cap)
4. C4a (J2469 2020 multiplicative uplift, other, medium_high cold_start cold)
5. C4b (J2505 2020–2021 residual clamp, other, medium_high cold_start cold)

#### Pinned guardrail set
Pinned as a composed set:

- `guardrail_v2 = { name, components=[...five components...] }`

#### All-years evaluation outcome (Year × stability_group)
After the stability-group scope fix:

- **Stable groups**
  - `stable_all_years`: unchanged in every year (expected unchanged and rates identical).
- **No-new-cat**
  - `Year==2023 & 2023_only`: `new_cat_% = 0.0`
- **Improvements occur only where intended**
  - `2020 other`:
    - Baseline: `37.818025%`
    - V2: `7.869771%`
    - Delta: `-29.948254 pp`
  - `2021 other`:
    - Baseline: `15.749040%`
    - V2: `12.895555%`
    - Delta: `-2.853485 pp`
  - `2023 2023_only`:
    - Baseline: `17.100372%`
    - V2: `11.059480%`
    - Delta: `-6.040892 pp`
  - All other strata: unchanged.

#### Canonical saved outputs
The all-years evaluation saved canonical baseline and guarded scored dataframes:

- `baseline_scored_V2_all_years`
- `guard_scored_V2_all_years`

These provide a stable reference for future iterations (V3+) and for presentation tables.

---

### Practical notes (why this structure works)

- You used a consistent workflow:
  1. Identify the highest ROI strata via leaderboard (burden and rate).
  2. Diagnose failure mode (OE-driven vs residual-driven).
  3. Implement a minimal targeted rule with strict scope.
  4. Enforce invariants (expected bounded by observed for modified rows).
  5. Validate no-new-cat in protected slices.
  6. Compose guardrails in a fixed order and re-evaluate across all years.

- You standardized guardrail apply functions to write canonical “guard columns”, enabling predictable composition:
  - `expected_cost_guard`, `residual_guard`, `abs_residual_guard`, `oe_ratio_guard`
  - `guardrail_applied`, `guardrail_name`

- You used robust sanity checks to catch leakage early:
  - stable_all_years expected unchanged across all years
  - no-new-cat on 2023_only
  - per-component “stable rows changed” audits when needed.

This is the foundation for the next phase (beyond monster codes): iterating through additional high-burden HCPCS codes and/or stratified fixes while preserving the same guarantees.